<a href="https://colab.research.google.com/github/Vishesh-Goyal7/NARAD-QC/blob/main/NARAD_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NARAD-QC
### Network Architecture for Routing, Authentication and Delivery in Quantum Communications

## Phase 0 : 06-07-2026T16:15:32 -- 06-07-2026T16:48:53

Goal : Rebuild point to point CrypTon key generation and AES baseline.

Objective : Prove the original point-to-point system is working as the baseline.

In [ ]:
# @title Dependency Installation for Phase 0

!pip install qiskit qiskit-aer pycryptodome pandas matplotlib --quiet

In [ ]:
# @title Imports and Global Config

import time
import secrets
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

RNG = np.random.default_rng()
BACKEND = AerSimulator()

QBER_CUTOFF = 0.11
NONCE_LEN = 12

In [ ]:
# @title AES-GCM Engine

def aes_gcm_encrypt(data: bytes, key: bytes) -> bytes:
    nonce = get_random_bytes(NONCE_LEN)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return nonce + ciphertext + tag


def aes_gcm_decrypt(blob: bytes, key: bytes) -> bytes:
    nonce = blob[:NONCE_LEN]
    rest = blob[NONCE_LEN:]
    ciphertext = rest[:-16]
    tag = rest[-16:]

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    return cipher.decrypt_and_verify(ciphertext, tag)

In [ ]:
# @title BB84 Step 1 : QRNG

def random_bitstring(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()


def random_bases(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()

In [ ]:
# @title BB84 Step 2 : Qubit Encoding

def encode_qubits(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    circuits = []

    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)

        if bit == 1:
            qc.x(0)

        if basis == 1:
            qc.h(0)

        circuits.append(qc)

    return circuits

In [ ]:
# @title BB84 Step 3 : Measure Qubits With Photon Loss Support

def measure_qubits(
    qubits: list,
    bob_bases: list[int],
    eve_present: bool = False
) -> list:

    measured_bits = []

    for i, prepared_qubit in enumerate(qubits):

        # Photon lost in QuantumLink
        if prepared_qubit is None:
            measured_bits.append(None)
            continue

        circuit = prepared_qubit.copy()

        if eve_present:
            eve_basis = int(RNG.integers(0, 2))

            eve_circuit = circuit.copy()

            if eve_basis == 1:
                eve_circuit.h(0)

            eve_circuit.measure(0, 0)

            transpiled_eve = transpile(eve_circuit, BACKEND)
            eve_result = BACKEND.run(
                transpiled_eve,
                shots=1,
                memory=True
            ).result()

            eve_bit = int(eve_result.get_memory()[0])

            circuit = QuantumCircuit(1, 1)

            if eve_bit == 1:
                circuit.x(0)

            if eve_basis == 1:
                circuit.h(0)

        if bob_bases[i] == 1:
            circuit.h(0)

        circuit.measure(0, 0)

        transpiled_bob = transpile(circuit, BACKEND)
        bob_result = BACKEND.run(
            transpiled_bob,
            shots=1,
            memory=True
        ).result()

        measured_bits.append(int(bob_result.get_memory()[0]))

    return measured_bits

In [ ]:
# @title QBER, Sifting, and Key Reconciliation With Photon Loss

def sift_keys(
    alice_bits: list[int],
    alice_bases: list[int],
    bob_bits: list,
    bob_bases: list[int]
) -> tuple[list[int], list[int]]:

    alice_sifted = []
    bob_sifted = []

    for i in range(len(alice_bits)):
        if bob_bits[i] is None:
            continue

        if alice_bases[i] == bob_bases[i]:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])

    return alice_sifted, bob_sifted

def calculate_qber(alice_key: list[int], bob_key: list[int]) -> float:
    if len(alice_key) == 0:
        return 1.0

    errors = sum(a != b for a, b in zip(alice_key, bob_key))
    return errors / len(alice_key)

In [ ]:
# @title One BB84 Round

def bb84_round(
    n_qubits: int = 256,
    eve_present: bool = False
) -> dict:

    alice_bits = random_bitstring(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)

    qubits = encode_qubits(alice_bits, alice_bases)
    bob_bits = measure_qubits(qubits, bob_bases, eve_present=eve_present)

    alice_sifted, bob_sifted = sift_keys(
        alice_bits,
        alice_bases,
        bob_bits,
        bob_bases
    )

    qber = calculate_qber(alice_sifted, bob_sifted)

    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,
        "eve_present": eve_present
    }

In [ ]:
# @title Accumulate Session Key

def bits_to_bytes(bits: list[int]) -> bytes:
    usable_len = len(bits) - (len(bits) % 8)
    bits = bits[:usable_len]

    return bytes(
        int("".join(map(str, bits[i:i+8])), 2)
        for i in range(0, usable_len, 8)
    )


def generate_bb84_session_key(
    target_bits: int = 128,
    n_qubits_per_round: int = 256,
    max_rounds: int = 30,
    eve_present: bool = False,
    verbose: bool = True
) -> tuple[bytes, dict]:

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):
        result = bb84_round(
            n_qubits=n_qubits_per_round,
            eve_present=eve_present
        )

        round_logs.append({
            "round": round_id,
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"]
        })

        if verbose:
            print(
                f"Round {round_id}: "
                f"sifted={result['sifted_length']}, "
                f"QBER={result['qber']:.2%}, "
                f"accepted={result['accepted']}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough accepted BB84 key bits."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    metadata = {
        "target_bits": target_bits,
        "rounds_used": len(round_logs),
        "accepted_rounds": sum(r["accepted"] for r in round_logs),
        "rejected_rounds": sum(not r["accepted"] for r in round_logs),
        "total_sifted_bits": sum(r["sifted_length"] for r in round_logs),
        "keygen_time_s": elapsed,
        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title Phase 0 End to End System COMMENTED OUT

# message = b"NARAD-QC Phase 0: CrypTon baseline reconstruction."

# session_key, meta = generate_bb84_session_key(
#     target_bits=128,
#     n_qubits_per_round=256,
#     max_rounds=30,
#     eve_present=False,
#     verbose=True
# )

# ciphertext = aes_gcm_encrypt(message, session_key)
# decrypted = aes_gcm_decrypt(ciphertext, session_key)

# print("\nSession Key:", session_key.hex())
# print("Key Metadata:", meta)
# print("Ciphertext:", ciphertext.hex())
# print("Decrypted:", decrypted.decode())
# print("Roundtrip OK:", decrypted == message)

In [ ]:
# @title Eve Presence Validation

# try:
#     eve_key, eve_meta = generate_bb84_session_key(
#         target_bits=128,
#         n_qubits_per_round=256,
#         max_rounds=5,
#         eve_present=True,
#         verbose=True
#     )

#     print("Unexpectedly generated key with Eve present.")
#     print(eve_meta)

# except RuntimeError as e:
#     print("Eve disrupted key generation as expected.")
#     print(str(e))

## Phase 1 : 06-07-2026T17:08:12 -- 07-07-2026T16:37:51

Goal : Add distance-dependent photon loss, QBER, keygen latency, and accepted/rejected rounds.

Objective : Evaluate how distance affects QKD in terms of photon loss, QBER, Key generation latency.

Question : How does increasing physical separation affect secure key establishment in CrypTon?

### Phase 1A

QuantumLink and Node classes

In [ ]:
# @title Node Class Definition

ROLE_USER = "USER"
ROLE_REPEATER = "REPEATER"

class Node:
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role

    def __str__(self):
        return f"{self.name} ({self.role})"

In [ ]:
# @title Create 2 User Nodes

alice = Node(name="Alice", role=ROLE_USER)
bob = Node(name="Bob", role=ROLE_USER)

print(alice)
print(bob)

In [ ]:
# @title Quantum Link Class

import random
import math

STANDARD_FIBER_ATTENUATION = 0.2  # dB/km


class QuantumLink:
    """
    NARAD-QC Phase 1.1 / 1.2

    Models a physical quantum channel with distance-dependent photon loss.
    Lost photons are represented as None.
    """

    def __init__(
        self,
        source: Node,
        destination: Node,
        distance_km: float,
        attenuation_db_per_km: float = STANDARD_FIBER_ATTENUATION
    ):
        self.source = source
        self.destination = destination
        self.distance_km = distance_km
        self.attenuation = attenuation_db_per_km
        self.status = "ACTIVE"

        # η(d) = 10^(-(αd)/10)
        self.survival_probability = 10 ** (
            -(self.attenuation * self.distance_km) / 10
        )

        self.loss_probability = 1 - self.survival_probability

        self.statistics = {
            "lifetime_sent": 0,
            "lifetime_received": 0,
            "lifetime_lost": 0
        }

    def transmit(self, qubit):
        """
        Transmit one qubit through the quantum link.
        Returns the qubit if it survives, otherwise None.
        """

        self.statistics["lifetime_sent"] += 1

        if random.random() < self.survival_probability:
            self.statistics["lifetime_received"] += 1
            return qubit

        self.statistics["lifetime_lost"] += 1
        return None

    def transmit_batch(self, qubits: list):
        """
        Transmit a batch of qubits.

        Returns:
        - transmitted_qubits: list containing QuantumCircuit objects or None
        - session-level transmission statistics
        """

        transmitted_qubits = []
        survived = 0

        for qubit in qubits:
            received = self.transmit(qubit)
            transmitted_qubits.append(received)

            if received is not None:
                survived += 1

        sent = len(qubits)
        lost = sent - survived

        return {
            "transmitted_qubits": transmitted_qubits,
            "sent": sent,
            "survived": survived,
            "lost": lost,
            "expected_survival_probability": self.survival_probability,
            "observed_survival_probability": survived / sent if sent else 0,
            "survival_percentage": (survived / sent) * 100 if sent else 0
        }

    def reset_statistics(self):
        self.statistics = {
            "lifetime_sent": 0,
            "lifetime_received": 0,
            "lifetime_lost": 0
        }

    def __str__(self):
        return (
            f"\n{'='*60}\n"
            f"Quantum Link\n"
            f"{'='*60}\n"
            f"Source                          : {self.source.name}\n"
            f"Destination                     : {self.destination.name}\n"
            f"Distance                        : {self.distance_km} km\n"
            f"Attenuation                     : {self.attenuation} dB/km\n"
            f"Status                          : {self.status}\n"
            f"Photon Survival Probability     : {self.survival_probability*100:.2f}%\n"
            f"Photon Loss Probability         : {self.loss_probability*100:.2f}%\n"
            f"{'='*60}"
        )

In [ ]:
# @title Testing one Quantum Link

link = QuantumLink(
    source=alice,
    destination=bob,
    distance_km=50
)

print(link)

In [ ]:
# @title Testing Batch Transmission

dummy_qubits = [f"Photon_{i}" for i in range(1000)]

result = link.transmit_batch(dummy_qubits)

print("Transmission Statistics\n")

print(f"Sent        : {result['sent']}")
print(f"Survived    : {result['survived']}")
print(f"Lost        : {result['lost']}")
print(f"Survival %  : {result['survival_percentage']:.2f}%")

In [ ]:
# @title Distance Comparison

distances = [10, 20, 50, 80, 100, 150, 200]

print(
    f"{'Distance':<12}"
    f"{'Survival %':<15}"
    f"{'Loss %':<15}"
)

print("-" * 42)

for d in distances:
    test_link = QuantumLink(alice, bob, d)

    print(
        f"{d:<12}"
        f"{test_link.survival_probability*100:<15.2f}"
        f"{test_link.loss_probability*100:<15.2f}"
    )

### Phase 1B

Distance --> Attenuation Model --> Testing Over Multiple Distances

In [ ]:
# @title BB84 / Biased-Basis Generator

def random_bases(n: int, z_probability: float = 0.5) -> list[int]:
    """
    Generate random bases.

    0 = Z / computational basis
    1 = X / diagonal basis

    z_probability = 0.5      -> Standard BB84
    z_probability = 0.8      -> Biased BB84
    z_probability = 15/16    -> T12-style biased basis
    """

    return [
        0 if random.random() < z_probability else 1
        for _ in range(n)
    ]


def basis_match_probability(z_probability: float) -> float:
    """
    P(match) = P(Z)^2 + P(X)^2
    """

    return z_probability**2 + (1 - z_probability)**2

In [ ]:
# @title One Distance-Aware BB84 Round With Estimated Photon Budget

import math

def bb84_round_over_link(
    link: QuantumLink,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5
) -> dict:

    p_basis = basis_match_probability(z_probability)

    estimated_photons_required = math.ceil(
        (target_bits * transmission_margin) /
        (p_basis * link.survival_probability)
    )

    n_qubits = max(1, estimated_photons_required)

    alice_bits = random_bitstring(n_qubits)
    alice_bases = random_bases(n_qubits, z_probability=z_probability)
    bob_bases = random_bases(n_qubits, z_probability=z_probability)

    encoded_qubits = encode_qubits(alice_bits, alice_bases)

    transmission = link.transmit_batch(encoded_qubits)
    received_qubits = transmission["transmitted_qubits"]

    bob_bits = measure_qubits(received_qubits, bob_bases, eve_present=eve_present)

    alice_sifted, bob_sifted = sift_keys(alice_bits, alice_bases, bob_bits, bob_bases)

    qber = calculate_qber(alice_sifted, bob_sifted)
    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,
        "eve_present": eve_present,
        "z_probability": z_probability,
        "basis_match_probability": p_basis,
        "target_bits_for_round": target_bits,
        "transmission_margin": transmission_margin,
        "estimated_photons_required": estimated_photons_required,
        "photons_sent": transmission["sent"],
        "photons_received": transmission["survived"],
        "photons_lost": transmission["lost"],
        "expected_survival_probability": transmission["expected_survival_probability"],
        "observed_survival_probability": transmission["observed_survival_probability"]
    }

In [ ]:
# @title Generate Session Key Over Quantum Link

def generate_bb84_session_key_over_link(
    link: QuantumLink,
    target_bits: int = 128,
    max_rounds: int = 100,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5,
    verbose: bool = True
) -> tuple[bytes, dict]:

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):

        remaining_bits = target_bits - len(collected_bits)

        result = bb84_round_over_link(
            link=link,
            target_bits=remaining_bits,
            transmission_margin=transmission_margin,
            eve_present=eve_present,
            z_probability=z_probability
        )

        round_log = {
            "round": round_id,
            "distance_km": link.distance_km,
            "photons_sent": result["photons_sent"],
            "photons_received": result["photons_received"],
            "photons_lost": result["photons_lost"],
            "observed_survival_probability": result["observed_survival_probability"],
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"],
            "z_probability": z_probability,
            "basis_match_probability": result["basis_match_probability"],
            "target_bits_for_round": result["target_bits_for_round"],
            "transmission_margin": result["transmission_margin"],
            "estimated_photons_required": result["estimated_photons_required"],
        }

        round_logs.append(round_log)

        if verbose:
            print(
                f"Round {round_id}: "
                f"distance={link.distance_km} km | "
                f"sent={result['photons_sent']} | "
                f"received={result['photons_received']} | "
                f"sifted={result['sifted_length']} | "
                f"QBER={result['qber']:.2%} | "
                f"accepted={result['accepted']}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough accepted BB84 key bits over the quantum link."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    total_sent = sum(r["photons_sent"] for r in round_logs)
    total_received = sum(r["photons_received"] for r in round_logs)
    total_lost = sum(r["photons_lost"] for r in round_logs)
    total_sifted = sum(r["sifted_length"] for r in round_logs)

    metadata = {
        "source": link.source.name,
        "destination": link.destination.name,
        "distance_km": link.distance_km,
        "target_bits": target_bits,
        "z_probability": z_probability,
        "basis_match_probability": basis_match_probability(z_probability),
        "rounds_used": len(round_logs),
        "accepted_rounds": sum(r["accepted"] for r in round_logs),
        "rejected_rounds": sum(not r["accepted"] for r in round_logs),
        "total_photons_sent": total_sent,
        "total_photons_received": total_received,
        "total_photons_lost": total_lost,
        "observed_survival_probability": total_received / total_sent if total_sent else 0,
        "total_sifted_bits": total_sifted,
        "effective_key_yield": target_bits / total_sent if total_sent else 0,
        "keygen_time_s": elapsed,
        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title End-to-End Distance-Aware Test

# distance_link = QuantumLink(
#     source=alice,
#     destination=bob,
#     distance_km=50
# )

# session_key, meta = generate_bb84_session_key_over_link(
#     link=distance_link,
#     target_bits=128,
#     max_rounds=100,
#     eve_present=False,
#     z_probability=0.5,   # Standard BB84
#     verbose=True,
#     transmission_margin = 1.3,
# )

# message = b"NARAD-QC Phase 1.2: Distance-aware BB84 session key."

# ciphertext = aes_gcm_encrypt(message, session_key)
# decrypted = aes_gcm_decrypt(ciphertext, session_key)

# print("\n" + "="*60)
# print("NARAD-QC Phase 1.2 Session Report")
# print("="*60)
# print(f"Source                     : {meta['source']}")
# print(f"Destination                : {meta['destination']}")
# print(f"Distance                   : {meta['distance_km']} km")
# print(f"Target Key Bits            : {meta['target_bits']}")
# print(f"Z-Basis Probability        : {meta['z_probability']}")
# print(f"Basis Match Probability    : {meta['basis_match_probability']*100:.2f}%")
# print(f"Rounds Used                : {meta['rounds_used']}")
# print(f"Total Photons Sent         : {meta['total_photons_sent']}")
# print(f"Total Photons Received     : {meta['total_photons_received']}")
# print(f"Total Photons Lost         : {meta['total_photons_lost']}")
# print(f"Observed Survival          : {meta['observed_survival_probability']*100:.2f}%")
# print(f"Total Sifted Bits          : {meta['total_sifted_bits']}")
# print(f"Effective Key Yield        : {meta['effective_key_yield']*100:.4f}%")
# print(f"Keygen Time                : {meta['keygen_time_s']:.4f} s")
# print(f"AES Roundtrip OK           : {decrypted == message}")
# print("="*60)

In [ ]:
# @title Standard BB84 vs T12-Style Biased Basis

# test_distance = 50
# target_bits = 128

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# comparison_results = []

# for protocol_name, z_prob in protocols.items():

#     test_link = QuantumLink(alice, bob, test_distance)

#     key, meta = generate_bb84_session_key_over_link(
#         link=test_link,
#         target_bits=target_bits,
#         max_rounds=100,
#         eve_present=False,
#         z_probability=z_prob,
#         verbose=False,
#         transmission_margin = 1.3
#     )

#     comparison_results.append({
#         "Protocol": protocol_name,
#         "Distance (km)": test_distance,
#         "Z Probability": z_prob,
#         "Basis Match Probability (%)": meta["basis_match_probability"] * 100,
#         "Rounds Used": meta["rounds_used"],
#         "Photons Sent": meta["total_photons_sent"],
#         "Photons Received": meta["total_photons_received"],
#         "Sifted Bits": meta["total_sifted_bits"],
#         "Effective Key Yield (%)": meta["effective_key_yield"] * 100,
#         "Keygen Time (s)": meta["keygen_time_s"]
#     })

# df_protocol_comparison = pd.DataFrame(comparison_results)
# display(df_protocol_comparison)

In [ ]:
# @title Distance Performance : BB84 vs Biased BB84 vs T12

# distances = [10, 50, 80, 100, 150, 200]
# target_bits = 128
# transmission_margin = 1.3

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# distance_sweep_results = []

# for d in distances:
#     for protocol_name, z_prob in protocols.items():

#         test_link = QuantumLink(
#             source=alice,
#             destination=bob,
#             distance_km=d
#         )

#         try:
#             key, meta = generate_bb84_session_key_over_link(
#                 link=test_link,
#                 target_bits=target_bits,
#                 transmission_margin=transmission_margin,
#                 max_rounds=10,
#                 eve_present=False,
#                 z_probability=z_prob,
#                 verbose=False
#             )

#             distance_sweep_results.append({
#                 "Protocol": protocol_name,
#                 "Distance (km)": d,
#                 "Z Probability": z_prob,
#                 "Basis Match Probability (%)": meta["basis_match_probability"] * 100,
#                 "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                 "Observed Photon Survival (%)": meta["observed_survival_probability"] * 100,
#                 "Rounds Used": meta["rounds_used"],
#                 "Photons Sent": meta["total_photons_sent"],
#                 "Photons Received": meta["total_photons_received"],
#                 "Photons Lost": meta["total_photons_lost"],
#                 "Sifted Bits": meta["total_sifted_bits"],
#                 "Effective Key Yield (%)": meta["effective_key_yield"] * 100,
#                 "Keygen Time (s)": meta["keygen_time_s"],
#                 "Status": "SUCCESS"
#             })

#         except RuntimeError as e:
#             distance_sweep_results.append({
#                 "Protocol": protocol_name,
#                 "Distance (km)": d,
#                 "Z Probability": z_prob,
#                 "Basis Match Probability (%)": basis_match_probability(z_prob) * 100,
#                 "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                 "Observed Photon Survival (%)": None,
#                 "Rounds Used": None,
#                 "Photons Sent": None,
#                 "Photons Received": None,
#                 "Photons Lost": None,
#                 "Sifted Bits": None,
#                 "Effective Key Yield (%)": None,
#                 "Keygen Time (s)": None,
#                 "Status": "FAILED"
#             })

# df_distance_sweep = pd.DataFrame(distance_sweep_results)

# display(df_distance_sweep)

# df_distance_sweep.to_csv("distance_sweep_results.csv", index=False)

In [ ]:
# @title Plot: Distance vs Key Generation Time

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Keygen Time (s)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Key Generation Time (s)")
# plt.title("Distance vs Key Generation Time")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Key Generation Time", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Effective Key Yield

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Effective Key Yield (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Effective Key Yield (%)")
# plt.title("Distance vs Effective Key Yield")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Effective Key Yield", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Photon Survival

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Observed Photon Survival (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Observed Photon Survival (%)")
# plt.title("Distance vs Observed Photon Survival")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Observed Photon Survival", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Photons Sent

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Photons Sent"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Photons Sent")
# plt.title("Distance vs Photons Required for 128-bit Key")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Photons Required for 128-bit Key", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Sifted Bits

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Sifted Bits"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Sifted Bits")
# plt.title("Distance vs Sifted Bits Generated")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Sifted Bits Generated", dpi=300, bbox_inches="tight")
# plt.show()

### Phase 1C

Monte Carlo - 100 Trials

In [ ]:
# @title Phase 1 Monte Carlo Experiment — 100 Trials

# import gc
# import os
# import time
# import pandas as pd

# PHASE1_MONTE_CARLO_TRIALS = 100
# PHASE1_CHECKPOINT_INTERVAL = 1

# phase1_mc_distances = [10, 50, 80, 100, 150, 200]

# phase1_mc_protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15 / 16
# }

# phase1_mc_target_bits = 128
# phase1_mc_transmission_margin = 1.3
# phase1_mc_max_rounds = 10

# PHASE1_RAW_FILE = "phase1_monte_carlo_raw_100_trials.csv"
# PHASE1_LATEST_CHECKPOINT_FILE = "phase1_monte_carlo_latest_checkpoint.csv"

# # Set this to True only when resuming from a saved latest checkpoint.
# RESUME_FROM_CHECKPOINT = True

# phase1_raw_results = []
# starting_trial = 98

# if RESUME_FROM_CHECKPOINT and os.path.exists(PHASE1_LATEST_CHECKPOINT_FILE):

#     existing_checkpoint = pd.read_csv(PHASE1_LATEST_CHECKPOINT_FILE)

#     phase1_raw_results = existing_checkpoint.to_dict("records")

#     if not existing_checkpoint.empty:
#         last_completed_trial = int(existing_checkpoint["Trial"].max())
#         starting_trial = last_completed_trial + 1

#         print(
#             f"Resuming Phase 1 Monte Carlo experiment "
#             f"from Trial {starting_trial}."
#         )

# else:
#     print("Starting a new Phase 1 Monte Carlo experiment.")


# # ---------------------------------------------------------
# # Monte Carlo execution
# # ---------------------------------------------------------

# experiment_start_time = time.perf_counter()

# for trial in range(
#     starting_trial,
#     PHASE1_MONTE_CARLO_TRIALS + 1
# ):

#     trial_start_time = time.perf_counter()

#     print(
#         f"\n{'=' * 72}\n"
#         f"Phase 1 Monte Carlo — Trial "
#         f"{trial}/{PHASE1_MONTE_CARLO_TRIALS}\n"
#         f"{'=' * 72}"
#     )

#     for distance_km in phase1_mc_distances:

#         for protocol_name, z_probability in phase1_mc_protocols.items():

#             configuration_start_time = time.perf_counter()

#             test_link = QuantumLink(
#                 source=alice,
#                 destination=bob,
#                 distance_km=distance_km
#             )

#             try:
#                 session_key, metadata = (
#                     generate_bb84_session_key_over_link(
#                         link=test_link,
#                         target_bits=phase1_mc_target_bits,
#                         transmission_margin=(
#                             phase1_mc_transmission_margin
#                         ),
#                         max_rounds=phase1_mc_max_rounds,
#                         eve_present=False,
#                         z_probability=z_probability,
#                         verbose=False
#                     )
#                 )

#                 configuration_elapsed = (
#                     time.perf_counter()
#                     - configuration_start_time
#                 )

#                 phase1_raw_results.append({
#                     "Trial": trial,
#                     "Protocol": protocol_name,
#                     "Distance (km)": distance_km,
#                     "Target Key Bits": phase1_mc_target_bits,
#                     "Transmission Margin": phase1_mc_transmission_margin,
#                     "Z Probability": z_probability,
#                     "Basis Match Probability (%)": metadata["basis_match_probability"] * 100,
#                     "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                     "Observed Photon Survival (%)": metadata["observed_survival_probability"] * 100,
#                     "Rounds Used": metadata["rounds_used"],
#                     "Accepted Rounds": metadata["accepted_rounds"],
#                     "Rejected Rounds": metadata["rejected_rounds"],
#                     "Photons Sent": metadata["total_photons_sent"],
#                     "Photons Received": metadata["total_photons_received"],
#                     "Photons Lost": metadata["total_photons_lost"],
#                     "Sifted Bits": metadata["total_sifted_bits"],
#                     "Effective Key Yield (%)": metadata["effective_key_yield"] * 100,
#                     "Keygen Time (s)": metadata["keygen_time_s"],
#                     "Configuration Runtime (s)": configuration_elapsed,
#                     "Status": "SUCCESS",
#                     "Error": None
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"sent={metadata['total_photons_sent']:,} | "
#                     f"received="
#                     f"{metadata['total_photons_received']:,} | "
#                     f"time="
#                     f"{metadata['keygen_time_s']:.3f}s"
#                 )

#             except Exception as error:

#                 configuration_elapsed = (time.perf_counter() - configuration_start_time)
#                 phase1_raw_results.append({
#                     "Trial": trial,
#                     "Protocol": protocol_name,
#                     "Distance (km)": distance_km,
#                     "Target Key Bits": phase1_mc_target_bits,
#                     "Transmission Margin": phase1_mc_transmission_margin,
#                     "Z Probability": z_probability,
#                     "Basis Match Probability (%)": basis_match_probability(z_probability) * 100,
#                     "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                     "Observed Photon Survival (%)": None,
#                     "Rounds Used": None,
#                     "Accepted Rounds": None,
#                     "Rejected Rounds": None,
#                     "Photons Sent": None,
#                     "Photons Received": None,
#                     "Photons Lost": None,
#                     "Sifted Bits": None,
#                     "Effective Key Yield (%)": None,
#                     "Keygen Time (s)": None,
#                     "Configuration Runtime (s)": configuration_elapsed,
#                     "Status": "FAILED",
#                     "Error": str(error)
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"FAILED: {error}"
#                 )

#             # Qiskit circuit batches can consume substantial memory.
#             del test_link

#             if "session_key" in locals():
#                 del session_key

#             if "metadata" in locals():
#                 del metadata

#             gc.collect()

#     trial_elapsed = time.perf_counter() - trial_start_time

#     print(
#         f"\nTrial {trial} completed in "
#         f"{trial_elapsed / 60:.2f} minutes."
#     )

#     if (trial % PHASE1_CHECKPOINT_INTERVAL == 0 or trial == PHASE1_MONTE_CARLO_TRIALS):

#         checkpoint_df = pd.DataFrame(
#             phase1_raw_results
#         )

#         numbered_checkpoint_file = (
#             f"phase1_monte_carlo_checkpoint_"
#             f"trial_{trial}.csv"
#         )

#         checkpoint_df.to_csv(
#             numbered_checkpoint_file,
#             index=False
#         )

#         checkpoint_df.to_csv(
#             PHASE1_LATEST_CHECKPOINT_FILE,
#             index=False
#         )

#         print(
#             f"\nCheckpoint saved after Trial {trial}:"
#             f"\n  {numbered_checkpoint_file}"
#             f"\n  {PHASE1_LATEST_CHECKPOINT_FILE}"
#             f"\n  Rows saved: {len(checkpoint_df):,}"
#         )

# df_phase1_monte_carlo_raw = pd.DataFrame(phase1_raw_results)

# df_phase1_monte_carlo_raw.to_csv(PHASE1_RAW_FILE, index=False)

# experiment_elapsed = time.perf_counter() - experiment_start_time

# print(
#     f"\n{'=' * 72}\n"
#     f"PHASE 1 MONTE CARLO COMPLETE\n"
#     f"{'=' * 72}\n"
#     f"Trials completed       : "
#     f"{PHASE1_MONTE_CARLO_TRIALS}\n"
#     f"Configurations/trial   : "
#     f"{len(phase1_mc_distances) * len(phase1_mc_protocols)}\n"
#     f"Raw rows generated     : "
#     f"{len(df_phase1_monte_carlo_raw):,}\n"
#     f"Expected raw rows      : "
#     f"{PHASE1_MONTE_CARLO_TRIALS * len(phase1_mc_distances) * len(phase1_mc_protocols):,}\n"
#     f"Total runtime          : "
#     f"{experiment_elapsed / 3600:.2f} hours\n"
#     f"Final raw CSV          : "
#     f"{PHASE1_RAW_FILE}\n"
#     f"{'=' * 72}"
# )

# display(
#     df_phase1_monte_carlo_raw.head(18)
# )

In [ ]:
# @title Phase 1 Monte Carlo Aggregate Statistics

# import numpy as np
# import pandas as pd

# # Load from disk so this cell also works after a runtime restart.
# df_phase1_monte_carlo_raw = pd.read_csv(
#     "phase1_monte_carlo_raw_100_trials.csv"
# )

# successful_phase1_runs = (
#     df_phase1_monte_carlo_raw[df_phase1_monte_carlo_raw["Status"]== "SUCCESS"].copy()
# )

# phase1_group_columns = [
#     "Protocol",
#     "Distance (km)"
# ]

# phase1_metrics_to_aggregate = [
#     "Expected Photon Survival (%)",
#     "Observed Photon Survival (%)",
#     "Rounds Used",
#     "Accepted Rounds",
#     "Rejected Rounds",
#     "Photons Sent",
#     "Photons Received",
#     "Photons Lost",
#     "Sifted Bits",
#     "Effective Key Yield (%)",
#     "Keygen Time (s)",
#     "Configuration Runtime (s)"
# ]

# phase1_summary = (
#     successful_phase1_runs
#     .groupby(
#         phase1_group_columns,
#         as_index=False
#     )[phase1_metrics_to_aggregate]
#     .agg([
#         "count",
#         "mean",
#         "std",
#         "median",
#         "min",
#         "max"
#     ])
# )

# # Flatten Pandas multi-level column names.
# phase1_summary.columns = [
#     " ".join(
#         str(part)
#         for part in column
#         if str(part) != ""
#     ).strip()
#     if isinstance(column, tuple)
#     else column
#     for column in phase1_summary.columns
# ]

# # Restore readable group-column names after flattening.
# phase1_summary = phase1_summary.rename(
#     columns={
#         "Protocol": "Protocol",
#         "Distance (km)": "Distance (km)"
#     }
# )

# phase1_run_counts = (
#     df_phase1_monte_carlo_raw
#     .groupby(
#         phase1_group_columns
#     )
#     .agg(
#         Total_Trials=("Trial", "count"),
#         Successful_Trials=(
#             "Status",
#             lambda values: (
#                 values == "SUCCESS"
#             ).sum()
#         ),
#         Failed_Trials=(
#             "Status",
#             lambda values: (
#                 values == "FAILED"
#             ).sum()
#         )
#     )
#     .reset_index()
# )

# phase1_run_counts[
#     "Success Rate (%)"
# ] = (
#     phase1_run_counts["Successful_Trials"]
#     / phase1_run_counts["Total_Trials"]
#     * 100
# )

# df_phase1_monte_carlo_summary = (
#     phase1_run_counts.merge(
#         phase1_summary,
#         on=phase1_group_columns,
#         how="left"
#     )
# )

# # Sort consistently for the table and graphs.
# protocol_order = {
#     "Standard BB84 (50-50)": 0,
#     "Biased BB84 (80-20)": 1,
#     "T12-style (15/16 - 1/16)": 2
# }

# df_phase1_monte_carlo_summary[
#     "_Protocol Order"
# ] = (
#     df_phase1_monte_carlo_summary[
#         "Protocol"
#     ].map(protocol_order)
# )

# df_phase1_monte_carlo_summary = (
#     df_phase1_monte_carlo_summary
#     .sort_values(
#         by=[
#             "_Protocol Order",
#             "Distance (km)"
#         ]
#     )
#     .drop(
#         columns=["_Protocol Order"]
#     )
#     .reset_index(drop=True)
# )

# PHASE1_SUMMARY_FILE = (
#     "phase1_monte_carlo_summary_100_trials.csv"
# )

# df_phase1_monte_carlo_summary.to_csv(
#     PHASE1_SUMMARY_FILE,
#     index=False
# )

# print(
#     f"Phase 1 aggregate summary saved to "
#     f"{PHASE1_SUMMARY_FILE}"
# )

# print(
#     f"Raw rows       : "
#     f"{len(df_phase1_monte_carlo_raw):,}"
# )

# print(
#     f"Summary rows   : "
#     f"{len(df_phase1_monte_carlo_summary):,}"
# )

# print(
#     f"Failed runs    : "
#     f"{(df_phase1_monte_carlo_raw['Status'] == 'FAILED').sum():,}"
# )

# display(
#     df_phase1_monte_carlo_summary
# )

In [ ]:
# @title Phase 1 Compact Paper-Ready Summary

# phase1_paper_columns = [
#     "Protocol",
#     "Distance (km)",
#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",

#     "Observed Photon Survival (%) mean",
#     "Observed Photon Survival (%) std",

#     "Photons Sent mean",
#     "Photons Sent std",

#     "Effective Key Yield (%) mean",
#     "Effective Key Yield (%) std",

#     "Keygen Time (s) mean",
#     "Keygen Time (s) std",

#     "Sifted Bits mean",
#     "Sifted Bits std"
# ]

# df_phase1_paper_summary = (
#     df_phase1_monte_carlo_summary[phase1_paper_columns].copy()
# )

# df_phase1_paper_summary.to_csv(
#     "phase1_monte_carlo_paper_summary_100_trials.csv",
#     index=False
# )

# display(df_phase1_paper_summary)

In [ ]:
# @title Phase 1 Monte Carlo Paper-Ready Graphs

# import matplotlib.pyplot as plt
# import pandas as pd

# summary_df = pd.read_csv(
#     "phase1_monte_carlo_summary_100_trials.csv"
# )

# phase1_plot_protocols = [
#     "Standard BB84 (50-50)",
#     "Biased BB84 (80-20)",
#     "T12-style (15/16 - 1/16)"
# ]


# # =========================================================
# # Graph 1: Distance vs mean key-generation time

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Keygen Time (s) mean"],
#         yerr=subset["Keygen Time (s) std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Key Generation Time (s)")
# plt.title(
#     "Distance vs Key Generation Time "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Key Generation Time.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 2: Distance vs mean effective key yield

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Effective Key Yield (%) mean"],
#         yerr=subset["Effective Key Yield (%) std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Effective Key Yield (%)")
# plt.title(
#     "Distance vs Effective Key Yield "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Effective Key Yield.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 3: Distance vs observed photon survival

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Observed Photon Survival (%) mean"
#         ],
#         yerr=subset[
#             "Observed Photon Survival (%) std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Observed Photon Survival (%)")
# plt.title(
#     "Distance vs Observed Photon Survival "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Observed Photon Survival.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 4: Distance vs photons required

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Photons Sent mean"],
#         yerr=subset["Photons Sent std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Photons Sent")
# plt.title(
#     "Distance vs Photon Requirement for a "
#     "128-bit Key (100 Trials)"
# )
# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Photons Required.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 5: Distance vs sifted bits

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Sifted Bits mean"],
#         yerr=subset["Sifted Bits std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Sifted Bits")
# plt.title(
#     "Distance vs Sifted Key Bits "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Sifted Bits.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 6: Success rate

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset["Success Rate (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Successful Sessions (%)")
# plt.title(
#     "Distance vs Session Success Rate "
#     "(100 Trials)"
# )
# plt.ylim(0, 105)
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Session Success Rate.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

## Phase 2 : 07-07-2026T16:50:12 -- 09-07-2026T02:17:32

Goal : Add a single repeater so that Alice <--> Bob chain becomes ===> Alice <--> R1 <--> Bob


Objective : Measure whether splitting distance improves stability and key availability.

Question : Can a single repeater improve the effective QKD range and reliability compared to direct transmission?

### Phase 2A

Single Quantum Repeater Abstraction

In [ ]:
# @title Single Repeater Node

r1 = Node(name="R1", role=ROLE_REPEATER)

print(r1)

In [ ]:
# @title Bell Pair Abstraction

class BellPair:
    """
    Represents an entangled Bell pair between two nodes.

    In Phase 2A, this is a network-level abstraction.
    It does not simulate the full photonic hardware circuit.
    """

    def __init__(
        self,
        node_a: Node,
        node_b: Node,
        fidelity: float = 1.0,
        source_link_distance_km: float = None
    ):
        self.node_a = node_a
        self.node_b = node_b
        self.fidelity = fidelity
        self.source_link_distance_km = source_link_distance_km

    def involves(self, node: Node) -> bool:
        return self.node_a == node or self.node_b == node

    def other_node(self, node: Node) -> Node:
        if self.node_a == node:
            return self.node_b
        if self.node_b == node:
            return self.node_a

        raise ValueError("Node is not part of this Bell pair.")

    def __str__(self):
        return (
            f"BellPair({self.node_a.name} ↔ {self.node_b.name}, "
            f"F={self.fidelity:.4f})"
        )

In [ ]:
# @title Bell Pair Generation Attempt

def attempt_bell_pair_generation(
    link: QuantumLink,
    source_fidelity: float = 1.0
):
    """
    Attempt to generate a Bell pair across one elementary quantum link.

    Phase 2A abstraction:
    - Success probability is approximated using link photon survival probability.
    - Successful attempt returns a BellPair object.
    - Failed attempt returns None.
    """

    if random.random() < link.survival_probability:
        return BellPair(
            node_a=link.source,
            node_b=link.destination,
            fidelity=source_fidelity,
            source_link_distance_km=link.distance_km
        )

    return None

In [ ]:
# @title Entanglement Swapping

def entanglement_swap(
    left_pair: BellPair,
    right_pair: BellPair,
    repeater: Node,
    swapping_success_probability: float = 1.0
):
    """
    Perform entanglement swapping at the repeater.

    Example:
        left_pair  = BellPair(Alice ↔ R1)
        right_pair = BellPair(R1 ↔ Bob)

    Output:
        BellPair(Alice ↔ Bob)

    The repeater does not learn the key/message.
    It only helps convert two elementary entangled links
    into one end-to-end entangled link.
    """

    if left_pair is None or right_pair is None:
        return None

    if not left_pair.involves(repeater):
        raise ValueError("Left Bell pair does not involve the repeater.")

    if not right_pair.involves(repeater):
        raise ValueError("Right Bell pair does not involve the repeater.")

    if random.random() > swapping_success_probability:
        return None

    end_a = left_pair.other_node(repeater)
    end_b = right_pair.other_node(repeater)

    combined_fidelity = left_pair.fidelity * right_pair.fidelity

    return BellPair(
        node_a=end_a,
        node_b=end_b,
        fidelity=combined_fidelity,
        source_link_distance_km=(left_pair.source_link_distance_km or 0) + (right_pair.source_link_distance_km or 0)
    )

In [ ]:
# @title Single Repeater Swapping Attempt Without Memory

def attempt_single_repeater_connection(
    link_left: QuantumLink,
    link_right: QuantumLink,
    repeater: Node,
    swapping_success_probability: float = 1.0
) -> dict:

    left_pair = attempt_bell_pair_generation(link_left)
    right_pair = attempt_bell_pair_generation(link_right)

    end_to_end_pair = None

    if left_pair is not None and right_pair is not None:
        end_to_end_pair = entanglement_swap(
            left_pair=left_pair,
            right_pair=right_pair,
            repeater=repeater,
            swapping_success_probability=swapping_success_probability
        )

    return {
        "left_pair": left_pair,
        "right_pair": right_pair,
        "end_to_end_pair": end_to_end_pair,
        "left_success": left_pair is not None,
        "right_success": right_pair is not None,
        "swap_success": end_to_end_pair is not None,
        "end_to_end_success": end_to_end_pair is not None,
        "left_survival_probability": link_left.survival_probability,
        "right_survival_probability": link_right.survival_probability,
        "swapping_success_probability": swapping_success_probability
    }

In [ ]:
# @title Direct vs Single Repeater Theoretical Comparison

def compare_direct_vs_single_repeater_theory(
    source: Node,
    repeater: Node,
    destination: Node,
    total_distance_km: float,
    swapping_success_probability: float = 1.0
) -> dict:

    direct_link = QuantumLink(source, destination, total_distance_km)

    segment_distance = total_distance_km / 2

    left_link = QuantumLink(source, repeater, segment_distance)
    right_link = QuantumLink(repeater, destination, segment_distance)

    direct_success_probability = direct_link.survival_probability

    no_memory_repeater_success_probability = left_link.survival_probability * right_link.survival_probability * swapping_success_probability

    direct_expected_attempts = 1 / direct_success_probability if direct_success_probability > 0 else float("inf")

    no_memory_repeater_expected_attempts = 1 / no_memory_repeater_success_probability if no_memory_repeater_success_probability > 0 else float("inf")

    return {
        "total_distance_km": total_distance_km,
        "direct_distance_km": total_distance_km,
        "segment_distance_km": segment_distance,
        "direct_success_probability": direct_success_probability,
        "left_segment_success_probability": left_link.survival_probability,
        "right_segment_success_probability": right_link.survival_probability,
        "swapping_success_probability": swapping_success_probability,
        "no_memory_repeater_success_probability": no_memory_repeater_success_probability,
        "direct_expected_attempts": direct_expected_attempts,
        "no_memory_repeater_expected_attempts": no_memory_repeater_expected_attempts
    }

In [ ]:
# @title Memory-Assisted Single Repeater Model With Bell Pairs

def generate_entanglement_via_single_repeater_with_memory(
    source: Node,
    repeater: Node,
    destination: Node,
    total_distance_km: float,
    swapping_success_probability: float = 1.0,
    memory_lifetime_attempts: int = 10000,
    max_attempts: int = 1000000,
    verbose: bool = False
) -> dict:
    """
    Memory-assisted single quantum repeater model.

    Topology:
        source ---- repeater ---- destination

    Process:
    1. Try generating BellPair(source ↔ repeater).
    2. Try generating BellPair(repeater ↔ destination).
    3. Store successful Bell pairs in quantum memory.
    4. Retry only failed side.
    5. Once both Bell pairs exist, perform entanglement swapping.
    6. Output BellPair(source ↔ destination).
    """

    segment_distance = total_distance_km / 2

    link_left = QuantumLink(source, repeater, segment_distance)
    link_right = QuantumLink(repeater, destination, segment_distance)
    direct_link = QuantumLink(source, destination, total_distance_km)

    left_pair = None
    right_pair = None

    left_memory_age = 0
    right_memory_age = 0

    left_attempts = 0
    right_attempts = 0

    swap_attempts = 0

    left_memory_expirations = 0
    right_memory_expirations = 0

    start_time = time.perf_counter()

    for global_attempt in range(1, max_attempts + 1):

        if left_pair is not None:
            left_memory_age += 1

            if left_memory_age > memory_lifetime_attempts:
                left_pair = None
                left_memory_age = 0
                left_memory_expirations += 1

        if right_pair is not None:
            right_memory_age += 1

            if right_memory_age > memory_lifetime_attempts:
                right_pair = None
                right_memory_age = 0
                right_memory_expirations += 1

        if left_pair is None:
            left_attempts += 1
            generated_left = attempt_bell_pair_generation(link_left)

            if generated_left is not None:
                left_pair = generated_left
                left_memory_age = 0

        if right_pair is None:
            right_attempts += 1
            generated_right = attempt_bell_pair_generation(link_right)

            if generated_right is not None:
                right_pair = generated_right
                right_memory_age = 0

        if left_pair is not None and right_pair is not None:
            swap_attempts += 1
            end_to_end_pair = entanglement_swap(
                left_pair=left_pair,
                right_pair=right_pair,
                repeater=repeater,
                swapping_success_probability=swapping_success_probability
            )

            if end_to_end_pair is not None:
                elapsed = time.perf_counter() - start_time

                return {
                    "source": source.name,
                    "repeater": repeater.name,
                    "destination": destination.name,
                    "total_distance_km": total_distance_km,
                    "segment_distance_km": segment_distance,
                    "direct_survival_probability": direct_link.survival_probability,
                    "segment_survival_probability": link_left.survival_probability,
                    "swapping_success_probability": swapping_success_probability,
                    "memory_lifetime_attempts": memory_lifetime_attempts,
                    "global_attempts_required": global_attempt,
                    "left_attempts": left_attempts,
                    "right_attempts": right_attempts,
                    "swap_attempts": swap_attempts,
                    "left_memory_expirations": left_memory_expirations,
                    "right_memory_expirations": right_memory_expirations,
                    "final_bell_pair": str(end_to_end_pair),
                    "final_fidelity": end_to_end_pair.fidelity,
                    "success": True,
                    "time_s": elapsed
                }

            left_pair = None
            right_pair = None
            left_memory_age = 0
            right_memory_age = 0

        if verbose and global_attempt <= 20:
            print(
                f"Attempt {global_attempt}: "
                f"L_pair={left_pair is not None}, "
                f"R_pair={right_pair is not None}, "
                f"L_age={left_memory_age}, "
                f"R_age={right_memory_age}"
            )

    elapsed = time.perf_counter() - start_time

    return {
        "source": source.name,
        "repeater": repeater.name,
        "destination": destination.name,
        "total_distance_km": total_distance_km,
        "segment_distance_km": segment_distance,
        "direct_survival_probability": direct_link.survival_probability,
        "segment_survival_probability": link_left.survival_probability,
        "swapping_success_probability": swapping_success_probability,
        "memory_lifetime_attempts": memory_lifetime_attempts,
        "global_attempts_required": max_attempts,
        "left_attempts": left_attempts,
        "right_attempts": right_attempts,
        "swap_attempts": swap_attempts,
        "left_memory_expirations": left_memory_expirations,
        "right_memory_expirations": right_memory_expirations,
        "final_bell_pair": None,
        "final_fidelity": None,
        "success": False,
        "time_s": elapsed
    }

In [ ]:
# @title Single Repeater Test

result = generate_entanglement_via_single_repeater_with_memory(
    source=alice,
    repeater=r1,
    destination=bob,
    total_distance_km=100,
    swapping_success_probability=1.0,
    memory_lifetime_attempts=10000,
    max_attempts=1000000,
    verbose=True
)

print("\n")
print("=" * 70)
print("Phase 2A Single Repeater Result")
print("=" * 70)

for k, v in result.items():
    if isinstance(v, float):
        print(f"{k:<35}: {v:.6f}")
    else:
        print(f"{k:<35}: {v}")

print("=" * 70)

In [ ]:
# @title Distance Sweep Direct vs Single Repeater

# phase2_distances = [50, 80, 100, 150, 200, 250, 300]

# phase2_results = []

# for d in phase2_distances:

#     direct_link = QuantumLink(alice, bob, d)
#     repeater_result = generate_entanglement_via_single_repeater_with_memory(
#         source=alice,
#         repeater=r1,
#         destination=bob,
#         total_distance_km=d,
#         swapping_success_probability=1.0,
#         memory_lifetime_attempts=10000,
#         max_attempts=1000000,
#         verbose=False
#     )

#     phase2_results.append({
#         "Distance (km)": d,
#         "Direct Survival Probability (%)": direct_link.survival_probability * 100,
#         "Direct Expected Attempts": 1 / direct_link.survival_probability,
#         "Segment Distance (km)": d / 2,
#         "Segment Survival Probability (%)": repeater_result["segment_survival_probability"] * 100,
#         "Repeater Global Attempts": repeater_result["global_attempts_required"],
#         "Left Link Attempts": repeater_result["left_attempts"],
#         "Right Link Attempts": repeater_result["right_attempts"],
#         "Swap Attempts": repeater_result["swap_attempts"],
#         "Final Bell Pair": repeater_result["final_bell_pair"],
#         "Final Fidelity": repeater_result["final_fidelity"],
#         "Memory Lifetime": repeater_result["memory_lifetime_attempts"],
#         "Left Memory Expirations": repeater_result["left_memory_expirations"],
#         "Right Memory Expirations": repeater_result["right_memory_expirations"],
#         "Repeater Success": repeater_result["success"],
#         "Improvement Factor": (
#             (1 / direct_link.survival_probability)
#             / repeater_result["global_attempts_required"]
#             if repeater_result["global_attempts_required"] > 0
#             else None
#         )
#     })

# df_phase2_single_repeater = pd.DataFrame(phase2_results)

# display(df_phase2_single_repeater)

# df_phase2_single_repeater.to_csv(
#     "phase2_single_repeater_results.csv",
#     index=False
# )

In [ ]:
# @title Plot : Repeater Improvement Factor

# plt.figure(figsize=(10, 6))

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Improvement Factor"],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title("Single Repeater Improvement Over Direct Transmission")
# plt.grid(True)

# plt.savefig(
#     "Phase 2 Single Repeater Improvement Over Direct Transmission.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Direct Expected Attempts vs Repeater Attempts

# plt.figure(figsize=(10, 6))

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Direct Expected Attempts"],
#     marker="o",
#     label="Direct Expected Attempts"
# )

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Repeater Global Attempts"],
#     marker="s",
#     label="Single Repeater Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Attempts")
# plt.title("Direct Transmission vs Single Repeater Attempts")
# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2 Direct vs Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2A --- Monte Carlo Trials

100 Trial runs for Phase 2A

In [ ]:
# @title Monte Carlo Experiment

# import gc
# import math
# import os
# import time
# import pandas as pd

# PHASE2A_MONTE_CARLO_TRIALS = 100
# PHASE2A_CHECKPOINT_INTERVAL = 50

# phase2a_mc_distances = [50, 80, 100, 150, 200, 250, 300]

# PHASE2A_SWAP_SUCCESS_PROBABILITY = 1.0
# PHASE2A_MEMORY_LIFETIME_ATTEMPTS = 10000
# PHASE2A_MAX_ATTEMPTS = 1000000

# PHASE2A_RAW_FILE = ("phase2a_monte_carlo_raw_100_trials.csv")

# PHASE2A_LATEST_CHECKPOINT_FILE = ("phase2a_monte_carlo_latest_checkpoint.csv")

# # Set to True only when resuming after a runtime crash.
# RESUME_PHASE2A_FROM_CHECKPOINT = False

# # =========================================================
# # Theoretical Ideal-Memory Expectation
# # =========================================================

# def expected_attempts_with_ideal_memory(
#     segment_success_probability: float
# ) -> float:
#     """
#     Expected number of parallel global attempts required
#     for two independent geometric elementary-link processes.

#     Assumptions:
#     - Equal segment success probability p
#     - Both elementary links attempted in parallel
#     - A successful Bell pair is stored indefinitely
#     - Perfect entanglement swapping
#     """

#     p = segment_success_probability

#     if p <= 0:
#         return float("inf")

#     return ((2 / p) - (1 / (p * (2 - p))))


# # =========================================================
# # Resume Support
# # =========================================================

# phase2a_raw_results = []
# phase2a_starting_trial = 1

# if (RESUME_PHASE2A_FROM_CHECKPOINT and os.path.exists(PHASE2A_LATEST_CHECKPOINT_FILE)):
#     existing_checkpoint = pd.read_csv(PHASE2A_LATEST_CHECKPOINT_FILE)

#     phase2a_raw_results = existing_checkpoint.to_dict("records")

#     if not existing_checkpoint.empty:
#         last_completed_trial = int(existing_checkpoint["Trial"].max())

#         phase2a_starting_trial = last_completed_trial + 1

#         print("Resuming Phase 2A Monte Carlo "f"from Trial {phase2a_starting_trial}.")

# else:
#     print("Starting a new Phase 2A Monte Carlo experiment.")


# # =========================================================
# # Monte Carlo Execution
# # =========================================================

# phase2a_experiment_start = time.perf_counter()

# for trial in range(phase2a_starting_trial, PHASE2A_MONTE_CARLO_TRIALS + 1):

#     trial_start = time.perf_counter()

#     print(
#         f"\n{'=' * 78}\n"
#         f"Phase 2A Monte Carlo — Trial {trial}/{PHASE2A_MONTE_CARLO_TRIALS}\n"
#         f"{'=' * 78}"
#     )

#     for distance_km in phase2a_mc_distances:

#         configuration_start = time.perf_counter()

#         direct_link = None
#         repeater_result = None

#         try:
#             direct_link = QuantumLink(source=alice, destination=bob, distance_km=distance_km)

#             segment_distance_km = (distance_km / 2)

#             segment_link = QuantumLink(source=alice, destination=r1, distance_km=segment_distance_km)

#             direct_success_probability = (direct_link.survival_probability)

#             segment_success_probability = (segment_link.survival_probability)

#             direct_expected_attempts = 1 / direct_success_probability if direct_success_probability > 0 else float("inf")

#             no_memory_success_probability = segment_success_probability * segment_success_probability * PHASE2A_SWAP_SUCCESS_PROBABILITY

#             no_memory_expected_attempts = 1 / no_memory_success_probability if no_memory_success_probability > 0 else float("inf")

#             ideal_memory_expected_attempts = expected_attempts_with_ideal_memory(segment_success_probability)

#             repeater_result = (
#                 generate_entanglement_via_single_repeater_with_memory(
#                     source=alice,
#                     repeater=r1,
#                     destination=bob,
#                     total_distance_km=distance_km,
#                     swapping_success_probability=PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                     memory_lifetime_attempts=PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                     max_attempts=PHASE2A_MAX_ATTEMPTS,
#                     verbose=False
#                 )
#             )

#             configuration_runtime = time.perf_counter() - configuration_start

#             observed_global_attempts = repeater_result["global_attempts_required"]

#             observed_improvement_factor = direct_expected_attempts / observed_global_attempts if observed_global_attempts > 0 else None

#             theoretical_improvement_factor = direct_expected_attempts / ideal_memory_expected_attempts if ideal_memory_expected_attempts > 0 else None

#             phase2a_raw_results.append({
#                 "Trial":trial,
#                 "Distance (km)":distance_km,
#                 "Segment Distance (km)":segment_distance_km,
#                 "Direct Survival Probability (%)":direct_success_probability * 100,
#                 "Segment Survival Probability (%)":segment_success_probability * 100,
#                 "Swap Success Probability":PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                 "Memory Lifetime (Attempts)":PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                 "Maximum Allowed Attempts":PHASE2A_MAX_ATTEMPTS,

#                 # Theoretical baseline metrics
#                 "Direct Expected Attempts":direct_expected_attempts,
#                 "No-Memory Repeater Expected Attempts":no_memory_expected_attempts,
#                 "Ideal-Memory Repeater Expected Attempts":ideal_memory_expected_attempts,
#                 "Theoretical Improvement Factor":theoretical_improvement_factor,

#                 # Observed repeater metrics
#                 "Repeater Global Attempts":observed_global_attempts,
#                 "Left Link Attempts": repeater_result["left_attempts"],
#                 "Right Link Attempts": repeater_result["right_attempts"],
#                 "Total Elementary-Link Attempts": repeater_result["left_attempts"] + repeater_result["right_attempts"],

#                 "Swap Attempts": repeater_result["swap_attempts"],
#                 "Left Memory Expirations": repeater_result["left_memory_expirations"],
#                 "Right Memory Expirations": repeater_result["right_memory_expirations"],
#                 "Total Memory Expirations": repeater_result["left_memory_expirations"] + repeater_result["right_memory_expirations"],

#                 "Observed Improvement Factor":observed_improvement_factor,
#                 "Final Bell Pair": repeater_result["final_bell_pair"],
#                 "Final Fidelity": repeater_result["final_fidelity"],
#                 "Simulation Function Time (s)": repeater_result["time_s"],
#                 "Configuration Runtime (s)":configuration_runtime,
#                 "Status": "SUCCESS" if repeater_result["success"] else "FAILED",

#                 "Error": None
#             })

#             print(
#                 f"Trial {trial:3d} | "
#                 f"{distance_km:3d} km | "
#                 f"direct E[T]="
#                 f"{direct_expected_attempts:,.2f} | "
#                 f"ideal-memory E[T]="
#                 f"{ideal_memory_expected_attempts:,.2f} | "
#                 f"observed={observed_global_attempts:,} | "
#                 f"improvement="
#                 f"{observed_improvement_factor:,.2f}×"
#             )

#         except Exception as error:

#             configuration_runtime = time.perf_counter() - configuration_start
#             segment_distance_km = distance_km / 2
#             phase2a_raw_results.append({
#                 "Trial": trial,
#                 "Distance (km)": distance_km,
#                 "Segment Distance (km)": segment_distance_km,
#                 "Direct Survival Probability (%)": direct_link.survival_probability * 100 if direct_link is not None else None,
#                 "Segment Survival Probability (%)": None,
#                 "Swap Success Probability": PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                 "Memory Lifetime (Attempts)": PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                 "Maximum Allowed Attempts": PHASE2A_MAX_ATTEMPTS,
#                 "Direct Expected Attempts": None,
#                 "No-Memory Repeater Expected Attempts": None,
#                 "Ideal-Memory Repeater Expected Attempts": None,
#                 "Theoretical Improvement Factor": None,
#                 "Repeater Global Attempts": None,
#                 "Left Link Attempts": None,
#                 "Right Link Attempts": None,
#                 "Total Elementary-Link Attempts": None,
#                 "Swap Attempts": None,
#                 "Left Memory Expirations": None,
#                 "Right Memory Expirations": None,
#                 "Total Memory Expirations": None,
#                 "Observed Improvement Factor": None,
#                 "Final Bell Pair": None,
#                 "Final Fidelity": None,
#                 "Simulation Function Time (s)": None,
#                 "Configuration Runtime (s)": configuration_runtime,
#                 "Status": "FAILED",
#                 "Error": str(error)
#             })

#             print(
#                 f"Trial {trial:3d} | "
#                 f"{distance_km:3d} km | "
#                 f"FAILED: {error}"
#             )

#         del direct_link
#         del repeater_result

#         if "segment_link" in locals():
#             del segment_link

#         gc.collect()

#     trial_runtime = (
#         time.perf_counter()
#         - trial_start
#     )

#     print(
#         f"\nTrial {trial} completed in "
#         f"{trial_runtime:.3f} seconds."
#     )

#     # =====================================================
#     # Checkpoint Every 50 Fully Completed Trials
#     # =====================================================

#     if (trial % PHASE2A_CHECKPOINT_INTERVAL == 0 or trial == PHASE2A_MONTE_CARLO_TRIALS):
#         checkpoint_df = pd.DataFrame(phase2a_raw_results)

#         numbered_checkpoint_file = (f"phase2a_monte_carlo_checkpoint_trial_{trial}.csv")

#         checkpoint_df.to_csv(numbered_checkpoint_file, index=False)

#         checkpoint_df.to_csv(PHASE2A_LATEST_CHECKPOINT_FILE, index=False)

#         print(
#             f"\nCheckpoint saved after Trial {trial}:"
#             f"\n  {numbered_checkpoint_file}"
#             f"\n  {PHASE2A_LATEST_CHECKPOINT_FILE}"
#             f"\n  Rows saved: {len(checkpoint_df):,}"
#         )


# # =========================================================
# # Save Final Raw Dataset
# # =========================================================

# df_phase2a_monte_carlo_raw = pd.DataFrame(phase2a_raw_results)

# df_phase2a_monte_carlo_raw.to_csv(PHASE2A_RAW_FILE, index=False)

# phase2a_total_runtime = time.perf_counter() - phase2a_experiment_start

# expected_phase2a_rows = (PHASE2A_MONTE_CARLO_TRIALS * len(phase2a_mc_distances))

# print(
#     f"\n{'=' * 78}\n"
#     f"PHASE 2A MONTE CARLO COMPLETE\n"
#     f"{'=' * 78}\n"
#     f"Trials completed       : "
#     f"{PHASE2A_MONTE_CARLO_TRIALS}\n"
#     f"Distances per trial    : "
#     f"{len(phase2a_mc_distances)}\n"
#     f"Raw rows generated     : "
#     f"{len(df_phase2a_monte_carlo_raw):,}\n"
#     f"Expected raw rows      : "
#     f"{expected_phase2a_rows:,}\n"
#     f"Total runtime          : "
#     f"{phase2a_total_runtime / 60:.2f} minutes\n"
#     f"Final raw CSV          : "
#     f"{PHASE2A_RAW_FILE}\n"
#     f"{'=' * 78}"
# )

# display(
#     df_phase2a_monte_carlo_raw.head(14)
# )

In [ ]:
# @title Aggregate Statistics

# import pandas as pd

# df_phase2a_monte_carlo_raw = pd.read_csv("phase2a_monte_carlo_raw_100_trials.csv")

# successful_phase2a_runs = df_phase2a_monte_carlo_raw[df_phase2a_monte_carlo_raw["Status"] == "SUCCESS"].copy()

# phase2a_group_columns = "Distance (km)"

# phase2a_metrics_to_aggregate = [
#     "Segment Distance (km)",
#     "Direct Survival Probability (%)",
#     "Segment Survival Probability (%)",

#     "Direct Expected Attempts",
#     "No-Memory Repeater Expected Attempts",
#     "Ideal-Memory Repeater Expected Attempts",
#     "Theoretical Improvement Factor",

#     "Repeater Global Attempts",
#     "Left Link Attempts",
#     "Right Link Attempts",
#     "Total Elementary-Link Attempts",
#     "Swap Attempts",

#     "Left Memory Expirations",
#     "Right Memory Expirations",
#     "Total Memory Expirations",

#     "Observed Improvement Factor",
#     "Final Fidelity",

#     "Simulation Function Time (s)",
#     "Configuration Runtime (s)"
# ]

# phase2a_summary_statistics = (
#     successful_phase2a_runs
#     .groupby(
#         phase2a_group_columns,
#         as_index=False
#     )[phase2a_metrics_to_aggregate]
#     .agg([
#         "count",
#         "mean",
#         "std",
#         "median",
#         "min",
#         "max"
#     ])
# )

# # Flatten Pandas MultiIndex columns.
# phase2a_summary_statistics.columns = [
#     " ".join(
#         str(part)
#         for part in column
#         if str(part) != ""
#     ).strip()
#     if isinstance(column, tuple)
#     else column
#     for column in phase2a_summary_statistics.columns
# ]


# # =========================================================
# # Trial Counts and Success Rate
# # =========================================================

# phase2a_run_counts = (
#     df_phase2a_monte_carlo_raw
#     .groupby(
#         phase2a_group_columns
#     )
#     .agg(
#         Total_Trials=(
#             "Trial",
#             "count"
#         ),

#         Successful_Trials=(
#             "Status",
#             lambda values: (
#                 values == "SUCCESS"
#             ).sum()
#         ),

#         Failed_Trials=(
#             "Status",
#             lambda values: (
#                 values == "FAILED"
#             ).sum()
#         )
#     )
#     .reset_index()
# )

# phase2a_run_counts["Success Rate (%)"] = phase2a_run_counts["Successful_Trials"] / phase2a_run_counts["Total_Trials"] * 100

# df_phase2a_monte_carlo_summary = (
#     phase2a_run_counts.merge(
#         phase2a_summary_statistics,
#         on=phase2a_group_columns,
#         how="left"
#     )
# )

# df_phase2a_monte_carlo_summary = df_phase2a_monte_carlo_summary.sort_values("Distance (km)").reset_index(drop=True)

# PHASE2A_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# df_phase2a_monte_carlo_summary.to_csv(PHASE2A_SUMMARY_FILE, index=False)

# print(f"Phase 2A aggregate summary saved to {PHASE2A_SUMMARY_FILE}")

# print(f"Raw rows       : {len(df_phase2a_monte_carlo_raw):,}")

# print(f"Summary rows   : {len(df_phase2a_monte_carlo_summary):,}")

# print(f"Failed runs    : {(df_phase2a_monte_carlo_raw['Status']== 'FAILED').sum():,}")

# display(df_phase2a_monte_carlo_summary)

In [ ]:
# @title Add Aggregate Improvement and Variability Metrics

# import numpy as np
# import pandas as pd

# PHASE2A_RAW_FILE = "phase2a_monte_carlo_raw_100_trials.csv"

# PHASE2A_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# PHASE2A_AUGMENTED_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# df_phase2a_raw = pd.read_csv(PHASE2A_RAW_FILE)

# df_phase2a_summary = pd.read_csv(PHASE2A_SUMMARY_FILE)

# df_phase2a_successful = df_phase2a_raw[df_phase2a_raw["Status"] == "SUCCESS"].copy()

# phase2a_additional_metrics = (
#     df_phase2a_successful
#     .groupby(
#         "Distance (km)",
#         as_index=False
#     )
#     .agg(
#         Direct_Expected_Attempts_Aggregate=("Direct Expected Attempts", "mean"),
#         Repeater_Global_Attempts_Aggregate=("Repeater Global Attempts", "mean"),
#         Repeater_Global_Attempts_SD=("Repeater Global Attempts", "std"),
#         Repeater_Global_Attempts_Median=("Repeater Global Attempts", "median")
#     )
# )

# phase2a_additional_metrics["Aggregate Improvement Factor"] = phase2a_additional_metrics["Direct_Expected_Attempts_Aggregate"] / phase2a_additional_metrics["Repeater_Global_Attempts_Aggregate"]

# phase2a_additional_metrics["Repeater Attempts Coefficient of Variation"] = phase2a_additional_metrics["Repeater_Global_Attempts_SD"] / phase2a_additional_metrics["Repeater_Global_Attempts_Aggregate"]

# phase2a_additional_metrics["Repeater Attempts Coefficient of Variation (%)"] = phase2a_additional_metrics["Repeater Attempts Coefficient of Variation"] * 100

# if ("Theoretical Improvement Factor mean" in df_phase2a_summary.columns):

#     theoretical_lookup = df_phase2a_summary[["Distance (km)", "Theoretical Improvement Factor mean"]].copy()

#     phase2a_additional_metrics = phase2a_additional_metrics.merge(theoretical_lookup, on="Distance (km)", how="left")

#     phase2a_additional_metrics["Aggregate vs Theory Difference"] = phase2a_additional_metrics["Aggregate Improvement Factor"] - phase2a_additional_metrics["Theoretical Improvement Factor mean"]

#     phase2a_additional_metrics["Aggregate vs Theory Error (%)"] = phase2a_additional_metrics["Aggregate vs Theory Difference"] / phase2a_additional_metrics["Theoretical Improvement Factor mean"] * 100

# columns_to_merge = [
#     "Distance (km)",
#     "Aggregate Improvement Factor",
#     "Repeater Attempts Coefficient of Variation",
#     "Repeater Attempts Coefficient of Variation (%)"
# ]

# if ("Aggregate vs Theory Difference" in phase2a_additional_metrics.columns):
#     columns_to_merge.extend([
#         "Aggregate vs Theory Difference",
#         "Aggregate vs Theory Error (%)"
#     ])

# df_phase2a_summary_augmented = (
#     df_phase2a_summary
#     .merge(
#         phase2a_additional_metrics[
#             columns_to_merge
#         ],
#         on="Distance (km)",
#         how="left"
#     )
#     .sort_values("Distance (km)")
#     .reset_index(drop=True)
# )


# # =========================================================
# # Save Augmented Full Summary
# # =========================================================

# df_phase2a_summary_augmented.to_csv(PHASE2A_AUGMENTED_SUMMARY_FILE, index=False)

# print(f"Augmented Phase 2A summary saved to : {PHASE2A_AUGMENTED_SUMMARY_FILE}")

# display(df_phase2a_summary_augmented)

In [ ]:
# @title Compact Paper-Ready Summary

# phase2a_paper_columns = [
#     "Distance (km)",

#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",

#     "Direct Survival Probability (%) mean",
#     "Segment Survival Probability (%) mean",

#     "Direct Expected Attempts mean",

#     "Ideal-Memory Repeater Expected Attempts mean",

#     "Repeater Global Attempts mean",
#     "Repeater Global Attempts std",
#     "Repeater Global Attempts median",
#     "Repeater Global Attempts min",
#     "Repeater Global Attempts max",

#     "Left Link Attempts mean",
#     "Left Link Attempts std",

#     "Right Link Attempts mean",
#     "Right Link Attempts std",

#     "Total Elementary-Link Attempts mean",
#     "Total Elementary-Link Attempts std",

#     "Observed Improvement Factor mean",
#     "Observed Improvement Factor std",

#     "Theoretical Improvement Factor mean",

#     "Total Memory Expirations mean",
#     "Total Memory Expirations std",

#     "Configuration Runtime (s) mean",
#     "Configuration Runtime (s) std"
# ]

# df_phase2a_paper_summary = df_phase2a_monte_carlo_summary[phase2a_paper_columns].copy()

# PHASE2A_PAPER_SUMMARY_FILE = "phase2a_monte_carlo_paper_summary_100_trials.csv"

# df_phase2a_paper_summary.to_csv(PHASE2A_PAPER_SUMMARY_FILE, index=False)

# print(f"Phase 2A paper-ready summary saved to {PHASE2A_PAPER_SUMMARY_FILE}")

# display(df_phase2a_paper_summary)

In [ ]:
# @title Create Augmented Paper-Ready Summary

# import pandas as pd

# PHASE2A_AUGMENTED_SUMMARY_FILE = ("phase2a_monte_carlo_summary_100_trials.csv")

# PHASE2A_AUGMENTED_PAPER_FILE = ("phase2a_monte_carlo_paper_summary_100_trials.csv")

# df_phase2a_summary_augmented = pd.read_csv(PHASE2A_AUGMENTED_SUMMARY_FILE)

# phase2a_augmented_paper_columns = [
#     "Distance (km)",
#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",
#     "Direct Survival Probability (%) mean",
#     "Segment Survival Probability (%) mean",
#     "Direct Expected Attempts mean",
#     "Ideal-Memory Repeater Expected Attempts mean",
#     "Repeater Global Attempts mean",
#     "Repeater Global Attempts std",
#     "Repeater Global Attempts median",
#     "Repeater Global Attempts min",
#     "Repeater Global Attempts max",
#     "Observed Improvement Factor mean",
#     "Observed Improvement Factor std",
#     "Aggregate Improvement Factor",
#     "Theoretical Improvement Factor mean",
#     "Repeater Attempts Coefficient of Variation",
#     "Repeater Attempts Coefficient of Variation (%)",
#     "Left Link Attempts mean",
#     "Right Link Attempts mean",
#     "Total Elementary-Link Attempts mean",
#     "Total Memory Expirations mean",
#     "Configuration Runtime (s) mean"
# ]

# optional_columns = [
#     "Aggregate vs Theory Difference",
#     "Aggregate vs Theory Error (%)"
# ]

# for column in optional_columns:
#     if column in df_phase2a_summary_augmented.columns:
#         phase2a_augmented_paper_columns.append(
#             column
#         )

# available_paper_columns = [
#     column
#     for column
#     in phase2a_augmented_paper_columns
#     if column
#     in df_phase2a_summary_augmented.columns
# ]


# df_phase2a_paper_summary_augmented = df_phase2a_summary_augmented[available_paper_columns].copy()

# df_phase2a_paper_summary_augmented.to_csv(
#     PHASE2A_AUGMENTED_PAPER_FILE,
#     index=False
# )


# print(
#     f"Augmented Phase 2A paper summary saved to : {PHASE2A_AUGMENTED_PAPER_FILE}"
# )

# display(df_phase2a_paper_summary_augmented)

In [ ]:
# @title Monte Carlo Paper-Ready Graphs

# import matplotlib.pyplot as plt
# import pandas as pd


# phase2a_summary_df = pd.read_csv("phase2a_monte_carlo_summary_100_trials.csv")

# phase2a_summary_df = phase2a_summary_df.sort_values("Distance (km)")


# # =========================================================
# # Graph 1:
# # Direct Expected Attempts vs Observed Repeater Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Direct Expected Attempts mean"
#     ],
#     marker="o",
#     label="Direct Expected Attempts"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Repeater Global Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label=(
#         "Single Repeater Observed Attempts "
#         "(Mean ± SD)"
#     )
# )

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Ideal-Memory Repeater "
#         "Expected Attempts mean"
#     ],
#     marker="^",
#     linestyle="--",
#     label=(
#         "Ideal-Memory Theoretical Expectation"
#     )
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Attempts")
# plt.title(
#     "Direct Transmission vs "
#     "Memory-Assisted Single Repeater "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Direct vs Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 2:
# # Distance vs Observed Improvement Factor
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Observed Improvement Factor mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Observed Improvement Factor std"
#     ],
#     marker="o",
#     capsize=4,
#     label=(
#         "Observed Improvement "
#         "(Mean ± SD)"
#     )
# )

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Theoretical Improvement Factor mean"
#     ],
#     marker="s",
#     linestyle="--",
#     label="Ideal-Memory Theoretical Improvement"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title(
#     "Single-Repeater Improvement "
#     "Over Direct Transmission "
#     "(100 Trials)"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Improvement Factor.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 3:
# # Left vs Right Elementary-Link Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Left Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Left Link Attempts std"
#     ],
#     marker="o",
#     capsize=4,
#     label="Left Elementary Link"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Right Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Right Link Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label="Right Elementary Link"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Attempts")
# plt.title(
#     "Elementary-Link Attempt Balance "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Left vs Right Link Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 4:
# # Global Attempts vs Total Elementary-Link Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Repeater Global Attempts std"
#     ],
#     marker="o",
#     capsize=4,
#     label="Global Time Slots"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Total Elementary-Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Total Elementary-Link Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label="Total Elementary-Link Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Attempts")
# plt.title(
#     "Global Latency Cost vs "
#     "Physical Link-Attempt Cost "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Global vs Elementary Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 5:
# # Memory Expirations
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Total Memory Expirations mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Total Memory Expirations std"
#     ],
#     marker="o",
#     capsize=4
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Memory Expirations")
# plt.title(
#     "Distance vs Quantum-Memory Expirations "
#     "(100 Trials)"
# )

# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Distance vs Memory Expirations.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 6:
# # Session Success Rate
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Success Rate (%)"
#     ],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Successful Repeater Sessions (%)")
# plt.title(
#     "Distance vs Single-Repeater "
#     "Session Success Rate "
#     "(100 Trials)"
# )

# plt.ylim(0, 105)
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Distance vs Success Rate.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Aggregate Improvement Factor

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv("phase2a_monte_carlo_summary_100_trials.csv")

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Aggregate Improvement Factor"
#     ],
#     marker="o",
#     label="Aggregate Improvement Factor"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title(
#     "Single-Repeater Aggregate Improvement "
#     "Over Direct Transmission"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Aggregate Improvement Factor.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Improvement Metric Comparison

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Observed Improvement Factor mean"
#     ],
#     yerr=summary_df[
#         "Observed Improvement Factor std"
#     ],
#     marker="o",
#     capsize=4,
#     label=(
#         "Mean of Trial-Wise Ratios "
#         "(Mean ± SD)"
#     )
# )


# # New, stable aggregate metric

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Aggregate Improvement Factor"
#     ],
#     marker="s",
#     linestyle="--",
#     label=(
#         "Ratio of Aggregate Means"
#     )
# )


# # Theoretical ideal-memory result

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Theoretical Improvement Factor mean"
#     ],
#     marker="^",
#     linestyle=":",
#     label=(
#         "Ideal-Memory Theoretical Improvement"
#     )
# )


# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")

# plt.title(
#     "Comparison of Single-Repeater "
#     "Improvement Metrics"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Improvement Metric Comparison.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Repeater Attempt Variability

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Attempts Coefficient of Variation (%)"
#     ],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Coefficient of Variation (%)")

# plt.title(
#     "Relative Variability of "
#     "Repeater Establishment Attempts"
# )

# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Attempts Coefficient of Variation.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Mean vs Median Repeater Attempts

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     marker="o",
#     label="Mean Global Attempts"
# )

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Global Attempts median"
#     ],
#     marker="s",
#     label="Median Global Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Repeater Global Attempts")

# plt.title(
#     "Mean vs Median Repeater "
#     "Establishment Attempts"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Mean vs Median Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Trial Distribution of Repeater Attempts

# import matplotlib.pyplot as plt
# import pandas as pd


# raw_df = pd.read_csv(
#     "phase2a_monte_carlo_raw_100_trials.csv"
# )

# successful_raw_df = (
#     raw_df[
#         raw_df["Status"] == "SUCCESS"
#     ]
#     .copy()
# )


# ordered_distances = sorted(
#     successful_raw_df[
#         "Distance (km)"
#     ].unique()
# )


# boxplot_data = [
#     successful_raw_df[
#         successful_raw_df[
#             "Distance (km)"
#         ] == distance
#     ][
#         "Repeater Global Attempts"
#     ].dropna().values

#     for distance in ordered_distances
# ]


# plt.figure(figsize=(11, 6))

# plt.boxplot(
#     boxplot_data,
#     labels=[
#         str(distance)
#         for distance
#         in ordered_distances
#     ],
#     showfliers=True
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Repeater Global Attempts")

# plt.title(
#     "Distribution of Single-Repeater "
#     "Establishment Attempts Across 100 Trials"
# )

# plt.yscale("log")
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Attempt Distribution Boxplot.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2B

Repeater Assisted QKD Protocols --> AES Key Generation

In [ ]:
# @title Physical Bell Pair Generation

def attempt_bell_pair_generation_physical(
    link: QuantumLink,
    source_fidelity: float = 1.0
):
    """
    Attempts Bell-pair generation over a physical QuantumLink.

    This uses QuantumLink.transmit(), so fiber attenuation and photon loss
    are recorded in the physical layer statistics.
    """

    received = link.transmit("photon")

    if received is not None:
        return BellPair(
            node_a=link.source,
            node_b=link.destination,
            fidelity=source_fidelity,
            source_link_distance_km=link.distance_km
        )

    return None

In [ ]:
# @title Repeater-Assisted Quantum Channel With Layered Statistics

class RepeaterAssistedQuantumChannel:
    """
    Repeater-assisted quantum channel.

    Physical Layer:
    - photons emitted
    - photons survived/lost
    - Bell-pair generation attempts
    - memory expirations
    - entanglement swapping attempts

    Quantum Protocol Layer:
    - qubits requested
    - qubits delivered
    - qubits lost at protocol level
    """

    def __init__(
        self,
        source: Node,
        repeater: Node,
        destination: Node,
        total_distance_km: float,
        swapping_success_probability: float = 1.0,
        memory_lifetime_attempts: int = 10000,
        max_entanglement_attempts: int = 1000000
    ):
        self.source = source
        self.repeater = repeater
        self.destination = destination

        self.total_distance_km = total_distance_km
        self.segment_distance_km = total_distance_km / 2

        self.swapping_success_probability = swapping_success_probability
        self.memory_lifetime_attempts = memory_lifetime_attempts
        self.max_entanglement_attempts = max_entanglement_attempts

        self.left_link = QuantumLink(source, repeater, self.segment_distance_km)
        self.right_link = QuantumLink(repeater, destination, self.segment_distance_km)
        self.direct_link = QuantumLink(source, destination, total_distance_km)

        self.physical_stats = {
            "left_photons_sent": 0,
            "left_photons_survived": 0,
            "left_photons_lost": 0,

            "right_photons_sent": 0,
            "right_photons_survived": 0,
            "right_photons_lost": 0,

            "left_bell_pair_attempts": 0,
            "right_bell_pair_attempts": 0,

            "successful_left_bell_pairs": 0,
            "successful_right_bell_pairs": 0,
            "successful_end_to_end_bell_pairs": 0,

            "swap_attempts": 0,

            "memory_expirations_left": 0,
            "memory_expirations_right": 0,

            "total_entanglement_attempts": 0
        }

        self.protocol_stats = {
            "qubits_requested": 0,
            "qubits_delivered": 0,
            "qubits_lost": 0
        }

    def _sync_physical_stats_from_links(self):
        self.physical_stats["left_photons_sent"] = self.left_link.statistics["lifetime_sent"]
        self.physical_stats["left_photons_survived"] = self.left_link.statistics["lifetime_received"]
        self.physical_stats["left_photons_lost"] = self.left_link.statistics["lifetime_lost"]

        self.physical_stats["right_photons_sent"] = self.right_link.statistics["lifetime_sent"]
        self.physical_stats["right_photons_survived"] = self.right_link.statistics["lifetime_received"]
        self.physical_stats["right_photons_lost"] = self.right_link.statistics["lifetime_lost"]

        self.physical_stats["total_entanglement_attempts"] = (
            self.physical_stats["left_bell_pair_attempts"]
            + self.physical_stats["right_bell_pair_attempts"]
        )

    def generate_end_to_end_bell_pair(self):
        left_pair = None
        right_pair = None

        left_memory_age = 0
        right_memory_age = 0

        for global_attempt in range(1, self.max_entanglement_attempts + 1):

            if left_pair is not None:
                left_memory_age += 1
                if left_memory_age > self.memory_lifetime_attempts:
                    left_pair = None
                    left_memory_age = 0
                    self.physical_stats["memory_expirations_left"] += 1

            if right_pair is not None:
                right_memory_age += 1
                if right_memory_age > self.memory_lifetime_attempts:
                    right_pair = None
                    right_memory_age = 0
                    self.physical_stats["memory_expirations_right"] += 1

            if left_pair is None:
                self.physical_stats["left_bell_pair_attempts"] += 1
                generated_left = attempt_bell_pair_generation_physical(self.left_link)

                if generated_left is not None:
                    left_pair = generated_left
                    left_memory_age = 0
                    self.physical_stats["successful_left_bell_pairs"] += 1

            if right_pair is None:
                self.physical_stats["right_bell_pair_attempts"] += 1
                generated_right = attempt_bell_pair_generation_physical(self.right_link)

                if generated_right is not None:
                    right_pair = generated_right
                    right_memory_age = 0
                    self.physical_stats["successful_right_bell_pairs"] += 1

            if left_pair is not None and right_pair is not None:
                self.physical_stats["swap_attempts"] += 1

                end_to_end_pair = entanglement_swap(
                    left_pair=left_pair,
                    right_pair=right_pair,
                    repeater=self.repeater,
                    swapping_success_probability=self.swapping_success_probability
                )

                if end_to_end_pair is not None:
                    self.physical_stats["successful_end_to_end_bell_pairs"] += 1
                    self._sync_physical_stats_from_links()

                    return {
                        "success": True,
                        "end_to_end_pair": end_to_end_pair,
                        "global_attempts_required": global_attempt
                    }

                left_pair = None
                right_pair = None
                left_memory_age = 0
                right_memory_age = 0

        self._sync_physical_stats_from_links()

        return {
            "success": False,
            "end_to_end_pair": None,
            "global_attempts_required": self.max_entanglement_attempts
        }

    def transmit(self, qubit):
        self.protocol_stats["qubits_requested"] += 1

        result = self.generate_end_to_end_bell_pair()

        if result["success"]:
            self.protocol_stats["qubits_delivered"] += 1
            return qubit

        self.protocol_stats["qubits_lost"] += 1
        return None

    def transmit_batch(self, qubits: list):
        transmitted_qubits = [self.transmit(q) for q in qubits]

        sent = len(qubits)
        delivered = sum(q is not None for q in transmitted_qubits)
        lost = sent - delivered

        self._sync_physical_stats_from_links()

        avg_attempts_per_delivered_qubit = (
            self.physical_stats["total_entanglement_attempts"] / delivered
            if delivered else None
        )

        return {
            "transmitted_qubits": transmitted_qubits,
            "sent": sent,
            "survived": delivered,
            "lost": lost,
            "observed_survival_probability": delivered / sent if sent else 0,
            "physical_stats": self.physical_stats.copy(),
            "protocol_stats": self.protocol_stats.copy(),
            "average_entanglement_attempts_per_delivered_qubit": avg_attempts_per_delivered_qubit
        }

In [ ]:
# @title BB84 Round Over Repeater-Assisted Channel

def bb84_round_over_repeater_channel(
    channel: RepeaterAssistedQuantumChannel,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5
) -> dict:

    p_basis = basis_match_probability(z_probability)

    # Important:
    # In repeater-assisted mode, fiber attenuation is paid inside Bell-pair generation.
    # Therefore protocol qubit budget depends only on basis matching.
    estimated_qubits_required = math.ceil(
        (target_bits * transmission_margin) / p_basis
    )

    alice_bits = random_bitstring(estimated_qubits_required)
    alice_bases = random_bases(estimated_qubits_required, z_probability=z_probability)
    bob_bases = random_bases(estimated_qubits_required, z_probability=z_probability)

    encoded_qubits = encode_qubits(alice_bits, alice_bases)

    transmission = channel.transmit_batch(encoded_qubits)
    received_qubits = transmission["transmitted_qubits"]

    bob_bits = measure_qubits(
        received_qubits,
        bob_bases,
        eve_present=eve_present
    )

    alice_sifted, bob_sifted = sift_keys(
        alice_bits,
        alice_bases,
        bob_bits,
        bob_bases
    )

    qber = calculate_qber(alice_sifted, bob_sifted)
    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,

        "z_probability": z_probability,
        "basis_match_probability": p_basis,
        "estimated_qubits_required": estimated_qubits_required,

        "qubits_sent": transmission["sent"],
        "qubits_received": transmission["survived"],
        "qubits_lost": transmission["lost"],

        "physical_stats": transmission["physical_stats"],
        "protocol_stats": transmission["protocol_stats"],
        "average_entanglement_attempts_per_delivered_qubit":
            transmission["average_entanglement_attempts_per_delivered_qubit"]
    }

In [ ]:
# @title Generate Session Key Over Repeater-Assisted Channel

def generate_bb84_session_key_over_repeater_channel(
    channel: RepeaterAssistedQuantumChannel,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    max_rounds: int = 100,
    eve_present: bool = False,
    z_probability: float = 0.5,
    verbose: bool = True
):

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):

        remaining_bits = target_bits - len(collected_bits)

        result = bb84_round_over_repeater_channel(
            channel=channel,
            target_bits=remaining_bits,
            transmission_margin=transmission_margin,
            eve_present=eve_present,
            z_probability=z_probability
        )

        round_logs.append({
            "round": round_id,
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"],
            "qubits_sent": result["qubits_sent"],
            "qubits_received": result["qubits_received"],
            "qubits_lost": result["qubits_lost"],
            "average_entanglement_attempts_per_delivered_qubit":
                result["average_entanglement_attempts_per_delivered_qubit"]
        })

        if verbose:
            ps = result["physical_stats"]
            avg_attempts = result["average_entanglement_attempts_per_delivered_qubit"]

            print(
                f"Round {round_id}: "
                f"sent={result['qubits_sent']} | "
                f"received={result['qubits_received']} | "
                f"sifted={result['sifted_length']} | "
                f"QBER={result['qber']:.2%} | "
                f"accepted={result['accepted']} | "
                f"entanglement_attempts={ps['total_entanglement_attempts']} | "
                f"avg_attempts/qubit={avg_attempts:.2f}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough key bits over repeater-assisted channel."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    total_qubits_sent = sum(r["qubits_sent"] for r in round_logs)
    total_qubits_received = sum(r["qubits_received"] for r in round_logs)
    total_qubits_lost = sum(r["qubits_lost"] for r in round_logs)

    ps = channel.physical_stats
    qs = channel.protocol_stats

    total_photons_sent_physical = ps["left_photons_sent"] + ps["right_photons_sent"]
    total_photons_survived_physical = ps["left_photons_survived"] + ps["right_photons_survived"]
    total_photons_lost_physical = ps["left_photons_lost"] + ps["right_photons_lost"]

    metadata = {
        "source": channel.source.name,
        "repeater": channel.repeater.name,
        "destination": channel.destination.name,

        "total_distance_km": channel.total_distance_km,
        "segment_distance_km": channel.segment_distance_km,

        "target_bits": target_bits,
        "z_probability": z_probability,
        "basis_match_probability": basis_match_probability(z_probability),

        "rounds_used": len(round_logs),

        # Protocol layer
        "total_qubits_sent": total_qubits_sent,
        "total_qubits_received": total_qubits_received,
        "total_qubits_lost": total_qubits_lost,
        "effective_key_yield": target_bits / total_qubits_sent if total_qubits_sent else 0,
        "keygen_time_s": elapsed,

        # Physical layer
        "left_photons_sent": ps["left_photons_sent"],
        "left_photons_survived": ps["left_photons_survived"],
        "left_photons_lost": ps["left_photons_lost"],

        "right_photons_sent": ps["right_photons_sent"],
        "right_photons_survived": ps["right_photons_survived"],
        "right_photons_lost": ps["right_photons_lost"],

        "total_photons_sent_physical": total_photons_sent_physical,
        "total_photons_survived_physical": total_photons_survived_physical,
        "total_photons_lost_physical": total_photons_lost_physical,

        "total_entanglement_attempts": ps["total_entanglement_attempts"],
        "average_entanglement_attempts_per_delivered_qubit":
            ps["total_entanglement_attempts"] / total_qubits_received
            if total_qubits_received else None,

        "successful_end_to_end_bell_pairs": ps["successful_end_to_end_bell_pairs"],
        "swap_attempts": ps["swap_attempts"],
        "memory_expirations_left": ps["memory_expirations_left"],
        "memory_expirations_right": ps["memory_expirations_right"],

        "protocol_qubits_requested": qs["qubits_requested"],
        "protocol_qubits_delivered": qs["qubits_delivered"],
        "protocol_qubits_lost": qs["qubits_lost"],

        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title Fixed Distance Protocol Comparison

# test_distance = 100
# target_bits = 128
# transmission_margin = 1.3

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# phase2b_fixed_results = []

# for protocol_name, z_prob in protocols.items():

#     channel = RepeaterAssistedQuantumChannel(
#         source=alice,
#         repeater=r1,
#         destination=bob,
#         total_distance_km=test_distance,
#         swapping_success_probability=1.0,
#         memory_lifetime_attempts=10000,
#         max_entanglement_attempts=1000000
#     )

#     key, meta = generate_bb84_session_key_over_repeater_channel(
#         channel=channel,
#         target_bits=target_bits,
#         transmission_margin=transmission_margin,
#         max_rounds=100,
#         eve_present=False,
#         z_probability=z_prob,
#         verbose=False
#     )

#     phase2b_fixed_results.append({
#         "Protocol": protocol_name,
#         "Distance (km)": test_distance,
#         "Segment Distance (km)": meta["segment_distance_km"],

#         "Z Probability": z_prob,
#         "Basis Match Probability (%)": meta["basis_match_probability"] * 100,

#         "Rounds Used": meta["rounds_used"],

#         # Protocol layer
#         "Qubits Sent": meta["total_qubits_sent"],
#         "Qubits Received": meta["total_qubits_received"],
#         "Qubits Lost Protocol": meta["total_qubits_lost"],
#         "Effective Key Yield (%)": meta["effective_key_yield"] * 100,

#         # Physical layer
#         "Left Photons Sent": meta["left_photons_sent"],
#         "Left Photons Survived": meta["left_photons_survived"],
#         "Left Photons Lost": meta["left_photons_lost"],

#         "Right Photons Sent": meta["right_photons_sent"],
#         "Right Photons Survived": meta["right_photons_survived"],
#         "Right Photons Lost": meta["right_photons_lost"],

#         "Total Physical Photons Sent": meta["total_photons_sent_physical"],
#         "Total Physical Photons Survived": meta["total_photons_survived_physical"],
#         "Total Physical Photons Lost": meta["total_photons_lost_physical"],

#         "Entanglement Attempts": meta["total_entanglement_attempts"],
#         "Avg Entanglement Attempts / Delivered Qubit":
#             meta["average_entanglement_attempts_per_delivered_qubit"],

#         "Successful End-to-End Bell Pairs": meta["successful_end_to_end_bell_pairs"],
#         "Swap Attempts": meta["swap_attempts"],

#         "Keygen Time (s)": meta["keygen_time_s"]
#     })

# df_phase2b_fixed = pd.DataFrame(phase2b_fixed_results)

# df_phase2b_fixed.to_csv(
#     "phase2b_key_generation_at_100_km_with_repeater.csv",
#     index=False
# )

# display(df_phase2b_fixed)

### Phase 2C

Single repeater assisted distances and non ideal fidelity and non ideal entanglement swapping

In [ ]:
# @title Ideal Single-Repeater Distance Sweep

# import time
# import pandas as pd

# # =========================================================
# # Experiment Configuration
# # =========================================================

# phase2c_distances = [10, 50, 80, 100, 150, 200]

# phase2c_target_bits = 128
# phase2c_transmission_margin = 1.3
# phase2c_max_rounds = 100

# # Ideal repeater baseline
# phase2c_swapping_success_probability = 1.0
# phase2c_max_entanglement_attempts = 1_000_000

# # Setting memory lifetime equal to the maximum permitted
# # entanglement attempts effectively disables expiration
# # during one end-to-end Bell-pair establishment session.
# phase2c_memory_lifetime_attempts = (
#     phase2c_max_entanglement_attempts
# )

# phase2c_protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15 / 16
# }

# phase2c_results = []

# phase2c_start_time = time.perf_counter()


# # =========================================================
# # Execute One Session Per Distance and Protocol
# # =========================================================

# for distance_km in phase2c_distances:

#     for protocol_name, z_probability in phase2c_protocols.items():

#         configuration_start = time.perf_counter()

#         channel = RepeaterAssistedQuantumChannel(
#             source=alice,
#             repeater=r1,
#             destination=bob,
#             total_distance_km=distance_km,
#             swapping_success_probability=(
#                 phase2c_swapping_success_probability
#             ),
#             memory_lifetime_attempts=(
#                 phase2c_memory_lifetime_attempts
#             ),
#             max_entanglement_attempts=(
#                 phase2c_max_entanglement_attempts
#             )
#         )

#         try:
#             session_key, metadata = (
#                 generate_bb84_session_key_over_repeater_channel(
#                     channel=channel,
#                     target_bits=phase2c_target_bits,
#                     transmission_margin=(
#                         phase2c_transmission_margin
#                     ),
#                     max_rounds=phase2c_max_rounds,
#                     eve_present=False,
#                     z_probability=z_probability,
#                     verbose=False
#                 )
#             )

#             configuration_runtime = (
#                 time.perf_counter() - configuration_start
#             )

#             # Aggregate QBER across accepted round logs.
#             accepted_round_qbers = [
#                 round_log["qber"]
#                 for round_log in metadata["round_logs"]
#                 if round_log["accepted"]
#             ]

#             mean_accepted_qber = (
#                 sum(accepted_round_qbers)
#                 / len(accepted_round_qbers)
#                 if accepted_round_qbers
#                 else None
#             )

#             accepted_rounds = sum(
#                 bool(round_log["accepted"])
#                 for round_log in metadata["round_logs"]
#             )

#             rejected_rounds = (
#                 len(metadata["round_logs"])
#                 - accepted_rounds
#             )

#             phase2c_results.append({
#                 # -------------------------------------------------
#                 # Experiment identity
#                 # -------------------------------------------------
#                 "Protocol":
#                     protocol_name,

#                 "Distance (km)":
#                     distance_km,

#                 "Segment Distance (km)":
#                     metadata["segment_distance_km"],

#                 "Target Key Bits":
#                     phase2c_target_bits,

#                 "Transmission Margin":
#                     phase2c_transmission_margin,

#                 "Z Probability":
#                     z_probability,

#                 "Basis Match Probability (%)":
#                     metadata[
#                         "basis_match_probability"
#                     ] * 100,

#                 "Swap Success Probability":
#                     phase2c_swapping_success_probability,

#                 "Memory Lifetime (Attempts)":
#                     phase2c_memory_lifetime_attempts,

#                 # -------------------------------------------------
#                 # Quantum protocol layer
#                 # -------------------------------------------------
#                 "Rounds Used":
#                     metadata["rounds_used"],

#                 "Accepted Rounds":
#                     accepted_rounds,

#                 "Rejected Rounds":
#                     rejected_rounds,

#                 "Mean Accepted QBER (%)":
#                     (
#                         mean_accepted_qber * 100
#                         if mean_accepted_qber is not None
#                         else None
#                     ),

#                 "Protocol Qubits Requested":
#                     metadata["total_qubits_sent"],

#                 "Protocol Qubits Delivered":
#                     metadata["total_qubits_received"],

#                 "Protocol Qubits Lost":
#                     metadata["total_qubits_lost"],

#                 "Protocol Delivery Rate (%)":
#                     (
#                         metadata["total_qubits_received"]
#                         / metadata["total_qubits_sent"]
#                         * 100
#                         if metadata["total_qubits_sent"]
#                         else 0
#                     ),

#                 "Effective Key Yield (%)":
#                     metadata["effective_key_yield"] * 100,

#                 "Keygen Time (s)":
#                     metadata["keygen_time_s"],

#                 # -------------------------------------------------
#                 # Physical layer — left elementary link
#                 # -------------------------------------------------
#                 "Left Link Photon Attempts":
#                     metadata["left_photons_sent"],

#                 "Left Link Successful Transmissions":
#                     metadata["left_photons_survived"],

#                 "Left Link Photon Losses":
#                     metadata["left_photons_lost"],

#                 "Left Link Observed Survival (%)":
#                     (
#                         metadata["left_photons_survived"]
#                         / metadata["left_photons_sent"]
#                         * 100
#                         if metadata["left_photons_sent"]
#                         else 0
#                     ),

#                 # -------------------------------------------------
#                 # Physical layer — right elementary link
#                 # -------------------------------------------------
#                 "Right Link Photon Attempts":
#                     metadata["right_photons_sent"],

#                 "Right Link Successful Transmissions":
#                     metadata["right_photons_survived"],

#                 "Right Link Photon Losses":
#                     metadata["right_photons_lost"],

#                 "Right Link Observed Survival (%)":
#                     (
#                         metadata["right_photons_survived"]
#                         / metadata["right_photons_sent"]
#                         * 100
#                         if metadata["right_photons_sent"]
#                         else 0
#                     ),

#                 # -------------------------------------------------
#                 # Physical layer — path totals
#                 # -------------------------------------------------
#                 "Total Physical Photon Attempts":
#                     metadata["total_photons_sent_physical"],

#                 "Total Successful Physical Transmissions":
#                     metadata[
#                         "total_photons_survived_physical"
#                     ],

#                 "Total Physical Photon Losses":
#                     metadata["total_photons_lost_physical"],

#                 "Total Entanglement Attempts":
#                     metadata["total_entanglement_attempts"],

#                 "Average Entanglement Attempts per Delivered Qubit":
#                     metadata[
#                         "average_entanglement_attempts_per_delivered_qubit"
#                     ],

#                 "Successful End-to-End Bell Pairs":
#                     metadata[
#                         "successful_end_to_end_bell_pairs"
#                     ],

#                 "Swap Attempts":
#                     metadata["swap_attempts"],

#                 "Left Memory Expirations":
#                     metadata["memory_expirations_left"],

#                 "Right Memory Expirations":
#                     metadata["memory_expirations_right"],

#                 "Total Memory Expirations":
#                     (
#                         metadata["memory_expirations_left"]
#                         + metadata["memory_expirations_right"]
#                     ),

#                 "Configuration Runtime (s)":
#                     configuration_runtime,

#                 "Status":
#                     "SUCCESS",

#                 "Error":
#                     None
#             })

#             print(
#                 f"{distance_km:3d} km | "
#                 f"{protocol_name:<31} | "
#                 f"protocol qubits="
#                 f"{metadata['total_qubits_sent']:,} | "
#                 f"physical attempts="
#                 f"{metadata['total_entanglement_attempts']:,} | "
#                 f"avg attempts/qubit="
#                 f"{metadata['average_entanglement_attempts_per_delivered_qubit']:.2f}"
#             )

#         except Exception as error:

#             configuration_runtime = (
#                 time.perf_counter() - configuration_start
#             )

#             phase2c_results.append({
#                 "Protocol":
#                     protocol_name,

#                 "Distance (km)":
#                     distance_km,

#                 "Segment Distance (km)":
#                     distance_km / 2,

#                 "Target Key Bits":
#                     phase2c_target_bits,

#                 "Transmission Margin":
#                     phase2c_transmission_margin,

#                 "Z Probability":
#                     z_probability,

#                 "Basis Match Probability (%)":
#                     basis_match_probability(
#                         z_probability
#                     ) * 100,

#                 "Swap Success Probability":
#                     phase2c_swapping_success_probability,

#                 "Memory Lifetime (Attempts)":
#                     phase2c_memory_lifetime_attempts,

#                 "Configuration Runtime (s)":
#                     configuration_runtime,

#                 "Status":
#                     "FAILED",

#                 "Error":
#                     str(error)
#             })

#             print(
#                 f"{distance_km:3d} km | "
#                 f"{protocol_name:<31} | "
#                 f"FAILED: {error}"
#             )


# # =========================================================
# # Save Single-Run Dataset
# # =========================================================

# df_phase2c_distance_sweep = pd.DataFrame(
#     phase2c_results
# )

# df_phase2c_distance_sweep.to_csv(
#     "phase2c_single_repeater_distance_sweep_ideal.csv",
#     index=False
# )

# phase2c_elapsed = (
#     time.perf_counter() - phase2c_start_time
# )

# print(
#     f"\nPhase 2C ideal distance sweep completed "
#     f"in {phase2c_elapsed / 60:.2f} minutes."
# )

# display(
#     df_phase2c_distance_sweep
# )

In [ ]:
# @title Plot : Distance vs Total Entanglement Attempts

# import matplotlib.pyplot as plt

# successful_phase2c = (
#     df_phase2c_distance_sweep[
#         df_phase2c_distance_sweep["Status"]
#         == "SUCCESS"
#     ]
#     .copy()
# )

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_protocols.keys():

#     subset = successful_phase2c[
#         successful_phase2c["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset["Total Entanglement Attempts"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Total Elementary-Link Entanglement Attempts")

# plt.title(
#     "Distance vs Repeater Entanglement Resource Cost"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C Distance vs Total Entanglement Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Distance vs Average Entanglement Attempts per Delivered Qubit

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_protocols.keys():

#     subset = successful_phase2c[
#         successful_phase2c["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset[
#             "Average Entanglement Attempts per Delivered Qubit"
#         ],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel(
#     "Average Entanglement Attempts per Delivered Qubit"
# )

# plt.title(
#     "Distance vs Per-Qubit Repeater Resource Cost"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C Distance vs Average Entanglement Attempts per Delivered Qubit.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2C Monte Carlo Trials

100 Trials

In [ ]:
# @title Monte Carlo Experiment — 100 Trials

import gc
import os
import time
import pandas as pd


# =========================================================
# Experiment Configuration
# =========================================================

PHASE2C_MONTE_CARLO_TRIALS = 100
PHASE2C_CHECKPOINT_INTERVAL = 1

phase2c_mc_distances = [10, 50, 80, 100, 150, 200]

phase2c_mc_protocols = {
    "Standard BB84 (50-50)": 0.5,
    "Biased BB84 (80-20)": 0.8,
    "T12-style (15/16 - 1/16)": 15 / 16
}

PHASE2C_TARGET_BITS = 128
PHASE2C_TRANSMISSION_MARGIN = 1.3
PHASE2C_MAX_ROUNDS = 100

# Ideal repeater assumptions
PHASE2C_SWAP_SUCCESS_PROBABILITY = 1.0
PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS = 1_000_000

# Effectively ideal memory during one Bell-pair establishment
PHASE2C_MEMORY_LIFETIME_ATTEMPTS = (
    PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS
)

PHASE2C_RAW_FILE = (
    "phase2c_monte_carlo_raw_100_trials.csv"
)

PHASE2C_LATEST_CHECKPOINT_FILE = (
    "phase2c_monte_carlo_latest_checkpoint.csv"
)

# Change to True only when resuming after a crash.
RESUME_PHASE2C_FROM_CHECKPOINT = True


# =========================================================
# Resume Support
# =========================================================

phase2c_raw_results = []
phase2c_starting_trial = 1

if (
    RESUME_PHASE2C_FROM_CHECKPOINT
    and os.path.exists(
        PHASE2C_LATEST_CHECKPOINT_FILE
    )
):
    existing_checkpoint = pd.read_csv(
        PHASE2C_LATEST_CHECKPOINT_FILE
    )

    phase2c_raw_results = (
        existing_checkpoint.to_dict("records")
    )

    if not existing_checkpoint.empty:
        last_completed_trial = int(
            existing_checkpoint["Trial"].max()
        )

        phase2c_starting_trial = (
            last_completed_trial + 1
        )

        print(
            "Resuming Phase 2C Monte Carlo "
            f"from Trial {phase2c_starting_trial}."
        )

else:
    print(
        "Starting a new Phase 2C Monte Carlo experiment."
    )


# =========================================================
# Monte Carlo Execution
# =========================================================

phase2c_experiment_start = time.perf_counter()

for trial in range(
    phase2c_starting_trial,
    PHASE2C_MONTE_CARLO_TRIALS + 1
):

    trial_start = time.perf_counter()

    print(
        f"\n{'=' * 82}\n"
        f"Phase 2C Monte Carlo — "
        f"Trial {trial}/{PHASE2C_MONTE_CARLO_TRIALS}\n"
        f"{'=' * 82}"
    )

    for distance_km in phase2c_mc_distances:

        for protocol_name, z_probability in (
            phase2c_mc_protocols.items()
        ):

            configuration_start = time.perf_counter()

            channel = None
            session_key = None
            metadata = None

            try:
                channel = RepeaterAssistedQuantumChannel(
                    source=alice,
                    repeater=r1,
                    destination=bob,
                    total_distance_km=distance_km,
                    swapping_success_probability=(
                        PHASE2C_SWAP_SUCCESS_PROBABILITY
                    ),
                    memory_lifetime_attempts=(
                        PHASE2C_MEMORY_LIFETIME_ATTEMPTS
                    ),
                    max_entanglement_attempts=(
                        PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS
                    )
                )

                session_key, metadata = (
                    generate_bb84_session_key_over_repeater_channel(
                        channel=channel,
                        target_bits=PHASE2C_TARGET_BITS,
                        transmission_margin=(
                            PHASE2C_TRANSMISSION_MARGIN
                        ),
                        max_rounds=PHASE2C_MAX_ROUNDS,
                        eve_present=False,
                        z_probability=z_probability,
                        verbose=False
                    )
                )

                configuration_runtime = (
                    time.perf_counter()
                    - configuration_start
                )

                round_logs = metadata["round_logs"]

                accepted_rounds = sum(
                    bool(round_log["accepted"])
                    for round_log in round_logs
                )

                rejected_rounds = (
                    len(round_logs)
                    - accepted_rounds
                )

                accepted_qbers = [
                    round_log["qber"]
                    for round_log in round_logs
                    if round_log["accepted"]
                ]

                mean_accepted_qber = (
                    sum(accepted_qbers)
                    / len(accepted_qbers)
                    if accepted_qbers
                    else None
                )

                left_survival = (
                    metadata["left_photons_survived"]
                    / metadata["left_photons_sent"]
                    if metadata["left_photons_sent"]
                    else 0
                )

                right_survival = (
                    metadata["right_photons_survived"]
                    / metadata["right_photons_sent"]
                    if metadata["right_photons_sent"]
                    else 0
                )

                protocol_delivery_rate = (
                    metadata["total_qubits_received"]
                    / metadata["total_qubits_sent"]
                    if metadata["total_qubits_sent"]
                    else 0
                )

                phase2c_raw_results.append({
                    # =====================================
                    # Experiment Identity
                    # =====================================
                    "Trial":
                        trial,

                    "Protocol":
                        protocol_name,

                    "Distance (km)":
                        distance_km,

                    "Segment Distance (km)":
                        metadata["segment_distance_km"],

                    "Target Key Bits":
                        PHASE2C_TARGET_BITS,

                    "Transmission Margin":
                        PHASE2C_TRANSMISSION_MARGIN,

                    "Z Probability":
                        z_probability,

                    "Basis Match Probability (%)":
                        metadata[
                            "basis_match_probability"
                        ] * 100,

                    "Swap Success Probability":
                        PHASE2C_SWAP_SUCCESS_PROBABILITY,

                    "Memory Lifetime (Attempts)":
                        PHASE2C_MEMORY_LIFETIME_ATTEMPTS,

                    # =====================================
                    # Quantum Protocol Layer
                    # =====================================
                    "Rounds Used":
                        metadata["rounds_used"],

                    "Accepted Rounds":
                        accepted_rounds,

                    "Rejected Rounds":
                        rejected_rounds,

                    "Mean Accepted QBER (%)":
                        (
                            mean_accepted_qber * 100
                            if mean_accepted_qber is not None
                            else None
                        ),

                    "Protocol Qubits Requested":
                        metadata["total_qubits_sent"],

                    "Protocol Qubits Delivered":
                        metadata["total_qubits_received"],

                    "Protocol Qubits Lost":
                        metadata["total_qubits_lost"],

                    "Protocol Delivery Rate (%)":
                        protocol_delivery_rate * 100,

                    "Effective Key Yield (%)":
                        metadata[
                            "effective_key_yield"
                        ] * 100,

                    "Keygen Time (s)":
                        metadata["keygen_time_s"],

                    # =====================================
                    # Left Elementary Link
                    # =====================================
                    "Left Link Photon Attempts":
                        metadata["left_photons_sent"],

                    "Left Link Successful Transmissions":
                        metadata["left_photons_survived"],

                    "Left Link Photon Losses":
                        metadata["left_photons_lost"],

                    "Left Link Observed Survival (%)":
                        left_survival * 100,

                    # =====================================
                    # Right Elementary Link
                    # =====================================
                    "Right Link Photon Attempts":
                        metadata["right_photons_sent"],

                    "Right Link Successful Transmissions":
                        metadata["right_photons_survived"],

                    "Right Link Photon Losses":
                        metadata["right_photons_lost"],

                    "Right Link Observed Survival (%)":
                        right_survival * 100,

                    # =====================================
                    # Complete Physical Layer
                    # =====================================
                    "Total Physical Photon Attempts":
                        metadata[
                            "total_photons_sent_physical"
                        ],

                    "Total Successful Physical Transmissions":
                        metadata[
                            "total_photons_survived_physical"
                        ],

                    "Total Physical Photon Losses":
                        metadata[
                            "total_photons_lost_physical"
                        ],

                    "Total Entanglement Attempts":
                        metadata[
                            "total_entanglement_attempts"
                        ],

                    "Average Entanglement Attempts per Delivered Qubit":
                        metadata[
                            "average_entanglement_attempts_per_delivered_qubit"
                        ],

                    "Successful End-to-End Bell Pairs":
                        metadata[
                            "successful_end_to_end_bell_pairs"
                        ],

                    "Swap Attempts":
                        metadata["swap_attempts"],

                    "Left Memory Expirations":
                        metadata[
                            "memory_expirations_left"
                        ],

                    "Right Memory Expirations":
                        metadata[
                            "memory_expirations_right"
                        ],

                    "Total Memory Expirations":
                        (
                            metadata[
                                "memory_expirations_left"
                            ]
                            +
                            metadata[
                                "memory_expirations_right"
                            ]
                        ),

                    "Configuration Runtime (s)":
                        configuration_runtime,

                    "Status":
                        "SUCCESS",

                    "Error":
                        None
                })

                print(
                    f"Trial {trial:3d} | "
                    f"{distance_km:3d} km | "
                    f"{protocol_name:<31} | "
                    f"protocol qubits="
                    f"{metadata['total_qubits_sent']:,} | "
                    f"physical attempts="
                    f"{metadata['total_entanglement_attempts']:,} | "
                    f"avg attempts/qubit="
                    f"{metadata['average_entanglement_attempts_per_delivered_qubit']:.2f} | "
                    f"time="
                    f"{metadata['keygen_time_s']:.2f}s"
                )

            except Exception as error:

                configuration_runtime = (
                    time.perf_counter()
                    - configuration_start
                )

                phase2c_raw_results.append({
                    "Trial":
                        trial,

                    "Protocol":
                        protocol_name,

                    "Distance (km)":
                        distance_km,

                    "Segment Distance (km)":
                        distance_km / 2,

                    "Target Key Bits":
                        PHASE2C_TARGET_BITS,

                    "Transmission Margin":
                        PHASE2C_TRANSMISSION_MARGIN,

                    "Z Probability":
                        z_probability,

                    "Basis Match Probability (%)":
                        basis_match_probability(
                            z_probability
                        ) * 100,

                    "Swap Success Probability":
                        PHASE2C_SWAP_SUCCESS_PROBABILITY,

                    "Memory Lifetime (Attempts)":
                        PHASE2C_MEMORY_LIFETIME_ATTEMPTS,

                    "Configuration Runtime (s)":
                        configuration_runtime,

                    "Status":
                        "FAILED",

                    "Error":
                        str(error)
                })

                print(
                    f"Trial {trial:3d} | "
                    f"{distance_km:3d} km | "
                    f"{protocol_name:<31} | "
                    f"FAILED: {error}"
                )

            # Explicit cleanup for long Colab runs
            del channel
            del session_key
            del metadata

            gc.collect()

    trial_runtime = (
        time.perf_counter() - trial_start
    )

    completed_rows = len(
        phase2c_raw_results
    )

    print(
        f"\nTrial {trial} completed in "
        f"{trial_runtime / 60:.2f} minutes."
    )

    print(
        f"Rows collected so far: "
        f"{completed_rows:,}"
    )

    # =====================================================
    # Checkpoint Every 50 Completed Trials
    # =====================================================

    if (
        trial % PHASE2C_CHECKPOINT_INTERVAL == 0
        or trial == PHASE2C_MONTE_CARLO_TRIALS
    ):

        checkpoint_df = pd.DataFrame(
            phase2c_raw_results
        )

        numbered_checkpoint_file = (
            "phase2c_monte_carlo_checkpoint_"
            f"trial_{trial}.csv"
        )

        checkpoint_df.to_csv(
            numbered_checkpoint_file,
            index=False
        )

        checkpoint_df.to_csv(
            PHASE2C_LATEST_CHECKPOINT_FILE,
            index=False
        )

        print(
            f"\nCheckpoint saved after Trial {trial}:"
            f"\n  {numbered_checkpoint_file}"
            f"\n  {PHASE2C_LATEST_CHECKPOINT_FILE}"
            f"\n  Rows saved: {len(checkpoint_df):,}"
        )


# =========================================================
# Save Final Raw Dataset
# =========================================================

df_phase2c_monte_carlo_raw = pd.DataFrame(
    phase2c_raw_results
)

df_phase2c_monte_carlo_raw.to_csv(
    PHASE2C_RAW_FILE,
    index=False
)

phase2c_total_runtime = (
    time.perf_counter()
    - phase2c_experiment_start
)

expected_phase2c_rows = (
    PHASE2C_MONTE_CARLO_TRIALS
    * len(phase2c_mc_distances)
    * len(phase2c_mc_protocols)
)

print(
    f"\n{'=' * 82}\n"
    f"PHASE 2C MONTE CARLO COMPLETE\n"
    f"{'=' * 82}\n"
    f"Trials completed       : "
    f"{PHASE2C_MONTE_CARLO_TRIALS}\n"
    f"Distances per trial    : "
    f"{len(phase2c_mc_distances)}\n"
    f"Protocols per distance : "
    f"{len(phase2c_mc_protocols)}\n"
    f"Raw rows generated     : "
    f"{len(df_phase2c_monte_carlo_raw):,}\n"
    f"Expected raw rows      : "
    f"{expected_phase2c_rows:,}\n"
    f"Total runtime          : "
    f"{phase2c_total_runtime / 3600:.2f} hours\n"
    f"Final raw CSV          : "
    f"{PHASE2C_RAW_FILE}\n"
    f"{'=' * 82}"
)

display(
    df_phase2c_monte_carlo_raw.head(18)
)

In [ ]:
# @title Monte Carlo Aggregate Statistics

import pandas as pd


df_phase2c_monte_carlo_raw = pd.read_csv(
    "phase2c_monte_carlo_raw_100_trials.csv"
)

successful_phase2c_runs = (
    df_phase2c_monte_carlo_raw[
        df_phase2c_monte_carlo_raw[
            "Status"
        ] == "SUCCESS"
    ]
    .copy()
)

phase2c_group_columns = [
    "Protocol",
    "Distance (km)"
]

phase2c_metrics_to_aggregate = [
    "Segment Distance (km)",
    "Basis Match Probability (%)",

    "Rounds Used",
    "Accepted Rounds",
    "Rejected Rounds",
    "Mean Accepted QBER (%)",

    "Protocol Qubits Requested",
    "Protocol Qubits Delivered",
    "Protocol Qubits Lost",
    "Protocol Delivery Rate (%)",
    "Effective Key Yield (%)",
    "Keygen Time (s)",

    "Left Link Photon Attempts",
    "Left Link Successful Transmissions",
    "Left Link Photon Losses",
    "Left Link Observed Survival (%)",

    "Right Link Photon Attempts",
    "Right Link Successful Transmissions",
    "Right Link Photon Losses",
    "Right Link Observed Survival (%)",

    "Total Physical Photon Attempts",
    "Total Successful Physical Transmissions",
    "Total Physical Photon Losses",

    "Total Entanglement Attempts",
    "Average Entanglement Attempts per Delivered Qubit",

    "Successful End-to-End Bell Pairs",
    "Swap Attempts",

    "Left Memory Expirations",
    "Right Memory Expirations",
    "Total Memory Expirations",

    "Configuration Runtime (s)"
]


phase2c_summary_statistics = (
    successful_phase2c_runs
    .groupby(
        phase2c_group_columns,
        as_index=False
    )[phase2c_metrics_to_aggregate]
    .agg([
        "count",
        "mean",
        "std",
        "median",
        "min",
        "max"
    ])
)


# Flatten MultiIndex columns
phase2c_summary_statistics.columns = [
    " ".join(
        str(part)
        for part in column
        if str(part) != ""
    ).strip()
    if isinstance(column, tuple)
    else column
    for column in phase2c_summary_statistics.columns
]


# =========================================================
# Success and Failure Counts
# =========================================================

phase2c_run_counts = (
    df_phase2c_monte_carlo_raw
    .groupby(
        phase2c_group_columns
    )
    .agg(
        Total_Trials=(
            "Trial",
            "count"
        ),

        Successful_Trials=(
            "Status",
            lambda values: (
                values == "SUCCESS"
            ).sum()
        ),

        Failed_Trials=(
            "Status",
            lambda values: (
                values == "FAILED"
            ).sum()
        )
    )
    .reset_index()
)

phase2c_run_counts[
    "Success Rate (%)"
] = (
    phase2c_run_counts[
        "Successful_Trials"
    ]
    /
    phase2c_run_counts[
        "Total_Trials"
    ]
    * 100
)


df_phase2c_monte_carlo_summary = (
    phase2c_run_counts
    .merge(
        phase2c_summary_statistics,
        on=phase2c_group_columns,
        how="left"
    )
)


# =========================================================
# Coefficients of Variation
# =========================================================

df_phase2c_monte_carlo_summary[
    "Entanglement Attempts CV"
] = (
    df_phase2c_monte_carlo_summary[
        "Total Entanglement Attempts std"
    ]
    /
    df_phase2c_monte_carlo_summary[
        "Total Entanglement Attempts mean"
    ]
)

df_phase2c_monte_carlo_summary[
    "Entanglement Attempts CV (%)"
] = (
    df_phase2c_monte_carlo_summary[
        "Entanglement Attempts CV"
    ]
    * 100
)

df_phase2c_monte_carlo_summary[
    "Keygen Time CV"
] = (
    df_phase2c_monte_carlo_summary[
        "Keygen Time (s) std"
    ]
    /
    df_phase2c_monte_carlo_summary[
        "Keygen Time (s) mean"
    ]
)

df_phase2c_monte_carlo_summary[
    "Keygen Time CV (%)"
] = (
    df_phase2c_monte_carlo_summary[
        "Keygen Time CV"
    ]
    * 100
)


# =========================================================
# Sorting
# =========================================================

phase2c_protocol_order = {
    "Standard BB84 (50-50)": 0,
    "Biased BB84 (80-20)": 1,
    "T12-style (15/16 - 1/16)": 2
}

df_phase2c_monte_carlo_summary[
    "_Protocol Order"
] = (
    df_phase2c_monte_carlo_summary[
        "Protocol"
    ].map(phase2c_protocol_order)
)

df_phase2c_monte_carlo_summary = (
    df_phase2c_monte_carlo_summary
    .sort_values([
        "_Protocol Order",
        "Distance (km)"
    ])
    .drop(
        columns=["_Protocol Order"]
    )
    .reset_index(drop=True)
)


PHASE2C_SUMMARY_FILE = (
    "phase2c_monte_carlo_summary_100_trials.csv"
)

df_phase2c_monte_carlo_summary.to_csv(
    PHASE2C_SUMMARY_FILE,
    index=False
)

print(
    f"Phase 2C summary saved to "
    f"{PHASE2C_SUMMARY_FILE}"
)

print(
    f"Raw rows     : "
    f"{len(df_phase2c_monte_carlo_raw):,}"
)

print(
    f"Summary rows : "
    f"{len(df_phase2c_monte_carlo_summary):,}"
)

print(
    f"Failed runs  : "
    f"{(df_phase2c_monte_carlo_raw['Status']== 'FAILED').sum():,}"
)

display(
    df_phase2c_monte_carlo_summary
)

In [ ]:
# @title Compact Paper-Ready Summary

phase2c_paper_columns = [
    "Protocol",
    "Distance (km)",

    "Total_Trials",
    "Successful_Trials",
    "Failed_Trials",
    "Success Rate (%)",

    "Protocol Qubits Requested mean",
    "Protocol Qubits Requested std",

    "Protocol Qubits Delivered mean",
    "Protocol Qubits Delivered std",

    "Protocol Delivery Rate (%) mean",
    "Protocol Delivery Rate (%) std",

    "Effective Key Yield (%) mean",
    "Effective Key Yield (%) std",

    "Keygen Time (s) mean",
    "Keygen Time (s) std",

    "Total Physical Photon Attempts mean",
    "Total Physical Photon Attempts std",

    "Total Physical Photon Losses mean",
    "Total Physical Photon Losses std",

    "Total Entanglement Attempts mean",
    "Total Entanglement Attempts std",
    "Total Entanglement Attempts median",

    "Average Entanglement Attempts per Delivered Qubit mean",
    "Average Entanglement Attempts per Delivered Qubit std",

    "Successful End-to-End Bell Pairs mean",
    "Swap Attempts mean",

    "Total Memory Expirations mean",

    "Entanglement Attempts CV (%)",
    "Keygen Time CV (%)"
]

available_phase2c_paper_columns = [
    column
    for column in phase2c_paper_columns
    if column
    in df_phase2c_monte_carlo_summary.columns
]

df_phase2c_paper_summary = (
    df_phase2c_monte_carlo_summary[
        available_phase2c_paper_columns
    ]
    .copy()
)

PHASE2C_PAPER_SUMMARY_FILE = (
    "phase2c_monte_carlo_"
    "paper_summary_100_trials.csv"
)

df_phase2c_paper_summary.to_csv(
    PHASE2C_PAPER_SUMMARY_FILE,
    index=False
)

print(
    "Phase 2C paper-ready summary saved to:"
)

print(
    PHASE2C_PAPER_SUMMARY_FILE
)

display(
    df_phase2c_paper_summary
)

In [ ]:
# @title Plot : Distance vs Key Generation Time

import matplotlib.pyplot as plt
import pandas as pd

phase2c_summary_df = pd.read_csv(
    "phase2c_monte_carlo_summary_100_trials.csv"
)

phase2c_plot_protocols = [
    "Standard BB84 (50-50)",
    "Biased BB84 (80-20)",
    "T12-style (15/16 - 1/16)"
]

plt.figure(figsize=(10, 6))

for protocol_name in phase2c_plot_protocols:

    subset = phase2c_summary_df[
        phase2c_summary_df["Protocol"]
        == protocol_name
    ].sort_values("Distance (km)")

    plt.errorbar(
        subset["Distance (km)"],
        subset["Keygen Time (s) mean"],
        yerr=subset["Keygen Time (s) std"],
        marker="o",
        capsize=4,
        label=protocol_name
    )

plt.xlabel("Total Distance (km)")
plt.ylabel("Mean Simulation Key Generation Time (s)")

plt.title(
    "Repeater-Assisted QKD Key Generation Time "
    "(Mean ± SD, 100 Trials)"
)

plt.legend()
plt.grid(True)

plt.savefig(
    "Phase 2C MC Distance vs Key Generation Time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# @title Plot : Distance vs Effective Key Yield

plt.figure(figsize=(10, 6))

for protocol_name in phase2c_plot_protocols:

    subset = phase2c_summary_df[
        phase2c_summary_df["Protocol"]
        == protocol_name
    ].sort_values("Distance (km)")

    plt.errorbar(
        subset["Distance (km)"],
        subset[
            "Effective Key Yield (%) mean"
        ],
        yerr=subset[
            "Effective Key Yield (%) std"
        ],
        marker="o",
        capsize=4,
        label=protocol_name
    )

plt.xlabel("Total Distance (km)")
plt.ylabel("Mean Effective Key Yield (%)")

plt.title(
    "Repeater-Assisted QKD Effective Key Yield "
    "(Mean ± SD, 100 Trials)"
)

plt.legend()
plt.grid(True)

plt.savefig(
    "Phase 2C MC Distance vs Effective Key Yield.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# @title Plot : Repeater Physical Resource Cost

# =========================================================
# Total Entanglement Attempts
# =========================================================

plt.figure(figsize=(10, 6))

for protocol_name in phase2c_plot_protocols:

    subset = phase2c_summary_df[
        phase2c_summary_df["Protocol"]
        == protocol_name
    ].sort_values("Distance (km)")

    plt.errorbar(
        subset["Distance (km)"],
        subset[
            "Total Entanglement Attempts mean"
        ],
        yerr=subset[
            "Total Entanglement Attempts std"
        ],
        marker="o",
        capsize=4,
        label=protocol_name
    )

plt.xlabel("Total Distance (km)")
plt.ylabel("Mean Total Entanglement Attempts")

plt.title(
    "Distance vs Total Repeater Entanglement Attempts "
    "(100 Trials)"
)

plt.yscale("log")
plt.legend()
plt.grid(True)

plt.savefig(
    "Phase 2C MC Distance vs Total Entanglement Attempts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# Average Attempts per Delivered Qubit
# =========================================================

plt.figure(figsize=(10, 6))

for protocol_name in phase2c_plot_protocols:

    subset = phase2c_summary_df[
        phase2c_summary_df["Protocol"]
        == protocol_name
    ].sort_values("Distance (km)")

    plt.errorbar(
        subset["Distance (km)"],
        subset[
            "Average Entanglement Attempts per Delivered Qubit mean"
        ],
        yerr=subset[
            "Average Entanglement Attempts per Delivered Qubit std"
        ],
        marker="o",
        capsize=4,
        label=protocol_name
    )

plt.xlabel("Total Distance (km)")
plt.ylabel(
    "Mean Entanglement Attempts per Delivered Qubit"
)

plt.title(
    "Distance vs Per-Qubit Repeater Resource Cost "
    "(100 Trials)"
)

plt.yscale("log")
plt.legend()
plt.grid(True)

plt.savefig(
    "Phase 2C MC Distance vs Attempts per Delivered Qubit.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# Total Physical Photon Attempts
# =========================================================

plt.figure(figsize=(10, 6))

for protocol_name in phase2c_plot_protocols:

    subset = phase2c_summary_df[
        phase2c_summary_df["Protocol"]
        == protocol_name
    ].sort_values("Distance (km)")

    plt.errorbar(
        subset["Distance (km)"],
        subset[
            "Total Physical Photon Attempts mean"
        ],
        yerr=subset[
            "Total Physical Photon Attempts std"
        ],
        marker="o",
        capsize=4,
        label=protocol_name
    )

plt.xlabel("Total Distance (km)")
plt.ylabel("Mean Physical Photon Attempts")

plt.title(
    "Distance vs Physical Photon Resource Requirement "
    "(100 Trials)"
)

plt.yscale("log")
plt.legend()
plt.grid(True)

plt.savefig(
    "Phase 2C MC Distance vs Physical Photon Attempts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# @title Plot : Distribution Plots

import matplotlib.pyplot as plt
import pandas as pd


phase2c_raw_df = pd.read_csv(
    "phase2c_monte_carlo_raw_100_trials.csv"
)

phase2c_successful_raw = (
    phase2c_raw_df[
        phase2c_raw_df["Status"] == "SUCCESS"
    ]
    .copy()
)

ordered_distances = sorted(
    phase2c_successful_raw[
        "Distance (km)"
    ].unique()
)


# =========================================================
# Entanglement Attempt Distribution by Protocol
# =========================================================

for protocol_name in phase2c_plot_protocols:

    protocol_data = phase2c_successful_raw[
        phase2c_successful_raw[
            "Protocol"
        ] == protocol_name
    ]

    boxplot_values = [
        protocol_data[
            protocol_data[
                "Distance (km)"
            ] == distance
        ][
            "Total Entanglement Attempts"
        ].dropna().values

        for distance in ordered_distances
    ]

    plt.figure(figsize=(11, 6))

    plt.boxplot(
        boxplot_values,
        tick_labels=[
            str(distance)
            for distance in ordered_distances
        ],
        showfliers=True
    )

    plt.xlabel("Total Distance (km)")
    plt.ylabel("Total Entanglement Attempts")

    plt.title(
        f"{protocol_name}: Entanglement Attempt "
        f"Distribution Across 100 Trials"
    )

    plt.yscale("log")
    plt.grid(True)

    safe_protocol_name = (
        protocol_name
        .replace("/", "-")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    plt.savefig(
        f"Phase 2C MC {safe_protocol_name} "
        f"Entanglement Attempt Boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# =========================================================
# Key Generation Time Distribution by Protocol
# =========================================================

for protocol_name in phase2c_plot_protocols:

    protocol_data = phase2c_successful_raw[
        phase2c_successful_raw[
            "Protocol"
        ] == protocol_name
    ]

    boxplot_values = [
        protocol_data[
            protocol_data[
                "Distance (km)"
            ] == distance
        ][
            "Keygen Time (s)"
        ].dropna().values

        for distance in ordered_distances
    ]

    plt.figure(figsize=(11, 6))

    plt.boxplot(
        boxplot_values,
        tick_labels=[
            str(distance)
            for distance in ordered_distances
        ],
        showfliers=True
    )

    plt.xlabel("Total Distance (km)")
    plt.ylabel("Simulation Key Generation Time (s)")

    plt.title(
        f"{protocol_name}: Key Generation Time "
        f"Distribution Across 100 Trials"
    )

    plt.grid(True)

    safe_protocol_name = (
        protocol_name
        .replace("/", "-")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    plt.savefig(
        f"Phase 2C MC {safe_protocol_name} "
        f"Key Generation Time Boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## Phase 3 : 14-07-2026T10:42:16 --

Multiple Repeaters in a Linear Chain

### Phase 3A

Establishing the classes to follow through with the new decisions

In [ ]:
# @title Spatial Node Hierarchy

import math
from typing import Optional

class SpatialNode(Node):
    """
    Coordinate-aware extension of the existing NARAD-QC Node class.
    """

    def __init__(
        self,
        name: str,
        role: str,
        x: float,
        y: float,
        active: bool = True,
        authenticated: bool = True
    ):
        super().__init__(
            name=name,
            role=role
        )

        self.x = self._validate_coordinate(
            x,
            coordinate_name="x"
        )

        self.y = self._validate_coordinate(
            y,
            coordinate_name="y"
        )

        self.active = bool(active)
        self.authenticated = bool(authenticated)

        self.busy = False
        self.reserved_by_session = None

    @staticmethod
    def _validate_coordinate(
        value: float,
        coordinate_name: str
    ) -> float:
        """
        Validate and normalize a coordinate value.
        """

        if isinstance(value, bool):
            raise TypeError(
                f"{coordinate_name}-coordinate cannot be Boolean."
            )

        if not isinstance(value, (int, float)):
            raise TypeError(
                f"{coordinate_name}-coordinate must be numeric."
            )

        normalized_value = float(value)

        if not math.isfinite(normalized_value):
            raise ValueError(
                f"{coordinate_name}-coordinate must be finite."
            )

        return normalized_value

    @property
    def position(self) -> tuple[float, float]:
        """
        Return the node's fixed 2D position.
        """

        return self.x, self.y

    def distance_to(
        self,
        other: "SpatialNode"
    ) -> float:
        """
        Calculate Euclidean distance to another spatial node.

        Because one coordinate unit represents one kilometre,
        the returned value is measured in kilometres.
        """

        if not isinstance(other, SpatialNode):
            raise TypeError(
                "Distance can only be calculated between SpatialNode objects."
            )

        return math.hypot(
            other.x - self.x,
            other.y - self.y
        )

    def activate(self) -> None:
        self.active = True

    def deactivate(self) -> None:
        self.active = False

    def authenticate(self) -> None:
        self.authenticated = True

    def revoke_authentication(self) -> None:
        self.authenticated = False

    def reserve(
        self,
        session_id: str
    ) -> None:
        """
        Mark the node as reserved by a network session.

        The full reservation manager will be introduced later.
        """

        if not session_id:
            raise ValueError(
                "A non-empty session ID is required."
            )

        if self.busy:
            raise RuntimeError(
                f"Node {self.name} is already reserved by "
                f"session {self.reserved_by_session}."
            )

        self.busy = True
        self.reserved_by_session = session_id

    def release(
        self,
        session_id: Optional[str] = None
    ) -> None:
        """
        Release the node.

        When session_id is supplied, it must match the session
        currently holding the reservation.
        """

        if (
            session_id is not None
            and self.reserved_by_session is not None
            and session_id != self.reserved_by_session
        ):
            raise RuntimeError(
                f"Session {session_id} cannot release node "
                f"{self.name}; it is reserved by "
                f"{self.reserved_by_session}."
            )

        self.busy = False
        self.reserved_by_session = None

    def is_control_plane_eligible(self) -> bool:
        """
        Return whether the control plane may currently use this node.
        """

        return (
            self.active
            and self.authenticated
            and not self.busy
        )

    def status_summary(self) -> dict:
        return {
            "name": self.name,
            "role": self.role,
            "x_km": self.x,
            "y_km": self.y,
            "active": self.active,
            "authenticated": self.authenticated,
            "busy": self.busy,
            "reserved_by_session":
                self.reserved_by_session,
            "control_plane_eligible":
                self.is_control_plane_eligible()
        }

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name='{self.name}', "
            f"role='{self.role}', "
            f"x={self.x:.2f}, "
            f"y={self.y:.2f})"
        )

    def __str__(self) -> str:
        return (
            f"{self.name} ({self.role}) "
            f"at ({self.x:.2f}, {self.y:.2f}) km"
        )

class UserNode(SpatialNode):

    def __init__(
        self,
        name: str,
        x: float,
        y: float,
        active: bool = True,
        authenticated: bool = True
    ):
        super().__init__(
            name=name,
            role=ROLE_USER,
            x=x,
            y=y,
            active=active,
            authenticated=authenticated
        )

class RepeaterNode(SpatialNode):

    def __init__(
        self,
        name: str,
        x: float,
        y: float,
        memory_capacity: int = 2,
        memory_lifetime_attempts: int = 1000000,
        swapping_success_probability: float = 1.0,
        active: bool = True,
        authenticated: bool = True
    ):
        super().__init__(
            name=name,
            role=ROLE_REPEATER,
            x=x,
            y=y,
            active=active,
            authenticated=authenticated
        )

        if (
            not isinstance(memory_capacity, int)
            or isinstance(memory_capacity, bool)
            or memory_capacity < 2
        ):
            raise ValueError(
                "A linear-chain repeater requires a memory "
                "capacity of at least 2."
            )

        if (
            not isinstance(memory_lifetime_attempts, int)
            or isinstance(memory_lifetime_attempts, bool)
            or memory_lifetime_attempts <= 0
        ):
            raise ValueError(
                "Memory lifetime must be a positive integer."
            )

        if not (
            0.0
            <= swapping_success_probability
            <= 1.0
        ):
            raise ValueError(
                "Swapping success probability must lie "
                "between 0 and 1."
            )

        self.memory_capacity = memory_capacity

        self.memory_lifetime_attempts = (
            memory_lifetime_attempts
        )

        self.swapping_success_probability = float(
            swapping_success_probability
        )

        # Phase 3A counters only.
        # Their behaviour begins in later subphases.
        self.swap_attempts = 0
        self.successful_swaps = 0
        self.failed_swaps = 0

        # Placeholder for the QuantumMemory object introduced
        # in Phase 3B.
        self.quantum_memory = None

    def repeater_summary(self) -> dict:
        summary = self.status_summary()

        summary.update({
            "memory_capacity" : self.memory_capacity,
            "memory_lifetime_attempts" : self.memory_lifetime_attempts,
            "swapping_success_probability" : self.swapping_success_probability,
            "swap_attempts" : self.swap_attempts,
            "successful_swaps" : self.successful_swaps,
            "failed_swaps" : self.failed_swaps,
            "quantum_memory_initialized" : self.quantum_memory is not None
        })

        return summary

In [ ]:
# @title Sanity Test : Spatial Node

# phase3a_test_user = UserNode(name="TestUser", x=0, y=0)

# phase3a_test_repeater = RepeaterNode(name="TestRepeater", x=30, y=40)

# print(phase3a_test_user)
# print(phase3a_test_repeater)

# print(f"\nDistance : {phase3a_test_user.distance_to(phase3a_test_repeater):.2f} km")

# assert (
#     abs(phase3a_test_user.distance_to(phase3a_test_repeater) - 50.0) < 1e-12
# )

# assert (
#     phase3a_test_repeater.role == ROLE_REPEATER
# )

# assert (
#     phase3a_test_user.role == ROLE_USER
# )

# assert (
#     phase3a_test_repeater.memory_capacity >= 2
# )

# print(
#     "Phase 3A spatial-node sanity test passed."
# )

In [ ]:
# @title Spatial Quantum Link

class SpatialQuantumLink(QuantumLink):
    """
    Coordinate-aware extension of the existing QuantumLink.

    It automatically derives physical link distance from endpoint coordinates and then delegates attenuation and photon-loss modelling to the already validated QuantumLink implementation.
    """

    def __init__(
        self,
        source: SpatialNode,
        destination: SpatialNode,
        attenuation_db_per_km: float = STANDARD_FIBER_ATTENUATION,
        active: bool = True,
        link_id: Optional[str] = None
    ):
        if not isinstance(source, SpatialNode):
            raise TypeError(
                "SpatialQuantumLink source must be a SpatialNode."
            )

        if not isinstance(destination, SpatialNode):
            raise TypeError(
                "SpatialQuantumLink destination must be a SpatialNode."
            )

        if source is destination:
            raise ValueError(
                "A node cannot be linked to itself."
            )

        calculated_distance = source.distance_to(destination)

        if calculated_distance <= 0:
            raise ValueError(
                "Quantum-link distance must be greater than zero."
            )

        super().__init__(
            source=source,
            destination=destination,
            distance_km=calculated_distance,
            attenuation_db_per_km=attenuation_db_per_km
        )

        self.link_id = (
            link_id
            if link_id is not None
            else self.canonical_link_id(
                source.name,
                destination.name
            )
        )

        self.status = (
            "ACTIVE"
            if active
            else "INACTIVE"
        )

        self.bell_pair_attempts = 0
        self.successful_bell_pairs = 0

        self.reserved_by_session = None

    @staticmethod
    def canonical_link_id(
        node_a_name: str,
        node_b_name: str
    ) -> str:
        """
        Create one stable ID for an undirected physical link.

        Alice--R1 and R1--Alice therefore resolve to the same ID.
        """

        ordered_names = sorted([
            str(node_a_name),
            str(node_b_name)
        ])

        return (
            f"{ordered_names[0]}"
            f"--"
            f"{ordered_names[1]}"
        )

    @property
    def active(self) -> bool:
        return self.status == "ACTIVE"

    def activate(self) -> None:
        self.status = "ACTIVE"

    def deactivate(self) -> None:
        self.status = "INACTIVE"

    def involves(self, node: SpatialNode) -> bool:
        return (
            self.source is node
            or self.destination is node
        )

    def other_node(self, node: SpatialNode) -> SpatialNode:
        if self.source is node:
            return self.destination

        if self.destination is node:
            return self.source

        raise ValueError(
            f"Node {node.name} is not an endpoint "
            f"of link {self.link_id}."
        )

    def reserve(self, session_id: str) -> None:
        if not session_id:
            raise ValueError("A non-empty session ID is required.")

        if self.reserved_by_session is not None:
            raise RuntimeError(
                f"Link {self.link_id} is already reserved "
                f"by session {self.reserved_by_session}."
            )

        self.reserved_by_session = session_id

    def release(self, session_id: Optional[str] = None) -> None:
        if (
            session_id is not None
            and self.reserved_by_session is not None
            and session_id != self.reserved_by_session
        ):
            raise RuntimeError(
                f"Session {session_id} cannot release link "
                f"{self.link_id}; it is reserved by "
                f"{self.reserved_by_session}."
            )

        self.reserved_by_session = None

    def is_control_plane_eligible(self) -> bool:
        return (
            self.active
            and self.source.active
            and self.destination.active
            and self.source.authenticated
            and self.destination.authenticated
            and self.reserved_by_session is None
        )

    def link_summary(self) -> dict:
        return {
            "link_id" : self.link_id,
            "source" : self.source.name,
            "destination" : self.destination.name,
            "distance_km" : self.distance_km,
            "attenuation_db_per_km" : self.attenuation,
            "survival_probability" : self.survival_probability,
            "survival_probability_percent" : self.survival_probability * 100,
            "loss_probability" : self.loss_probability,
            "loss_probability_percent" : self.loss_probability * 100,
            "status" : self.status,
            "reserved_by_session" : self.reserved_by_session,
            "control_plane_eligible" : self.is_control_plane_eligible(),
            "lifetime_sent" : self.statistics["lifetime_sent"],
            "lifetime_received" : self.statistics["lifetime_received"],
            "lifetime_lost" : self.statistics["lifetime_lost"],
            "bell_pair_attempts" : self.bell_pair_attempts,
            "successful_bell_pairs" : self.successful_bell_pairs
        }

    def __repr__(self) -> str:
        return (
            f"SpatialQuantumLink("
            f"{self.source.name} ↔ "
            f"{self.destination.name}, "
            f"d={self.distance_km:.2f} km, "
            f"η={self.survival_probability:.6f})"
        )

In [ ]:
# @title Sanity Test : Spatial Quantum Link

# phase3a_test_link = SpatialQuantumLink(source = phase3a_test_user, destination = phase3a_test_repeater)

# print(phase3a_test_link)

# print(
#     f"\nAutomatically calculated distance : {phase3a_test_link.distance_km:.2f} km"
# )

# print(
#     f"Photon survival probability : {phase3a_test_link.survival_probability * 100:.4f}%"
# )

# assert (
#     abs(phase3a_test_link.distance_km - 50.0) < 1e-12
# )

# expected_survival = 10 ** (-(STANDARD_FIBER_ATTENUATION * 50.0) / 10)

# assert (
#     abs(phase3a_test_link.survival_probability - expected_survival) < 1e-12
# )

# print(
#     "\nPhase 3A spatial-link sanity test passed."
# )

In [ ]:
# @title Network Event Record

from dataclasses import dataclass, field
from typing import Any
import time


@dataclass
class NetworkEvent:

    event_index: int
    event_type: str
    message: str

    node_names: tuple[str, ...] = field(
        default_factory=tuple
    )

    link_ids: tuple[str, ...] = field(
        default_factory=tuple
    )

    session_id: Optional[str] = None

    simulation_step: Optional[int] = None

    timestamp_s: float = field(
        default_factory=time.perf_counter
    )

    metadata: dict[str, Any] = field(
        default_factory=dict
    )

    def as_dict(self) -> dict:
        return {
            "event_index":
                self.event_index,

            "event_type":
                self.event_type,

            "message":
                self.message,

            "node_names":
                list(self.node_names),

            "link_ids":
                list(self.link_ids),

            "session_id":
                self.session_id,

            "simulation_step":
                self.simulation_step,

            "timestamp_s":
                self.timestamp_s,

            "metadata":
                self.metadata.copy()
        }

In [ ]:
# @title Quantum Network Control Plane

class QuantumNetwork:
    """
    Central NARAD-QC topology and control-plane abstraction.

    The network stores one authoritative graph.

    Physical and control-plane views are derived from the state of
    the same nodes and links instead of maintaining duplicate graphs.

    Phase 3A responsibilities:
    - register spatial nodes;
    - create physical quantum links;
    - maintain adjacency information;
    - validate manually supplied paths;
    - report topology;
    - record topology events;
    - support static spatial visualization.

    Future responsibilities:
    - authentication policy;
    - path calculation;
    - reservations;
    - session orchestration;
    - event-driven animation;
    - concurrent resource scheduling.
    """

    def __init__(
        self,
        name: str = "NARAD-QC Network"
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Network name must be a non-empty string."
            )

        self.name = name.strip()
        self.nodes: dict[str, SpatialNode] = {}
        self.links: dict[str, SpatialQuantumLink] = {}
        self.adjacency: dict[str, set[str]] = {}
        self.sessions: dict[str, dict] = {}
        self.event_log: list[NetworkEvent] = []
        self._next_event_index = 1
        self.log_event(
            event_type="NETWORK_CREATED",
            message=(
                f"Quantum network '{self.name}' created."
            )
        )

    # =====================================================
    # Event Logging
    # =====================================================

    def log_event(
        self,
        event_type: str,
        message: str,
        node_names: tuple[str, ...] = (),
        link_ids: tuple[str, ...] = (),
        session_id: Optional[str] = None,
        simulation_step: Optional[int] = None,
        metadata: Optional[dict] = None
    ) -> NetworkEvent:

        if not event_type:
            raise ValueError(
                "Event type cannot be empty."
            )

        event = NetworkEvent(
            event_index=self._next_event_index,
            event_type=str(event_type),
            message=str(message),
            node_names=tuple(node_names),
            link_ids=tuple(link_ids),
            session_id=session_id,
            simulation_step=simulation_step,
            metadata=(
                metadata.copy()
                if metadata is not None
                else {}
            )
        )

        self.event_log.append(event)
        self._next_event_index += 1

        return event

    def events_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            event.as_dict()
            for event in self.event_log
        )

    # =====================================================
    # Node Management
    # =====================================================

    def add_node(
        self,
        node: SpatialNode
    ) -> SpatialNode:

        if not isinstance(node, SpatialNode):
            raise TypeError(
                "QuantumNetwork accepts SpatialNode objects only."
            )

        if node.name in self.nodes:
            raise ValueError(
                f"A node named '{node.name}' already exists in the network."
            )

        self.nodes[node.name] = node
        self.adjacency[node.name] = set()

        self.log_event(
            event_type="NODE_ADDED",
            message=(
                f"Node {node.name} added at "
                f"({node.x:.2f}, {node.y:.2f}) km."
            ),
            node_names=(node.name,),
            metadata=node.status_summary()
        )

        return node

    def add_nodes(
        self,
        nodes: list[SpatialNode]
    ) -> None:

        for node in nodes:
            self.add_node(node)

    def get_node(
        self,
        node_or_name
    ) -> SpatialNode:

        if isinstance(node_or_name, SpatialNode):
            registered_node = self.nodes.get(
                node_or_name.name
            )

            if registered_node is not node_or_name:
                raise KeyError(
                    f"Node '{node_or_name.name}' is not registered in this network instance."
                )

            return registered_node

        if isinstance(node_or_name, str):
            if node_or_name not in self.nodes:
                raise KeyError(
                    f"Node '{node_or_name}' does not exist "
                    f"in network '{self.name}'."
                )

            return self.nodes[node_or_name]

        raise TypeError(
            "Node reference must be a SpatialNode or node-name string."
        )

    def remove_node(
        self,
        node_or_name
    ) -> SpatialNode:

        node = self.get_node(node_or_name)

        if self.adjacency[node.name]:
            raise RuntimeError(
                f"Node {node.name} cannot be removed while it still has connected links."
            )

        del self.nodes[node.name]
        del self.adjacency[node.name]

        self.log_event(
            event_type="NODE_REMOVED",
            message=(
                f"Node {node.name} removed from network."
            ),
            node_names=(node.name,)
        )

        return node

    # =====================================================
    # Link Management
    # =====================================================

    @staticmethod
    def link_id_for(
        node_a_name: str,
        node_b_name: str
    ) -> str:
        return SpatialQuantumLink.canonical_link_id(
            node_a_name,
            node_b_name
        )

    def add_link(
        self,
        source,
        destination,
        attenuation_db_per_km: float = STANDARD_FIBER_ATTENUATION,
        active: bool = True
    ) -> SpatialQuantumLink:

        source_node = self.get_node(source)
        destination_node = self.get_node(destination)

        if source_node is destination_node:
            raise ValueError(
                "A self-link is not permitted."
            )

        link_id = self.link_id_for(
            source_node.name,
            destination_node.name
        )

        if link_id in self.links:
            raise ValueError(
                f"Link {link_id} already exists."
            )

        link = SpatialQuantumLink(
            source=source_node,
            destination=destination_node,
            attenuation_db_per_km=(
                attenuation_db_per_km
            ),
            active=active,
            link_id=link_id
        )

        self.links[link_id] = link

        self.adjacency[source_node.name].add(
            destination_node.name
        )

        self.adjacency[destination_node.name].add(
            source_node.name
        )

        self.log_event(
            event_type="LINK_ADDED",
            message=(
                f"Quantum link {link_id} added "
                f"({link.distance_km:.2f} km)."
            ),
            node_names=(
                source_node.name,
                destination_node.name
            ),
            link_ids=(link_id,),
            metadata=link.link_summary()
        )

        return link

    def get_link(
        self,
        node_a,
        node_b
    ) -> SpatialQuantumLink:

        first = self.get_node(node_a)
        second = self.get_node(node_b)

        link_id = self.link_id_for(
            first.name,
            second.name
        )

        if link_id not in self.links:
            raise KeyError(
                f"No quantum link exists between "
                f"{first.name} and {second.name}."
            )

        return self.links[link_id]

    def remove_link(
        self,
        node_a,
        node_b
    ) -> SpatialQuantumLink:

        link = self.get_link(
            node_a,
            node_b
        )

        del self.links[link.link_id]

        self.adjacency[
            link.source.name
        ].discard(
            link.destination.name
        )

        self.adjacency[
            link.destination.name
        ].discard(
            link.source.name
        )

        self.log_event(
            event_type="LINK_REMOVED",
            message=(
                f"Quantum link {link.link_id} removed."
            ),
            node_names=(
                link.source.name,
                link.destination.name
            ),
            link_ids=(link.link_id,)
        )

        return link

    # =====================================================
    # Topology Queries
    # =====================================================

    def neighbours(
        self,
        node_or_name,
        control_plane_only: bool = False
    ) -> list[SpatialNode]:

        node = self.get_node(node_or_name)

        neighbour_nodes = [
            self.nodes[neighbour_name]
            for neighbour_name
            in sorted(self.adjacency[node.name])
        ]

        if not control_plane_only:
            return neighbour_nodes

        eligible_neighbours = []

        for neighbour in neighbour_nodes:
            link = self.get_link(
                node,
                neighbour
            )

            if (
                neighbour.is_control_plane_eligible()
                and link.is_control_plane_eligible()
            ):
                eligible_neighbours.append(neighbour)

        return eligible_neighbours

    def degree(
        self,
        node_or_name
    ) -> int:
        node = self.get_node(node_or_name)

        return len(
            self.adjacency[node.name]
        )

    def total_geometric_link_length(self) -> float:
        return sum(
            link.distance_km
            for link in self.links.values()
        )

    # =====================================================
    # Manual Path Validation
    # =====================================================

    def validate_path(
        self,
        path,
        require_control_plane_eligibility: bool = True,
        require_repeater_intermediates: bool = True
    ) -> dict:
        """
        Validate one manually supplied ordered path.

        Routing is deliberately not performed in Phase 3A.
        Phase 4 will produce paths and pass them through this
        same validator.
        """

        if not isinstance(path, (list, tuple)):
            raise TypeError(
                "Path must be supplied as a list or tuple."
            )

        if len(path) < 2:
            raise ValueError(
                "A network path requires at least "
                "a source and destination."
            )

        resolved_path = [
            self.get_node(item)
            for item in path
        ]

        node_names = [
            node.name
            for node in resolved_path
        ]

        if len(set(node_names)) != len(node_names):
            raise ValueError(
                "Path contains a repeated node and therefore "
                "contains a loop."
            )

        source = resolved_path[0]
        destination = resolved_path[-1]
        intermediate_nodes = resolved_path[1:-1]

        if source.role != ROLE_USER:
            raise ValueError(
                "Path source must be a user node."
            )

        if destination.role != ROLE_USER:
            raise ValueError(
                "Path destination must be a user node."
            )

        if (
            require_repeater_intermediates
            and any(
                node.role != ROLE_REPEATER
                for node in intermediate_nodes
            )
        ):
            raise ValueError(
                "Every intermediate node in a Phase 3 "
                "repeater chain must be a repeater."
            )

        path_links = []

        for index in range(
            len(resolved_path) - 1
        ):
            current_node = resolved_path[index]
            next_node = resolved_path[index + 1]

            link = self.get_link(
                current_node,
                next_node
            )

            path_links.append(link)

        if require_control_plane_eligibility:
            ineligible_nodes = [
                node.name
                for node in resolved_path
                if not node.is_control_plane_eligible()
            ]

            ineligible_links = [
                link.link_id
                for link in path_links
                if not link.is_control_plane_eligible()
            ]

            if ineligible_nodes:
                raise RuntimeError(
                    "Path contains control-plane-ineligible "
                    f"nodes: {ineligible_nodes}"
                )

            if ineligible_links:
                raise RuntimeError(
                    "Path contains control-plane-ineligible "
                    f"links: {ineligible_links}"
                )

        segment_distances = [
            link.distance_km
            for link in path_links
        ]

        segment_survivals = [
            link.survival_probability
            for link in path_links
        ]

        summary = {
            "valid":
                True,

            "source":
                source.name,

            "destination":
                destination.name,

            "path":
                node_names,

            "path_string":
                " -> ".join(node_names),

            "hop_count":
                len(path_links),

            "repeater_count":
                len(intermediate_nodes),

            "total_path_distance_km":
                sum(segment_distances),

            "segment_distances_km":
                segment_distances,

            "maximum_segment_distance_km":
                max(segment_distances),

            "minimum_segment_distance_km":
                min(segment_distances),

            "segment_survival_probabilities":
                segment_survivals,

            "minimum_segment_survival_probability":
                min(segment_survivals),

            "link_ids":
                [
                    link.link_id
                    for link in path_links
                ]
        }

        self.log_event(
            event_type="PATH_VALIDATED",
            message=(
                f"Path validated: "
                f"{summary['path_string']}."
            ),
            node_names=tuple(node_names),
            link_ids=tuple(summary["link_ids"]),
            metadata=summary
        )

        return summary

    # =====================================================
    # Reporting
    # =====================================================

    def nodes_dataframe(self) -> pd.DataFrame:
        rows = []

        for node in self.nodes.values():
            if isinstance(node, RepeaterNode):
                rows.append(
                    node.repeater_summary()
                )
            else:
                rows.append(
                    node.status_summary()
                )

        return pd.DataFrame(rows)

    def links_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            link.link_summary()
            for link in self.links.values()
        )

    def topology_summary(self) -> dict:
        user_count = sum(
            node.role == ROLE_USER
            for node in self.nodes.values()
        )

        repeater_count = sum(
            node.role == ROLE_REPEATER
            for node in self.nodes.values()
        )

        active_link_count = sum(
            link.active
            for link in self.links.values()
        )

        return {
            "network_name":
                self.name,

            "total_nodes":
                len(self.nodes),

            "user_nodes":
                user_count,

            "repeater_nodes":
                repeater_count,

            "total_links":
                len(self.links),

            "active_links":
                active_link_count,

            "total_geometric_link_length_km":
                self.total_geometric_link_length(),

            "recorded_events":
                len(self.event_log),

            "active_sessions":
                len(self.sessions)
        }

    def print_topology_summary(self) -> None:
        summary = self.topology_summary()

        print("=" * 72)
        print(f"Quantum Network: {self.name}")
        print("=" * 72)

        for key, value in summary.items():
            formatted_key = key.replace(
                "_",
                " "
            ).title()

            if isinstance(value, float):
                print(
                    f"{formatted_key:<40}: "
                    f"{value:.4f}"
                )
            else:
                print(
                    f"{formatted_key:<40}: "
                    f"{value}"
                )

        print("=" * 72)

In [ ]:
# @title Static Spatial Network Visualisation

from matplotlib.lines import Line2D


def draw_quantum_network(
    network: QuantumNetwork,
    highlighted_path=None,
    show_distances: bool = True,
    show_survival_probabilities: bool = False,
    show_coordinates: bool = True,
    save_path: Optional[str] = None,
    figure_size: tuple = (13, 8)
):
    """
    Draw one spatial NARAD-QC topology using fixed node coordinates.

    Parameters
    ----------
    network:
        QuantumNetwork to draw.

    highlighted_path:
        Optional ordered node path. When supplied, its links receive
        stronger visual emphasis. The path is validated first.

    show_distances:
        Display physical link lengths in kilometres.

    show_survival_probabilities:
        Display per-link photon-survival probabilities.

    show_coordinates:
        Include node coordinates in labels.

    save_path:
        Optional output image path.

    Notes
    -----
    The animation layer will later replay NetworkEvent objects using
    the same coordinate system and rendering conventions.
    """

    if not isinstance(network, QuantumNetwork):
        raise TypeError(
            "network must be a QuantumNetwork instance."
        )

    highlighted_link_ids = set()
    highlighted_node_names = set()

    if highlighted_path is not None:
        validated_path = network.validate_path(highlighted_path, require_control_plane_eligibility=False)

        highlighted_link_ids = set(validated_path["link_ids"])

        highlighted_node_names = set(validated_path["path"])

    figure, axis = plt.subplots(figsize=figure_size)

    # =====================================================
    # Draw Links
    # =====================================================

    for link in network.links.values():

        x_values = [
            link.source.x,
            link.destination.x
        ]

        y_values = [
            link.source.y,
            link.destination.y
        ]

        is_highlighted = (
            link.link_id
            in highlighted_link_ids
        )

        line_width = (
            3.5
            if is_highlighted
            else 1.8
        )

        line_style = (
            "-"
            if link.active
            else "--"
        )

        line_alpha = (
            1.0
            if is_highlighted
            else 0.65
        )

        axis.plot(
            x_values,
            y_values,
            linewidth=line_width,
            linestyle=line_style,
            alpha=line_alpha,
            zorder=1
        )

        midpoint_x = (link.source.x + link.destination.x) / 2

        midpoint_y = (link.source.y + link.destination.y) / 2

        annotation_parts = []

        if show_distances:
            annotation_parts.append(f"{link.distance_km:.1f} km")

        if show_survival_probabilities:
            annotation_parts.append(
                f"η={link.survival_probability:.4f}"
            )

        if annotation_parts:
            axis.annotate(
                "\n".join(annotation_parts),
                xy=(
                    midpoint_x,
                    midpoint_y
                ),
                xytext=(0, 7),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
                bbox={
                    "boxstyle":
                        "round,pad=0.25",

                    "facecolor":
                        "white",

                    "alpha":
                        0.8,

                    "edgecolor":
                        "none"
                },
                zorder=3
            )

    # =====================================================
    # Draw Nodes
    # =====================================================

    for node in network.nodes.values():

        is_repeater = (
            node.role == ROLE_REPEATER
        )

        is_highlighted = (
            node.name
            in highlighted_node_names
        )

        marker = (
            "s"
            if is_repeater
            else "o"
        )

        marker_size = (
            210
            if is_highlighted
            else 155
        )

        edge_width = (
            2.8
            if is_highlighted
            else 1.5
        )

        alpha = (
            1.0
            if node.active
            else 0.35
        )

        axis.scatter(
            node.x,
            node.y,
            marker=marker,
            s=marker_size,
            linewidths=edge_width,
            edgecolors="black",
            alpha=alpha,
            zorder=4
        )

        coordinate_text = (
            f"\n({node.x:.1f}, {node.y:.1f})"
            if show_coordinates
            else ""
        )

        status_tags = []

        if not node.active:
            status_tags.append("OFFLINE")

        if not node.authenticated:
            status_tags.append("UNAUTHENTICATED")

        if node.busy:
            status_tags.append("BUSY")

        status_text = (
            "\n[" + ", ".join(status_tags) + "]"
            if status_tags
            else ""
        )

        axis.annotate(
            (
                f"{node.name}"
                f"\n{node.role}"
                f"{coordinate_text}"
                f"{status_text}"
            ),
            xy=(node.x, node.y),
            xytext=(0, 13),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight=(
                "bold"
                if is_highlighted
                else "normal"
            ),
            zorder=5
        )

    # =====================================================
    # Layout
    # =====================================================

    all_x = [
        node.x
        for node in network.nodes.values()
    ]

    all_y = [
        node.y
        for node in network.nodes.values()
    ]

    if all_x and all_y:
        x_span = max(all_x) - min(all_x)
        y_span = max(all_y) - min(all_y)

        x_margin = max(
            10.0,
            x_span * 0.10
        )

        y_margin = max(
            10.0,
            y_span * 0.15
        )

        axis.set_xlim(
            min(all_x) - x_margin,
            max(all_x) + x_margin
        )

        axis.set_ylim(
            min(all_y) - y_margin,
            max(all_y) + y_margin
        )

    axis.set_xlabel(
        "X Coordinate (km)"
    )

    axis.set_ylabel(
        "Y Coordinate (km)"
    )

    axis.set_title(
        f"{network.name}\n"
        "Spatial Quantum Network Topology"
    )

    axis.set_aspect(
        "equal",
        adjustable="box"
    )

    axis.grid(
        True,
        alpha=0.3
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markeredgecolor="black",
            markersize=10,
            label="User Node"
        ),
        Line2D(
            [0],
            [0],
            marker="s",
            linestyle="None",
            markeredgecolor="black",
            markersize=10,
            label="Repeater Node"
        ),
        Line2D(
            [0],
            [0],
            linestyle="-",
            linewidth=2,
            label="Active Quantum Link"
        ),
        Line2D(
            [0],
            [0],
            linestyle="--",
            linewidth=2,
            label="Inactive Quantum Link"
        )
    ]

    axis.legend(
        handles=legend_handles,
        loc="best"
    )

    figure.tight_layout()

    if save_path is not None:
        figure.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

    return figure, axis

In [ ]:
# @title Build Initial NARAD-QC Spatial Topology

phase3a_network = QuantumNetwork(
    name="NARAD-QC Phase 3A Network"
)

alice_p3 = UserNode(
    name="Alice",
    x=0,
    y=0
)

bob_p3 = UserNode(
    name="Bob",
    x=170,
    y=30
)

charlie_p3 = UserNode(
    name="Charlie",
    x=60,
    y=95
)

r1_p3 = RepeaterNode(
    name="R1",
    x=35,
    y=20
)

r2_p3 = RepeaterNode(
    name="R2",
    x=80,
    y=20
)

r3_p3 = RepeaterNode(
    name="R3",
    x=120,
    y=60
)

r4_p3 = RepeaterNode(
    name="R4",
    x=90,
    y=-25
)

phase3a_network.add_nodes([
    alice_p3,
    bob_p3,
    charlie_p3,
    r1_p3,
    r2_p3,
    r3_p3,
    r4_p3
])

phase3a_network.add_link(
    alice_p3,
    r1_p3
)

phase3a_network.add_link(
    r1_p3,
    r2_p3
)

phase3a_network.add_link(
    r2_p3,
    r3_p3
)

phase3a_network.add_link(
    r3_p3,
    bob_p3
)

print(
    "Phase 3A topology created successfully."
)

In [ ]:
# @title Validate Linear Repeater Chain

phase3a_linear_path = [
    alice_p3,
    r1_p3,
    r2_p3,
    r3_p3,
    bob_p3
]

phase3a_path_summary = (
    phase3a_network.validate_path(
        path=phase3a_linear_path,
        require_control_plane_eligibility=True,
        require_repeater_intermediates=True
    )
)

print("=" * 76)
print("Phase 3A Linear Repeater Path Validation")
print("=" * 76)

for key, value in phase3a_path_summary.items():

    formatted_key = key.replace(
        "_",
        " "
    ).title()

    if isinstance(value, float):
        print(
            f"{formatted_key:<44}: "
            f"{value:.6f}"
        )

    elif (
        isinstance(value, list)
        and value
        and all(
            isinstance(item, float)
            for item in value
        )
    ):
        formatted_values = [
            round(item, 6)
            for item in value
        ]

        print(
            f"{formatted_key:<44}: "
            f"{formatted_values}"
        )

    else:
        print(
            f"{formatted_key:<44}: "
            f"{value}"
        )

print("=" * 76)


# =========================================================
# Critical Assertions
# =========================================================

assert (
    phase3a_path_summary["valid"]
    is True
)

assert (
    phase3a_path_summary["source"]
    == "Alice"
)

assert (
    phase3a_path_summary["destination"]
    == "Bob"
)

assert (
    phase3a_path_summary["repeater_count"]
    == 3
)

assert (
    phase3a_path_summary["hop_count"]
    == 4
)

assert (
    len(
        phase3a_path_summary[
            "segment_distances_km"
        ]
    )
    == 4
)

print(
    "\nPhase 3A path validation passed."
)

In [ ]:
# @title Network Nodes and Links

print("NETWORK SUMMARY")
phase3a_network.print_topology_summary()


print("\nNODE TABLE")
df_phase3a_nodes = phase3a_network.nodes_dataframe()

display(df_phase3a_nodes)


print("\nLINK TABLE")
df_phase3a_links = phase3a_network.links_dataframe()

display(df_phase3a_links)


print("\nEVENT LOG")
df_phase3a_events = phase3a_network.events_dataframe()

display(df_phase3a_events)

In [ ]:
# @title Plot for Visualisation of Phase 3A Network

draw_quantum_network(
    network = phase3a_network,
    highlighted_path=None,
    show_distances = True,
    show_survival_probabilities= False,
    show_coordinates = True,
    save_path = "/content/Phase3A",
    figure_size = (13, 8)
)

### Phase 3B

Adding Quantum Memory Class for Repeaters

In [ ]:
# @title Memory Errors and Network Bell Pair Identity

import itertools
from typing import Optional, Any

# Quantum Memory Exceptions

class QuantumMemoryError(Exception):
    """
    Base error for all quantum-memory operations.
    """

class QuantumMemoryFullError(QuantumMemoryError):
    """
    Raised when no free memory slot is available.
    """

class QuantumMemoryPairNotFoundError(QuantumMemoryError):
    """
    Raised when a requested Bell pair is not stored.
    """

class QuantumMemoryDuplicatePairError(QuantumMemoryError):
    """
    Raised when the same Bell pair is stored twice in one memory.
    """

class QuantumMemoryOwnershipError(QuantumMemoryError):
    """
    Raised when a repeater tries to store a Bell pair
    that does not involve that repeater.
    """

# Phase 3 Bell Pair Identity

class NetworkBellPair(BellPair):

    _pair_counter = itertools.count(1)

    def __init__(
        self,
        node_a: Node,
        node_b: Node,
        session_id: str,
        source_link_id: Optional[str] = None,
        creation_attempt: int = 0,
        source_links: Optional[list[str]] = None,
        fidelity: float = 1.0,
        pair_id: Optional[str] = None
    ):
        if not isinstance(session_id, str) or not session_id.strip():
            raise ValueError(
                "NetworkBellPair requires a non-empty session ID."
            )

        if (
            not isinstance(creation_attempt, int)
            or isinstance(creation_attempt, bool)
            or creation_attempt < 0
        ):
            raise ValueError(
                "creation_attempt must be a non-negative integer."
            )

        if node_a is node_b:
            raise ValueError(
                "A Bell pair requires two different endpoint nodes."
            )

        if pair_id is None:
            pair_id = (
                f"BP-{next(self._pair_counter):06d}"
            )

        if source_links is None:
            source_links = (
                [source_link_id]
                if source_link_id is not None
                else []
            )

        normalized_source_links = [
            str(link_id)
            for link_id in source_links
            if link_id is not None
        ]

        source_distance = None

        if (
            source_link_id is not None
            and "phase3a_network" in globals()
            and source_link_id
            in phase3a_network.links
        ):
            source_distance = (
                phase3a_network
                .links[source_link_id]
                .distance_km
            )

        super().__init__(
            node_a=node_a,
            node_b=node_b,
            fidelity=fidelity,
            source_link_distance_km=source_distance
        )

        self.pair_id = str(pair_id)
        self.session_id = session_id.strip()
        self.source_link_id = source_link_id
        self.creation_attempt = creation_attempt
        self.source_links = normalized_source_links

        self.active = True
        self.consumed = False
        self.discard_reason = None

    def deactivate(
        self,
        reason: Optional[str] = None
    ) -> None:
        self.active = False
        self.discard_reason = reason

    def mark_consumed(
        self,
        reason: str = "ENTANGLEMENT_SWAP"
    ) -> None:
        self.active = False
        self.consumed = True
        self.discard_reason = reason

    def summary(self) -> dict:
        return {
            "pair_id":
                self.pair_id,

            "node_a":
                self.node_a.name,

            "node_b":
                self.node_b.name,

            "session_id":
                self.session_id,

            "source_link_id":
                self.source_link_id,

            "creation_attempt":
                self.creation_attempt,

            "source_links":
                self.source_links.copy(),

            "fidelity":
                self.fidelity,

            "active":
                self.active,

            "consumed":
                self.consumed,

            "discard_reason":
                self.discard_reason
        }

    def __repr__(self) -> str:
        return (
            f"NetworkBellPair("
            f"id='{self.pair_id}', "
            f"{self.node_a.name} ↔ {self.node_b.name}, "
            f"session='{self.session_id}')"
        )

    def __str__(self) -> str:
        return (
            f"{self.pair_id}: "
            f"BellPair("
            f"{self.node_a.name} ↔ {self.node_b.name}, "
            f"session={self.session_id}, "
            f"F={self.fidelity:.4f})"
        )

In [ ]:
# @title Quantum Memory Slot

class QuantumMemorySlot:

    def __init__(
        self,
        owner: RepeaterNode,
        slot_index: int
    ):
        if not isinstance(owner, RepeaterNode):
            raise TypeError(
                "QuantumMemorySlot owner must be a RepeaterNode."
            )

        if (
            not isinstance(slot_index, int)
            or isinstance(slot_index, bool)
            or slot_index < 0
        ):
            raise ValueError(
                "slot_index must be a non-negative integer."
            )

        self.owner = owner
        self.slot_index = slot_index
        self.slot_id = (
            f"{owner.name}-MEM-{slot_index:02d}"
        )

        self.stored_pair: Optional[NetworkBellPair] = None
        self.session_id: Optional[str] = None
        self.source_link_id: Optional[str] = None

        self.stored_at_attempt: Optional[int] = None
        self.age_attempts = 0

        self.last_clear_reason: Optional[str] = None

    @property
    def occupied(self) -> bool:
        return self.stored_pair is not None

    @property
    def pair_id(self) -> Optional[str]:
        return (
            self.stored_pair.pair_id
            if self.stored_pair is not None
            else None
        )

    def store(
        self,
        pair: NetworkBellPair,
        current_attempt: int
    ) -> None:

        if self.occupied:
            raise QuantumMemoryError(
                f"Memory slot {self.slot_id} is already occupied "
                f"by pair {self.pair_id}."
            )

        if not isinstance(pair, NetworkBellPair):
            raise TypeError(
                "QuantumMemorySlot stores NetworkBellPair "
                "objects only."
            )

        if not pair.active:
            raise QuantumMemoryError(
                f"Inactive pair {pair.pair_id} cannot be stored."
            )

        if not pair.involves(self.owner):
            raise QuantumMemoryOwnershipError(
                f"Repeater {self.owner.name} cannot store "
                f"{pair.pair_id}; the pair does not involve "
                f"that repeater."
            )

        if (
            not isinstance(current_attempt, int)
            or isinstance(current_attempt, bool)
            or current_attempt < 0
        ):
            raise ValueError(
                "current_attempt must be a non-negative integer."
            )

        self.stored_pair = pair
        self.session_id = pair.session_id
        self.source_link_id = pair.source_link_id

        self.stored_at_attempt = current_attempt
        self.age_attempts = 0
        self.last_clear_reason = None

    def increment_age(
        self,
        steps: int = 1
    ) -> int:

        if (
            not isinstance(steps, int)
            or isinstance(steps, bool)
            or steps < 0
        ):
            raise ValueError(
                "Age increment must be a non-negative integer."
            )

        if self.occupied:
            self.age_attempts += steps

        return self.age_attempts

    def has_expired(
        self,
        lifetime_attempts: int
    ) -> bool:
        """
        Expiration convention:
            age >= configured lifetime  -> expired
        """

        if (
            not isinstance(lifetime_attempts, int)
            or isinstance(lifetime_attempts, bool)
            or lifetime_attempts <= 0
        ):
            raise ValueError(
                "lifetime_attempts must be a positive integer."
            )

        return (
            self.occupied
            and self.age_attempts >= lifetime_attempts
        )

    def clear(
        self,
        reason: str = "CLEARED"
    ) -> Optional[dict]:
        """
        Empty the slot and return a record of the removed resource.
        """

        if not self.occupied:
            self.last_clear_reason = reason
            return None

        removed_record = {
            "slot_id":
                self.slot_id,

            "owner":
                self.owner.name,

            "pair":
                self.stored_pair,

            "pair_id":
                self.stored_pair.pair_id,

            "session_id":
                self.session_id,

            "source_link_id":
                self.source_link_id,

            "stored_at_attempt":
                self.stored_at_attempt,

            "age_attempts":
                self.age_attempts,

            "reason":
                reason
        }

        self.stored_pair = None
        self.session_id = None
        self.source_link_id = None
        self.stored_at_attempt = None
        self.age_attempts = 0
        self.last_clear_reason = reason

        return removed_record

    def summary(self) -> dict:
        return {
            "owner" : self.owner.name,
            "slot_id" : self.slot_id,
            "slot_index" : self.slot_index,
            "occupied" : self.occupied,
            "pair_id" : self.pair_id,
            "pair_endpoints":
                (
                    (
                        f"{self.stored_pair.node_a.name}"
                        f" ↔ "
                        f"{self.stored_pair.node_b.name}"
                    )
                    if self.stored_pair is not None
                    else None
                ),
            "session_id" : self.session_id,
            "source_link_id" : self.source_link_id,
            "stored_at_attempt" : self.stored_at_attempt,
            "age_attempts" : self.age_attempts,
            "last_clear_reason" : self.last_clear_reason
        }

    def __repr__(self) -> str:
        if not self.occupied:
            return (
                f"QuantumMemorySlot("
                f"id='{self.slot_id}', empty)"
            )

        return (
            f"QuantumMemorySlot("
            f"id='{self.slot_id}', "
            f"pair='{self.pair_id}', "
            f"age={self.age_attempts})"
        )

In [ ]:
# @title Quantum Memory

class QuantumMemory:
    """
    Session-aware quantum memory owned by one repeater.
    It does not generate entanglement and does not perform swapping.
    """

    def __init__(
        self,
        owner: RepeaterNode,
        capacity: int,
        lifetime_attempts: int,
        network: Optional[QuantumNetwork] = None
    ):
        if not isinstance(owner, RepeaterNode):
            raise TypeError(
                "QuantumMemory owner must be a RepeaterNode."
            )

        if (
            not isinstance(capacity, int)
            or isinstance(capacity, bool)
            or capacity <= 0
        ):
            raise ValueError(
                "Quantum-memory capacity must be a positive integer."
            )

        if (
            not isinstance(lifetime_attempts, int)
            or isinstance(lifetime_attempts, bool)
            or lifetime_attempts <= 0
        ):
            raise ValueError(
                "Memory lifetime must be a positive integer."
            )

        if (
            network is not None
            and not isinstance(network, QuantumNetwork)
        ):
            raise TypeError(
                "network must be a QuantumNetwork or None."
            )

        self.owner = owner
        self.capacity = capacity
        self.lifetime_attempts = lifetime_attempts
        self.network = network

        self.slots = [
            QuantumMemorySlot(
                owner=owner,
                slot_index=index
            )
            for index in range(capacity)
        ]

        self.statistics = {
            "store_attempts": 0,
            "successful_stores": 0,
            "failed_stores_full": 0,
            "failed_stores_duplicate": 0,
            "failed_stores_ownership": 0,

            "retrievals": 0,
            "discards": 0,
            "expirations": 0,
            "consumptions": 0,
            "session_clears": 0,
            "full_clears": 0,

            "total_age_updates": 0,
            "peak_occupied_slots": 0
        }

    # =====================================================
    # Internal Helpers
    # =====================================================

    def _log_event(
        self,
        event_type: str,
        message: str,
        pair: Optional[NetworkBellPair] = None,
        slot: Optional[QuantumMemorySlot] = None,
        metadata: Optional[dict] = None,
        simulation_step: Optional[int] = None
    ) -> None:

        if self.network is None:
            return

        event_metadata = (
            metadata.copy()
            if metadata is not None
            else {}
        )

        if pair is not None:
            event_metadata["pair"] = pair.summary()

        if slot is not None:
            event_metadata["slot"] = slot.summary()

        self.network.log_event(
            event_type=event_type,
            message=message,
            node_names=(self.owner.name,),
            link_ids=(
                (pair.source_link_id,)
                if (
                    pair is not None
                    and pair.source_link_id is not None
                )
                else ()
            ),
            session_id=(
                pair.session_id
                if pair is not None
                else None
            ),
            simulation_step=simulation_step,
            metadata=event_metadata
        )

    def _update_peak_occupancy(self) -> None:
        self.statistics[
            "peak_occupied_slots"
        ] = max(
            self.statistics["peak_occupied_slots"],
            self.occupied_count
        )

    def _find_slot_by_pair_id(
        self,
        pair_id: str
    ) -> Optional[QuantumMemorySlot]:

        for slot in self.slots:
            if slot.pair_id == pair_id:
                return slot

        return None

    def _first_available_slot(
        self
    ) -> Optional[QuantumMemorySlot]:

        for slot in self.slots:
            if not slot.occupied:
                return slot

        return None

    # =====================================================
    # Occupancy Properties
    # =====================================================

    @property
    def occupied_count(self) -> int:
        return sum(
            slot.occupied
            for slot in self.slots
        )

    @property
    def available_count(self) -> int:
        return self.capacity - self.occupied_count

    @property
    def utilisation(self) -> float:
        return (
            self.occupied_count / self.capacity
            if self.capacity
            else 0.0
        )

    @property
    def utilisation_percentage(self) -> float:
        return self.utilisation * 100

    @property
    def peak_utilisation_percentage(self) -> float:
        return (
            self.statistics["peak_occupied_slots"]
            / self.capacity
            * 100
        )

    @property
    def is_full(self) -> bool:
        return self.occupied_count >= self.capacity

    @property
    def is_empty(self) -> bool:
        return self.occupied_count == 0

    # =====================================================
    # Storage
    # =====================================================

    def store_pair(
        self,
        pair: NetworkBellPair,
        current_attempt: int = 0
    ) -> QuantumMemorySlot:

        self.statistics["store_attempts"] += 1

        if not isinstance(pair, NetworkBellPair):
            raise TypeError(
                "QuantumMemory stores NetworkBellPair objects only."
            )

        if self._find_slot_by_pair_id(pair.pair_id) is not None:
            self.statistics[
                "failed_stores_duplicate"
            ] += 1

            raise QuantumMemoryDuplicatePairError(
                f"Pair {pair.pair_id} is already stored in "
                f"{self.owner.name}'s memory."
            )

        if not pair.involves(self.owner):
            self.statistics[
                "failed_stores_ownership"
            ] += 1

            raise QuantumMemoryOwnershipError(
                f"Repeater {self.owner.name} cannot store "
                f"pair {pair.pair_id}; it does not own "
                f"one endpoint of that pair."
            )

        available_slot = self._first_available_slot()

        if available_slot is None:
            self.statistics[
                "failed_stores_full"
            ] += 1

            raise QuantumMemoryFullError(
                f"Quantum memory at {self.owner.name} is full "
                f"({self.occupied_count}/{self.capacity})."
            )

        available_slot.store(
            pair=pair,
            current_attempt=current_attempt
        )

        self.statistics["successful_stores"] += 1
        self._update_peak_occupancy()

        self._log_event(
            event_type="MEMORY_PAIR_STORED",
            message=(
                f"Pair {pair.pair_id} stored in "
                f"{available_slot.slot_id}."
            ),
            pair=pair,
            slot=available_slot,
            simulation_step=current_attempt
        )

        return available_slot

    # =====================================================
    # Lookup and Retrieval
    # =====================================================

    def contains_pair(
        self,
        pair_or_id
    ) -> bool:

        pair_id = (
            pair_or_id.pair_id
            if isinstance(pair_or_id, NetworkBellPair)
            else str(pair_or_id)
        )

        return (
            self._find_slot_by_pair_id(pair_id)
            is not None
        )

    def get_pair(
        self,
        pair_or_id
    ) -> NetworkBellPair:

        pair_id = (
            pair_or_id.pair_id
            if isinstance(pair_or_id, NetworkBellPair)
            else str(pair_or_id)
        )

        slot = self._find_slot_by_pair_id(pair_id)

        if slot is None:
            raise QuantumMemoryPairNotFoundError(
                f"Pair {pair_id} is not stored in "
                f"{self.owner.name}'s memory."
            )

        return slot.stored_pair

    def retrieve_pair(
        self,
        pair_or_id
    ) -> NetworkBellPair:
        """
        Remove and return one pair for a deliberate operation.

        Phase 3C will use a separate consume_pair() call
        during entanglement swapping.
        """

        pair_id = (
            pair_or_id.pair_id
            if isinstance(pair_or_id, NetworkBellPair)
            else str(pair_or_id)
        )

        slot = self._find_slot_by_pair_id(pair_id)

        if slot is None:
            raise QuantumMemoryPairNotFoundError(
                f"Pair {pair_id} is not stored in "
                f"{self.owner.name}'s memory."
            )

        pair = slot.stored_pair

        slot.clear(
            reason="RETRIEVED"
        )

        self.statistics["retrievals"] += 1

        self._log_event(
            event_type="MEMORY_PAIR_RETRIEVED",
            message=(
                f"Pair {pair.pair_id} retrieved from "
                f"{self.owner.name}'s quantum memory."
            ),
            pair=pair
        )

        return pair

    def consume_pair(
        self,
        pair_or_id,
        reason: str = "ENTANGLEMENT_SWAP"
    ) -> NetworkBellPair:
        """
        Remove one pair because its local half was consumed
        by a quantum operation.

        The pair object itself is not globally marked consumed here,
        because another repeater may still hold the mirrored reference.
        The Phase 3C controller will coordinate global consumption.
        """

        pair_id = (
            pair_or_id.pair_id
            if isinstance(pair_or_id, NetworkBellPair)
            else str(pair_or_id)
        )

        slot = self._find_slot_by_pair_id(pair_id)

        if slot is None:
            raise QuantumMemoryPairNotFoundError(
                f"Pair {pair_id} is not stored in "
                f"{self.owner.name}'s memory."
            )

        pair = slot.stored_pair

        slot.clear(
            reason=reason
        )

        self.statistics["consumptions"] += 1

        self._log_event(
            event_type="MEMORY_PAIR_CONSUMED",
            message=(
                f"Pair {pair.pair_id} consumed from "
                f"{self.owner.name}'s memory."
            ),
            pair=pair,
            metadata={
                "reason": reason
            }
        )

        return pair

    # =====================================================
    # Discard and Cleanup
    # =====================================================

    def discard_pair(
        self,
        pair_or_id,
        reason: str = "DISCARDED"
    ) -> NetworkBellPair:

        pair_id = (
            pair_or_id.pair_id
            if isinstance(pair_or_id, NetworkBellPair)
            else str(pair_or_id)
        )

        slot = self._find_slot_by_pair_id(pair_id)

        if slot is None:
            raise QuantumMemoryPairNotFoundError(
                f"Pair {pair_id} is not stored in "
                f"{self.owner.name}'s memory."
            )

        pair = slot.stored_pair

        slot.clear(
            reason=reason
        )

        self.statistics["discards"] += 1

        self._log_event(
            event_type="MEMORY_PAIR_DISCARDED",
            message=(
                f"Pair {pair.pair_id} discarded from "
                f"{self.owner.name}'s memory."
            ),
            pair=pair,
            metadata={
                "reason": reason
            }
        )

        return pair

    def clear_session(
        self,
        session_id: str,
        reason: str = "SESSION_CLEARED"
    ) -> list[NetworkBellPair]:

        if not isinstance(session_id, str) or not session_id:
            raise ValueError(
                "session_id must be a non-empty string."
            )

        removed_pairs = []

        for slot in self.slots:
            if (
                slot.occupied
                and slot.session_id == session_id
            ):
                pair = slot.stored_pair

                slot.clear(
                    reason=reason
                )

                removed_pairs.append(pair)

                self._log_event(
                    event_type="MEMORY_SESSION_PAIR_CLEARED",
                    message=(
                        f"Session {session_id} pair "
                        f"{pair.pair_id} cleared from "
                        f"{self.owner.name}."
                    ),
                    pair=pair,
                    metadata={
                        "reason": reason
                    }
                )

        if removed_pairs:
            self.statistics[
                "session_clears"
            ] += 1

        return removed_pairs

    def clear_all(
        self,
        reason: str = "MEMORY_RESET"
    ) -> list[NetworkBellPair]:

        removed_pairs = []

        for slot in self.slots:
            if slot.occupied:
                pair = slot.stored_pair

                slot.clear(
                    reason=reason
                )

                removed_pairs.append(pair)

                self._log_event(
                    event_type="MEMORY_PAIR_CLEARED",
                    message=(
                        f"Pair {pair.pair_id} cleared from "
                        f"{self.owner.name}."
                    ),
                    pair=pair,
                    metadata={
                        "reason": reason
                    }
                )

        self.statistics["full_clears"] += 1

        return removed_pairs

    # =====================================================
    # Ageing and Expiration
    # =====================================================

    def age_all(
        self,
        steps: int = 1,
        simulation_step: Optional[int] = None
    ) -> list[dict]:
        """
        Age all occupied slots and locally remove expired pairs.

        Returned expiration records are used by the network-level
        synchronizer to remove mirrored references from other memories.
        """

        if (
            not isinstance(steps, int)
            or isinstance(steps, bool)
            or steps < 0
        ):
            raise ValueError(
                "steps must be a non-negative integer."
            )

        expired_records = []

        for slot in self.slots:
            if not slot.occupied:
                continue

            slot.increment_age(
                steps=steps
            )

            self.statistics[
                "total_age_updates"
            ] += steps

            if slot.has_expired(
                self.lifetime_attempts
            ):
                pair = slot.stored_pair
                expired_age = slot.age_attempts

                record = slot.clear(
                    reason="EXPIRED"
                )

                record["pair"] = pair
                record["expired_age_attempts"] = (
                    expired_age
                )

                expired_records.append(record)

                self.statistics["expirations"] += 1

                self._log_event(
                    event_type="MEMORY_PAIR_EXPIRED",
                    message=(
                        f"Pair {pair.pair_id} expired in "
                        f"{self.owner.name}'s memory at age "
                        f"{expired_age}."
                    ),
                    pair=pair,
                    simulation_step=simulation_step,
                    metadata={
                        "expired_age_attempts":
                            expired_age,

                        "memory_lifetime_attempts":
                            self.lifetime_attempts
                    }
                )

        return expired_records

    # =====================================================
    # Reporting
    # =====================================================

    def slots_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            slot.summary()
            for slot in self.slots
        )

    def summary(self) -> dict:
        store_attempts = self.statistics[
            "store_attempts"
        ]

        successful_stores = self.statistics[
            "successful_stores"
        ]

        return {
            "owner":
                self.owner.name,

            "capacity":
                self.capacity,

            "occupied_slots":
                self.occupied_count,

            "available_slots":
                self.available_count,

            "utilisation_percentage":
                self.utilisation_percentage,

            "peak_occupied_slots":
                self.statistics[
                    "peak_occupied_slots"
                ],

            "peak_utilisation_percentage":
                self.peak_utilisation_percentage,

            "memory_lifetime_attempts":
                self.lifetime_attempts,

            "store_attempts":
                store_attempts,

            "successful_stores":
                successful_stores,

            "store_success_rate_percentage":
                (
                    successful_stores
                    / store_attempts
                    * 100
                    if store_attempts
                    else 0.0
                ),

            "failed_stores_full":
                self.statistics[
                    "failed_stores_full"
                ],

            "failed_stores_duplicate":
                self.statistics[
                    "failed_stores_duplicate"
                ],

            "failed_stores_ownership":
                self.statistics[
                    "failed_stores_ownership"
                ],

            "retrievals":
                self.statistics["retrievals"],

            "discards":
                self.statistics["discards"],

            "expirations":
                self.statistics["expirations"],

            "consumptions":
                self.statistics["consumptions"],

            "session_clears":
                self.statistics[
                    "session_clears"
                ],

            "full_clears":
                self.statistics[
                    "full_clears"
                ],

            "total_age_updates":
                self.statistics[
                    "total_age_updates"
                ]
        }

    def __repr__(self) -> str:
        return (
            f"QuantumMemory("
            f"owner='{self.owner.name}', "
            f"occupied={self.occupied_count}/"
            f"{self.capacity}, "
            f"lifetime={self.lifetime_attempts})"
        )

In [ ]:
# @title Attach and Coordinate Repeater Memories

def initialize_repeater_memories(
    network: QuantumNetwork,
    overwrite_existing: bool = False
) -> dict[str, QuantumMemory]:
    """
    Attach one QuantumMemory object to every RepeaterNode
    registered in the network.
    """

    if not isinstance(network, QuantumNetwork):
        raise TypeError(
            "network must be a QuantumNetwork."
        )

    initialized_memories = {}

    for node in network.nodes.values():

        if not isinstance(node, RepeaterNode):
            continue

        if (
            node.quantum_memory is not None
            and not overwrite_existing
        ):
            initialized_memories[node.name] = (
                node.quantum_memory
            )
            continue

        node.quantum_memory = QuantumMemory(
            owner=node,
            capacity=node.memory_capacity,
            lifetime_attempts=(
                node.memory_lifetime_attempts
            ),
            network=network
        )

        initialized_memories[node.name] = (
            node.quantum_memory
        )

        network.log_event(
            event_type="QUANTUM_MEMORY_INITIALIZED",
            message=(
                f"Quantum memory initialized at "
                f"{node.name} with capacity "
                f"{node.memory_capacity}."
            ),
            node_names=(node.name,),
            metadata=node.quantum_memory.summary()
        )

    return initialized_memories


def repeater_endpoints_for_pair(
    pair: NetworkBellPair
) -> list[RepeaterNode]:
    """
    Return repeater endpoints involved in one Bell pair.

    Endpoint-user memory remains abstract in Phase 3B.
    """

    endpoints = []

    for node in [
        pair.node_a,
        pair.node_b
    ]:
        if isinstance(node, RepeaterNode):
            endpoints.append(node)

    return endpoints


def store_pair_at_repeater_endpoints(
    pair: NetworkBellPair,
    current_attempt: int = 0
) -> list[QuantumMemorySlot]:
    """
    Store the same Bell-pair reference at every repeater endpoint.

    Examples
    --------
    Alice--R1:
        stored at R1 only.

    R1--R2:
        the same NetworkBellPair object is stored at R1 and R2.

    R3--Bob:
        stored at R3 only.
    """

    if not isinstance(pair, NetworkBellPair):
        raise TypeError(
            "pair must be a NetworkBellPair."
        )

    repeater_endpoints = (
        repeater_endpoints_for_pair(pair)
    )

    if not repeater_endpoints:
        raise QuantumMemoryOwnershipError(
            f"Pair {pair.pair_id} has no repeater endpoint."
        )

    stored_slots = []

    try:
        for repeater in repeater_endpoints:

            if repeater.quantum_memory is None:
                raise QuantumMemoryError(
                    f"Repeater {repeater.name} has no "
                    f"initialized quantum memory."
                )

            slot = (
                repeater.quantum_memory.store_pair(
                    pair=pair,
                    current_attempt=current_attempt
                )
            )

            stored_slots.append(slot)

    except Exception:
        # Roll back any partial multi-memory store.
        for slot in stored_slots:
            memory = slot.owner.quantum_memory

            if memory.contains_pair(pair.pair_id):
                memory.discard_pair(
                    pair.pair_id,
                    reason="MULTI_ENDPOINT_STORE_ROLLBACK"
                )

        raise

    return stored_slots


def find_pair_memories(
    network: QuantumNetwork,
    pair_or_id
) -> list[QuantumMemory]:
    """
    Find every repeater memory currently storing one pair ID.
    """

    pair_id = (
        pair_or_id.pair_id
        if isinstance(pair_or_id, NetworkBellPair)
        else str(pair_or_id)
    )

    matching_memories = []

    for node in network.nodes.values():

        if (
            isinstance(node, RepeaterNode)
            and node.quantum_memory is not None
            and node.quantum_memory.contains_pair(
                pair_id
            )
        ):
            matching_memories.append(
                node.quantum_memory
            )

    return matching_memories


def remove_pair_from_all_repeater_memories(
    network: QuantumNetwork,
    pair_or_id,
    reason: str = "NETWORK_WIDE_REMOVAL",
    mark_pair_inactive: bool = False
) -> list[str]:
    """
    Remove every local memory reference to one shared Bell pair.

    This keeps mirrored repeater-side references synchronized.
    """

    pair_id = (
        pair_or_id.pair_id
        if isinstance(pair_or_id, NetworkBellPair)
        else str(pair_or_id)
    )

    pair_object = (
        pair_or_id
        if isinstance(pair_or_id, NetworkBellPair)
        else None
    )

    removed_from = []

    for memory in find_pair_memories(
        network,
        pair_id
    ):
        pair = memory.discard_pair(
            pair_id,
            reason=reason
        )

        removed_from.append(
            memory.owner.name
        )

        if pair_object is None:
            pair_object = pair

    if (
        mark_pair_inactive
        and pair_object is not None
    ):
        pair_object.deactivate(
            reason=reason
        )

    if removed_from:
        network.log_event(
            event_type="PAIR_REMOVED_NETWORK_WIDE",
            message=(
                f"Pair {pair_id} removed from repeater "
                f"memories: {removed_from}."
            ),
            node_names=tuple(removed_from),
            session_id=(
                pair_object.session_id
                if pair_object is not None
                else None
            ),
            metadata={
                "pair_id":
                    pair_id,

                "reason":
                    reason,

                "removed_from":
                    removed_from
            }
        )

    return removed_from


def age_all_network_memories(
    network: QuantumNetwork,
    steps: int = 1,
    simulation_step: Optional[int] = None
) -> list[dict]:
    """
    Age all repeater memories once and synchronize expirations.

    If one local half expires, every mirrored reference to that
    shared Bell pair is invalidated across the repeater network.
    """

    expiration_records = []

    repeater_memories = [
        node.quantum_memory
        for node in network.nodes.values()
        if (
            isinstance(node, RepeaterNode)
            and node.quantum_memory is not None
        )
    ]

    # First age every memory and collect locally expired pairs.
    for memory in repeater_memories:

        local_expirations = memory.age_all(
            steps=steps,
            simulation_step=simulation_step
        )

        expiration_records.extend(
            local_expirations
        )

    expired_pair_ids = {
        record["pair_id"]
        for record in expiration_records
    }

    # Then remove any surviving mirrored references.
    for pair_id in expired_pair_ids:

        matching_memories = find_pair_memories(
            network,
            pair_id
        )

        for memory in matching_memories:
            memory.discard_pair(
                pair_id,
                reason="MIRRORED_HALF_EXPIRED"
            )

        pair_objects = [
            record["pair"]
            for record in expiration_records
            if record["pair_id"] == pair_id
        ]

        if pair_objects:
            pair_objects[0].deactivate(
                reason="MEMORY_EXPIRED"
            )

        network.log_event(
            event_type="PAIR_INVALIDATED_BY_EXPIRATION",
            message=(
                f"Pair {pair_id} invalidated because one "
                f"stored half exceeded memory lifetime."
            ),
            session_id=(
                pair_objects[0].session_id
                if pair_objects
                else None
            ),
            simulation_step=simulation_step,
            metadata={
                "pair_id":
                    pair_id,

                "expired_records":
                    [
                        {
                            key: value
                            for key, value
                            in record.items()
                            if key != "pair"
                        }
                        for record
                        in expiration_records
                        if record["pair_id"] == pair_id
                    ]
            }
        )

    return expiration_records


def clear_session_from_all_memories(
    network: QuantumNetwork,
    session_id: str,
    reason: str = "NETWORK_SESSION_CLEANUP"
) -> list[str]:
    """
    Remove every stored pair belonging to one session.
    """

    cleared_pair_ids = set()

    for node in network.nodes.values():

        if (
            isinstance(node, RepeaterNode)
            and node.quantum_memory is not None
        ):
            removed_pairs = (
                node.quantum_memory.clear_session(
                    session_id=session_id,
                    reason=reason
                )
            )

            cleared_pair_ids.update(
                pair.pair_id
                for pair in removed_pairs
            )

    network.log_event(
        event_type="NETWORK_SESSION_MEMORY_CLEARED",
        message=(
            f"All quantum-memory resources for session "
            f"{session_id} were cleared."
        ),
        session_id=session_id,
        metadata={
            "cleared_pair_ids":
                sorted(cleared_pair_ids),

            "reason":
                reason
        }
    )

    return sorted(cleared_pair_ids)


def memory_summary_dataframe(
    network: QuantumNetwork
) -> pd.DataFrame:
    """
    Return one summary row per repeater memory.
    """

    rows = []

    for node in network.nodes.values():

        if (
            isinstance(node, RepeaterNode)
            and node.quantum_memory is not None
        ):
            rows.append(
                node.quantum_memory.summary()
            )

    return pd.DataFrame(rows)


def memory_slots_dataframe(
    network: QuantumNetwork
) -> pd.DataFrame:
    """
    Return one row per quantum-memory slot.
    """

    rows = []

    for node in network.nodes.values():

        if (
            isinstance(node, RepeaterNode)
            and node.quantum_memory is not None
        ):
            rows.extend(
                slot.summary()
                for slot
                in node.quantum_memory.slots
            )

    return pd.DataFrame(rows)

In [ ]:
# @title Initialize Repeater Quantum Memories

phase3b_memories = (
    initialize_repeater_memories(
        network=phase3a_network,
        overwrite_existing=True
    )
)

print("=" * 72)
print("Phase 3B Quantum Memories Initialized")
print("=" * 72)

for repeater_name, memory in (
    phase3b_memories.items()
):
    print(
        f"{repeater_name:<12}: {memory}"
    )

print("=" * 72)


assert r1_p3.quantum_memory is not None
assert r2_p3.quantum_memory is not None
assert r3_p3.quantum_memory is not None
assert r4_p3.quantum_memory is not None

assert (
    r1_p3.quantum_memory.capacity
    == r1_p3.memory_capacity
)

assert (
    r2_p3.quantum_memory.lifetime_attempts
    == r2_p3.memory_lifetime_attempts
)

print(
    "\nPhase 3B memory initialization passed."
)

In [ ]:
# @title Create Ideal Elementary Bell Pairs

PHASE3B_TEST_SESSION_ID = (
    "PHASE3B-LINEAR-CHAIN-TEST"
)

phase3b_chain_links = [
    phase3a_network.get_link(
        alice_p3,
        r1_p3
    ),
    phase3a_network.get_link(
        r1_p3,
        r2_p3
    ),
    phase3a_network.get_link(
        r2_p3,
        r3_p3
    ),
    phase3a_network.get_link(
        r3_p3,
        bob_p3
    )
]


phase3b_elementary_pairs = []

for creation_attempt, link in enumerate(
    phase3b_chain_links,
    start=1
):
    pair = NetworkBellPair(
        node_a=link.source,
        node_b=link.destination,
        session_id=(
            PHASE3B_TEST_SESSION_ID
        ),
        source_link_id=link.link_id,
        source_links=[link.link_id],
        creation_attempt=creation_attempt,
        fidelity=1.0
    )

    phase3b_elementary_pairs.append(
        pair
    )


print("=" * 78)
print("Phase 3B Ideal Elementary Bell Pairs")
print("=" * 78)

for pair in phase3b_elementary_pairs:
    print(pair)

print("=" * 78)


assert len(phase3b_elementary_pairs) == 4

assert all(
    pair.active
    for pair in phase3b_elementary_pairs
)

assert len({
    pair.pair_id
    for pair in phase3b_elementary_pairs
}) == 4

print(
    "\nPhase 3B elementary-pair creation passed."
)

In [ ]:
# @title Store Elementary Pairs Across Repeater Endpoints

phase3b_stored_slots = {}

for pair in phase3b_elementary_pairs:

    stored_slots = (
        store_pair_at_repeater_endpoints(
            pair=pair,
            current_attempt=(
                pair.creation_attempt
            )
        )
    )

    phase3b_stored_slots[
        pair.pair_id
    ] = [
        slot.slot_id
        for slot in stored_slots
    ]


print("=" * 78)
print("Phase 3B Bell-Pair Memory Placement")
print("=" * 78)

for pair_id, slot_ids in (
    phase3b_stored_slots.items()
):
    print(
        f"{pair_id:<14}: {slot_ids}"
    )

print("=" * 78)


# =========================================================
# Expected memory state:
#
# R1 -> Alice-R1 and R1-R2
# R2 -> R1-R2 and R2-R3
# R3 -> R2-R3 and R3-Bob
# R4 -> empty
# =========================================================

assert r1_p3.quantum_memory.occupied_count == 2
assert r2_p3.quantum_memory.occupied_count == 2
assert r3_p3.quantum_memory.occupied_count == 2
assert r4_p3.quantum_memory.occupied_count == 0

assert r1_p3.quantum_memory.is_full
assert r2_p3.quantum_memory.is_full
assert r3_p3.quantum_memory.is_full
assert r4_p3.quantum_memory.is_empty


# Internal elementary pairs must use the same object
# reference in both endpoint memories.

pair_r1_r2 = phase3b_elementary_pairs[1]

assert (
    r1_p3.quantum_memory.get_pair(
        pair_r1_r2.pair_id
    )
    is
    r2_p3.quantum_memory.get_pair(
        pair_r1_r2.pair_id
    )
)


pair_r2_r3 = phase3b_elementary_pairs[2]

assert (
    r2_p3.quantum_memory.get_pair(
        pair_r2_r3.pair_id
    )
    is
    r3_p3.quantum_memory.get_pair(
        pair_r2_r3.pair_id
    )
)


print(
    "\nPhase 3B memory placement passed."
)

In [ ]:
# @title Quantum Memory State Tables

print("REPEATER MEMORY SUMMARY")

df_phase3b_memory_summary = (
    memory_summary_dataframe(
        phase3a_network
    )
)

display(
    df_phase3b_memory_summary
)


print("\nQUANTUM MEMORY SLOT STATE")

df_phase3b_memory_slots = (
    memory_slots_dataframe(
        phase3a_network
    )
)

display(
    df_phase3b_memory_slots
)

In [ ]:
# @title Quantum Memory Protection Tests

print("=" * 78)
print("Phase 3B Quantum Memory Protection Tests")
print("=" * 78)


# =========================================================
# Test 1: Duplicate local storage must fail
# =========================================================

duplicate_test_passed = False

try:
    r1_p3.quantum_memory.store_pair(
        pair=phase3b_elementary_pairs[0],
        current_attempt=10
    )

except QuantumMemoryDuplicatePairError as error:
    duplicate_test_passed = True

    print(
        "Duplicate-pair protection:"
        "\n  PASSED"
        f"\n  {error}\n"
    )

assert duplicate_test_passed


# =========================================================
# Test 2: Unrelated repeater ownership must fail
# =========================================================

unrelated_pair = NetworkBellPair(
    node_a=alice_p3,
    node_b=r1_p3,
    session_id="PHASE3B-OWNERSHIP-TEST",
    source_link_id=(
        phase3a_network
        .get_link(alice_p3, r1_p3)
        .link_id
    )
)

ownership_test_passed = False

try:
    r4_p3.quantum_memory.store_pair(
        pair=unrelated_pair,
        current_attempt=10
    )

except QuantumMemoryOwnershipError as error:
    ownership_test_passed = True

    print(
        "Ownership protection:"
        "\n  PASSED"
        f"\n  {error}\n"
    )

assert ownership_test_passed


# =========================================================
# Test 3: Full-memory protection must fail
# =========================================================

full_memory_test_pair = NetworkBellPair(
    node_a=r1_p3,
    node_b=r4_p3,
    session_id="PHASE3B-FULL-MEMORY-TEST"
)

full_memory_test_passed = False

try:
    r1_p3.quantum_memory.store_pair(
        pair=full_memory_test_pair,
        current_attempt=10
    )

except QuantumMemoryFullError as error:
    full_memory_test_passed = True

    print(
        "Full-memory protection:"
        "\n  PASSED"
        f"\n  {error}\n"
    )

assert full_memory_test_passed


# =========================================================
# Ensure valid state was not corrupted
# =========================================================

assert r1_p3.quantum_memory.occupied_count == 2
assert r2_p3.quantum_memory.occupied_count == 2
assert r3_p3.quantum_memory.occupied_count == 2
assert r4_p3.quantum_memory.occupied_count == 0


print("=" * 78)
print("All Phase 3B memory-protection tests passed.")
print("=" * 78)

In [ ]:
# @title Memory Ageing and Expiration Test

# Preserve the fully populated main-chain memories.
original_r1_memory = r1_p3.quantum_memory
original_r2_memory = r2_p3.quantum_memory


# Install isolated short-lifetime test memories.
r1_p3.quantum_memory = QuantumMemory(
    owner=r1_p3,
    capacity=2,
    lifetime_attempts=3,
    network=phase3a_network
)

r2_p3.quantum_memory = QuantumMemory(
    owner=r2_p3,
    capacity=2,
    lifetime_attempts=3,
    network=phase3a_network
)


expiration_test_pair = NetworkBellPair(
    node_a=r1_p3,
    node_b=r2_p3,
    session_id="PHASE3B-EXPIRATION-TEST",
    source_link_id=(
        phase3a_network
        .get_link(r1_p3, r2_p3)
        .link_id
    ),
    creation_attempt=0
)


store_pair_at_repeater_endpoints(
    pair=expiration_test_pair,
    current_attempt=0
)


print("=" * 78)
print("Phase 3B Synchronized Expiration Test")
print("=" * 78)


for global_attempt in range(1, 4):

    expiration_records = (
        age_all_network_memories(
            network=phase3a_network,
            steps=1,
            simulation_step=global_attempt
        )
    )

    print(
        f"After global attempt {global_attempt}:"
    )

    print(
        f"  R1 stores pair: "
        f"{r1_p3.quantum_memory.contains_pair(expiration_test_pair.pair_id)}"
    )

    print(
        f"  R2 stores pair: "
        f"{r2_p3.quantum_memory.contains_pair(expiration_test_pair.pair_id)}"
    )

    print(
        f"  Pair active: "
        f"{expiration_test_pair.active}"
    )

    print(
        f"  Expiration records: "
        f"{len(expiration_records)}\n"
    )


# With lifetime = 3 and age >= lifetime expiry:
# age 1 -> valid
# age 2 -> valid
# age 3 -> expired everywhere

assert not r1_p3.quantum_memory.contains_pair(
    expiration_test_pair.pair_id
)

assert not r2_p3.quantum_memory.contains_pair(
    expiration_test_pair.pair_id
)

assert expiration_test_pair.active is False

assert (
    r1_p3.quantum_memory.statistics[
        "expirations"
    ]
    +
    r2_p3.quantum_memory.statistics[
        "expirations"
    ]
    >= 1
)


print(
    "Synchronized memory expiration passed."
)

print("=" * 78)


# Restore the original fully populated chain memories.
r1_p3.quantum_memory = original_r1_memory
r2_p3.quantum_memory = original_r2_memory


assert r1_p3.quantum_memory.occupied_count == 2
assert r2_p3.quantum_memory.occupied_count == 2

print(
    "\nOriginal Phase 3B chain memories restored."
)

In [ ]:
# @title Network-Wide Session Cleanup Test

# Use R4's empty memory so the main chain remains untouched.

session_cleanup_pair = NetworkBellPair(
    node_a=r4_p3,
    node_b=charlie_p3,
    session_id="PHASE3B-SESSION-CLEANUP-TEST",
    creation_attempt=1
)


store_pair_at_repeater_endpoints(
    pair=session_cleanup_pair,
    current_attempt=1
)


assert r4_p3.quantum_memory.contains_pair(
    session_cleanup_pair.pair_id
)


cleared_pair_ids = (
    clear_session_from_all_memories(
        network=phase3a_network,
        session_id=(
            "PHASE3B-SESSION-CLEANUP-TEST"
        ),
        reason="PHASE3B_TEST_COMPLETED"
    )
)


assert (
    session_cleanup_pair.pair_id
    in cleared_pair_ids
)

assert not r4_p3.quantum_memory.contains_pair(
    session_cleanup_pair.pair_id
)


print("=" * 78)
print("Phase 3B Session Cleanup Test")
print("=" * 78)

print(
    "Cleared pair IDs:",
    cleared_pair_ids
)

print(
    "\nR4 memory occupancy:",
    f"{r4_p3.quantum_memory.occupied_count}/"
    f"{r4_p3.quantum_memory.capacity}"
)

print(
    "\nNetwork-wide session cleanup passed."
)

print("=" * 78)

In [ ]:
# @title Final Verification and Event Log

print("=" * 84)
print("NARAD-QC Phase 3B Final Verification")
print("=" * 84)


# =========================================================
# Verify final intended chain-memory state
# =========================================================

expected_occupancy = {
    "R1": 2,
    "R2": 2,
    "R3": 2,
    "R4": 0
}

actual_occupancy = {
    repeater.name:
        repeater.quantum_memory.occupied_count

    for repeater in [
        r1_p3,
        r2_p3,
        r3_p3,
        r4_p3
    ]
}


for repeater_name, expected in (
    expected_occupancy.items()
):
    actual = actual_occupancy[
        repeater_name
    ]

    print(
        f"{repeater_name:<10}: "
        f"occupied={actual}, "
        f"expected={expected}"
    )

    assert actual == expected


# =========================================================
# Verify mirrored internal-link references
# =========================================================

internal_pair_1 = (
    phase3b_elementary_pairs[1]
)

internal_pair_2 = (
    phase3b_elementary_pairs[2]
)


assert (
    len(
        find_pair_memories(
            phase3a_network,
            internal_pair_1.pair_id
        )
    )
    == 2
)

assert (
    len(
        find_pair_memories(
            phase3a_network,
            internal_pair_2.pair_id
        )
    )
    == 2
)


# =========================================================
# Final tables
# =========================================================

print("\nFINAL MEMORY SUMMARY")

df_phase3b_final_memory_summary = (
    memory_summary_dataframe(
        phase3a_network
    )
)

display(
    df_phase3b_final_memory_summary
)


print("\nFINAL MEMORY SLOT STATE")

df_phase3b_final_slots = (
    memory_slots_dataframe(
        phase3a_network
    )
)

display(
    df_phase3b_final_slots
)


print("\nPHASE 3B MEMORY EVENTS")

df_phase3b_memory_events = (
    phase3a_network
    .events_dataframe()
)

phase3b_event_types = [
    "QUANTUM_MEMORY_INITIALIZED",
    "MEMORY_PAIR_STORED",
    "MEMORY_PAIR_RETRIEVED",
    "MEMORY_PAIR_CONSUMED",
    "MEMORY_PAIR_DISCARDED",
    "MEMORY_PAIR_EXPIRED",
    "PAIR_INVALIDATED_BY_EXPIRATION",
    "PAIR_REMOVED_NETWORK_WIDE",
    "MEMORY_SESSION_PAIR_CLEARED",
    "NETWORK_SESSION_MEMORY_CLEARED"
]

df_phase3b_memory_events = (
    df_phase3b_memory_events[
        df_phase3b_memory_events[
            "event_type"
        ].isin(
            phase3b_event_types
        )
    ]
    .copy()
)

display(
    df_phase3b_memory_events
)


print("=" * 84)
print("PHASE 3B COMPLETED SUCCESSFULLY")
print("=" * 84)

print(
    "\nThe multi-repeater chain now possesses "
    "session-aware, capacity-limited, age-aware and "
    "network-synchronized quantum memories."
)